# CareSaathi: find the right care, with proof

**CMU Data Science Club × Databricks Datathon, Fall 2026.** *Saathi* means companion.

Code and history: https://github.com/beomjin-seo/caresaathi · The app runs on Databricks Apps (Free Edition).

## The problem
A family names a need (dialysis, a C-section tonight, bone surgery) and a place. Which nearby facility can really do it, why should they believe it, and what should they check before travelling? The facility table has 9,989 records written from websites, with no ground truth about which services are really delivered. So we do not predict. **We grade the evidence each record gives, and show it.**

## How CareSaathi decides (one score per facility and service)
1. **Read** every sentence about the service in the record (19 services, regex dictionary in `ontology.py`; the facility's own name never counts).
2. **Grade** it: a loose mention ×1, a named procedure ×4.8, equipment or a unit with detail ×9.1; the share of items the service needs (theatre, anaesthesia, blood for a C-section) found in the record, up to ×26; the facility's own website has a page for it ×2.9.
3. **One real place?** Coordinates, postcode and city all disagree ×0.13; the text names two or more towns over 60 km away ×0.19; the row was built from two or more other organisations' websites ×0.46.
4. **Score:** odds = 0.74 × evidence × one-real-place, trust = 100 × odds ÷ (1 + odds). Strong ≥ 90, some ≥ 70, weak below (left out).
5. **Rules outside the score:** a single-specialty facility (dental, eye, skin, physio, ayurveda or homeopathy) claiming care outside its field is left out with the reason; a service's must-have item (for example an operating theatre for a C-section) missing from the record caps the score just under "strong" and the card says to confirm it by phone.
6. **Rank:** emergency, nearest first (places within 3.6 km count as equally near, the measured error of map points; stronger evidence first among them); planned, highest trust first inside the smallest area that holds five places (scores within 2.1 points count as equal, the measured spread of the score over refits; nearer first among them).

## Where the numbers come from
The multipliers are logistic-regression coefficients fitted on 605 labelled claims: 387 labelled from facility names (a kidney centre claiming dialysis is likely real; a dental clinic claiming dialysis is likely false; names are never a feature) and 218 set by AI readers of the record. On claims held out from fitting (facilities kept whole across folds): AUC 0.88; likely real in 98% of "strong", 84% of "some", 44% of "weak". Weakest service: heart surgery (AUC 0.76). **Nobody visited or phoned these facilities**, so the score is a ranking index, not a probability, and every card ends with what to ask on the phone.

## What the app adds for people who cannot read well
Picture tiles, English and Hindi, read-aloud on every line with play and pause, free-text and voice input, the nearest place in an emergency with ambulance numbers (108, 102, 112), cost guidance (public facilities, Ayushman card), district context from NFHS-5, a health-worker list to print or read out, and "Who will treat you": doctors named on each hospital's own website, matched to the need (read on 22 Nov 2026, not verified, not scored).

## Known limits
The table is about 5% public facilities while most births in India happen in public ones. No text for a service does not mean the place lacks it. Labels are proxies, not site visits. Distances are straight lines. Full notes: `analysis/FINDINGS.md` in the repository.

## How to run this notebook on Databricks Free Edition
1. Get the Marketplace listing **Virtue Foundation Dataset (DAIS 2026)** so the catalog `databricks_virtue_foundation_dataset_dais_2026` exists.
2. Import this notebook, attach serverless compute, **Run all** (about 8 minutes).
3. The cells write the pipeline to `/tmp/caresaathi`, read the three source tables, run every step, save Delta tables `workspace.default.caresaathi_*`, and build the app page `app/index.html`.

In [ ]:
import os, sys, json, base64, gzip, io, tarfile, subprocess, shutil
WORK = "/tmp/caresaathi"
shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)
def write(path, text):
    full = os.path.join(WORK, path); os.makedirs(os.path.dirname(full), exist_ok=True)
    open(full, "w", encoding="utf-8").write(text.lstrip("\n"))
    print("wrote", path, len(text), "characters")

### `pipeline/common.py`
Loading the facility table: keep facilities, drop duplicate ids, parse list columns.

In [ ]:
write('pipeline/common.py', r'''
"""Shared loading/cleaning helpers. Run scripts from the project root."""
import os, json, re
import numpy as np, pandas as pd

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
OUT = os.path.join(ROOT, "pipeline", "out")
LIST_COLS = ["specialties", "procedure", "equipment", "capability", "source_types",
             "source_ids", "source_urls", "websites", "phone_numbers", "affiliationTypeIds"]
NULLS = {"", "null", "none", "nan", "[]", "n/a", "na"}


def jl(s):
    """Parse a JSON-list cell; anything else becomes an empty list."""
    if not isinstance(s, str):
        return []
    try:
        v = json.loads(s)
        return [x for x in v if isinstance(x, str) and x.strip().lower() not in NULLS] if isinstance(v, list) else []
    except Exception:
        return []


def clean_text(s):
    return "" if (not isinstance(s, str) or s.strip().lower() in NULLS) else s.strip()


def load_facilities():
    """10,088 raw rows -> rows where organization_type == 'facility', one per unique_id."""
    f = pd.read_csv(os.path.join(ROOT, "facilities.csv"), dtype=str, low_memory=False)
    f = f[f.organization_type == "facility"].drop_duplicates("unique_id").reset_index(drop=True)
    for c in LIST_COLS:
        f[c + "_l"] = f[c].map(jl)
    for c in ["latitude", "longitude", "numberDoctors", "capacity", "yearEstablished"]:
        f[c + "_n"] = pd.to_numeric(f[c], errors="coerce")
    for c in ["name", "description", "address_city", "address_stateOrRegion", "facilityTypeId", "operatorTypeId", "officialWebsite", "officialPhone"]:
        f[c] = f[c].map(clean_text)
    return f


def haversine(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = map(np.radians, [la1, lo1, la2, lo2])
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(h))


def norm_name(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()
''')

### `pipeline/ontology.py`
The service dictionary: what counts as a loose mention, a procedure or a unit, false friends to mask, and the items each service needs.

In [ ]:
write('pipeline/ontology.py', r'''
# Service ontology v2 -- paste-ready. All regexes are case-insensitive unless (?-i:...) is used.
# Keys per service:
#   infra   : named unit / equipment / infrastructure  (can reach STRONG)
#   proc    : named procedure or explicit service        (MEDIUM)
#   generic : loose topical mention                      (WEAK)
#   exclude : lexical false friends, masked out before matching
#   spec_direct  : regex over normalised specialties enum (lowercase, letters only) -> service-specific enum
#   spec_related : parent specialty enum (necessary-ish, far from sufficient)
#   coreq   : co-requirement name -> regex searched over ALL text of the facility
SERVICE_ONTOLOGY = {
 "dialysis": {
  "label": "Dialysis (haemodialysis / peritoneal)",
  "infra": [r"dialysis (unit|cent(re|er)|ward|wing|room|department|facility|machines?|stations?|beds?|chairs?|suite)",
            r"(ha?emo)?dialysis machines?", r"dialy[sz]ers?", r"\bRO (water )?plant\b", r"dialysis[- ]bed"],
  "proc": [r"ha?emo[- ]?dialysis", r"peritoneal dialysis", r"\bCAPD\b", r"\bCRRT\b", r"\bSLED\b", r"\bMHD\b",
           r"\bdialysis\b", r"ha?emodiafiltration", r"plasmapheresis"],
  "generic": [r"\bA[- ]?V fistula", r"renal replacement", r"kidney failure", r"chronic kidney disease|\bCKD\b", r"nephrolog"],
  "exclude": [r"without dialysis", r"avoid(ing)? dialysis", r"dialysis[- ]free", r"stop dialysis"],
  "spec_direct": r"dialysis",
  "spec_related": r"^(pa?ediatric)?nephrolog|^renal",
  "coreq": {"nephrologist": r"nephrolog", "RO_water": r"\bRO (water )?plant|reverse osmosis", "ICU": r"\bICU\b|intensive care",
            "sessions_or_machine_count": r"\d+\s*(dialysis )?(machines?|stations?)|\d[\d,]*\+?\s*dialys(is|es)( sessions)?"},
 },
 "emergency_obstetric": {
  "label": "Emergency C-section / high-risk delivery",
  "infra": [r"labou?r (room|ward|suite|theat(re|er)|delivery (room|suite))", r"\bLDR[P]?\b", r"obstetric (ICU|HDU|OT)",
            r"maternity (ward|wing|OT|operation theat(re|er)|block)", r"\bCEmONC\b|\bCEmOC\b|\bEmOC\b|\bFRU\b|first referral unit",
            r"delivery (room|suite|table)s?", r"\bMCH (wing|block)\b|\bLaQshya\b"],
  "proc": [r"c[- ]?section", r"ca?esar(e|i)an", r"\bLSCS\b", r"\b(emergency|elective) CS\b", r"high[- ]risk (pregnanc|deliver|obstetric|matern)",
           r"obstetric(al)? emergenc", r"emergency obstetric", r"emergency (deliver|c[- ]section|ca?esar)", r"\bVBAC\b",
           r"eclampsia|post[- ]?partum ha?emorrhage|\bPPH\b|placenta (pra?evia|accreta)", r"(operative|instrumental|complicated|assisted) deliver",
           r"painless (deliver|labou?r)|epidural (analgesia|deliver)", r"normal (and|&|/) (c[- ]?section|ca?esar(e|i)an|LSCS)"],
  "generic": [r"normal deliver", r"\bdeliver(y|ies)\b(?! (of|to) (care|service|health|quality|medic|treat))", r"maternity", r"obstetric", r"antenatal|\bANC\b|prenatal",
              r"pregnancy care"],
  "exclude": [r"home delivery|free delivery|delivery of (care|services|healthcare|medicines?|quality|treatment|results|reports)|(care|service|healthcare|medicine|drug|report|sample|oxygen) delivery|delivery (boy|charges|partner|system|model)",
              r"door[- ]?step delivery", r"ca?esarean scar"],
  "spec_direct": r"obstetric|maternalfetal|maternityc|perinatolog",
  "spec_related": r"gyn(a)?ecolog|maternalandchild",
  "coreq": {"operation_theatre": r"operation theat(re|er)|\bOT\b|operating (room|theat)", "anaesthesia": r"ana?esthe",
            "blood_bank": r"blood (bank|storage|cent(re|er))", "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?|round[- ]the[- ]clock",
            "newborn_care": r"\bNICU\b|\bSNCU\b|\bNBSU\b|neonatal|newborn care", "obstetrician": r"obstetric|gyn(a)?ecolog"},
 },
 "nicu": {
  "label": "NICU / sick-newborn care",
  "infra": [r"\bNICU\b", r"\bSNCU\b", r"\bNBSU\b", r"neonatal (intensive care|ICU|care unit|unit|HDU)", r"newborn (intensive care|ICU|care unit|stabili[sz]ation unit)",
            r"(special|sick) newborn care", r"level[- ]?(2|3|II|III|IIA|IIB|IIIA|IIIB)\b.{0,15}(NICU|neonat|nursery)", r"neonatal ventilat", r"radiant warmers?",
            r"(infant|neonatal|baby|transport|newborn) (incubators?|warmers?)", r"bubble CPAP|neonatal CPAP", r"phototherapy (units?|machines?|lights?)", r"kangaroo mother care|\bKMC\b (unit|ward)"],
  "proc": [r"neonatal (care|resuscitation|surger|transport)", r"(neonatal|newborn|jaundice).{0,25}phototherapy|phototherapy.{0,25}(neonat|newborn|jaundice|bab)", r"surfactant", r"exchange transfusion", r"(pre[- ]?term|premature|low birth weight|\bLBW\b|\bVLBW\b) (bab|infant|care|newborn)",
           r"neonatologists?\b"],
  "generic": [r"newborn care", r"new ?born", r"neonat", r"\bphototherapy\b", r"\bincubators?\b"],
  "exclude": [r"retinopathy of prematurity|\bROP\b", r"(CO2|carbon dioxide|lab(oratory)?|bacteriolog\w+|embryo|IVF|tri[- ]?gas|bench[- ]?top|culture|BOD|microbiolog\w+|shaker|platelet) incubators?|incubators? (for|in) (embryo|IVF|culture|lab)", r"phototherapy (for|in) (psoriasis|vitiligo|skin)|\b(NB[- ]?UVB|PUVA|narrow ?band|excimer|whole body|targeted|UVB?|UVA) phototherapy|phototherapy (chamber|cabin)"],
  "spec_direct": r"neonat",
  "spec_related": r"^pa?ediatrics$|pediatriccriticalcare|maternalfetal",
  "coreq": {"ventilator": r"ventilat(?!ed\b)(?!ion system)", "warmer_or_incubator": r"warmer|incubator", "neonatologist": r"neonatolog", "phototherapy": r"phototherap",
            "CPAP": r"\bCPAP\b", "paediatrician": r"pa?ediatric", "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?|round[- ]the[- ]clock"},
 },
 "cardiac_surgery": {
  "label": "Cardiac surgery (CABG / valve / open heart)",
  "infra": [r"cardiac (OT|operation theat(re|er)|operating room)", r"\bCT ?VS (OT|ICU|unit|department|theat)", r"cardio[- ]?thoracic (OT|ICU|unit|theat)",
            r"heart[- ]lung machine", r"cardiopulmonary bypass (machine|pump)|perfusion (machine|pump|system)", r"\bCTICU\b|cardiac surg(ery|ical) (ICU|unit|recovery)|post[- ]?cardiac surgery ICU"],
  "proc": [r"\bCABG\b", r"(coronary|heart|cardiac) (artery )?bypass", r"bypass (surgery|graft)", r"open[- ]heart", r"(valve|mitral|aortic) (valve )?(replacement|repair)",
           r"cardiac surger", r"heart surger", r"\bCT ?VS\b", r"cardio[- ]?(thoracic|vascular) (and (vascular|thoracic) )?surg", r"(beating heart|off[- ]pump|\bOPCAB\b|minimally invasive cardiac|\bMICS\b.{0,30}(cardiac|heart|CABG|valve)|(cardiac|heart|CABG|valve).{0,30}\bMICS\b)",
           r"heart transplant", r"(\bASD\b|\bVSD\b|\bTOF\b|congenital heart).{0,25}(surg|repair|closure|correction)", r"\bBentall\b|aortic (aneurysm|dissection) (repair|surgery)"],
  "generic": [r"cardiac surgeon", r"heart (hospital|institute|cent(re|er))", r"cardiac sciences"],
  "exclude": [r"gastric bypass|bariatric bypass|mini[- ]gastric|bypass (road|rd\b|highway)|\b(on|near|opp\.?|opposite) [\w ]{0,25}bypass\b|[A-Z]\w+ bypass,|(?-i:Bypass (Road|Rd|Chowk|Junction|Circle))",
              r"without (surgery|bypass)|bypass without|avoid(ing)? bypass|non[- ]surgical|no surgery|alternative to (bypass|surgery)|\bEECP\b|\bESMR\b|chelation"],
  "spec_direct": r"cardiacsurg|cardiothoracic(surg|andvasc)|cardiovascularsurg|\bctvs\b|heartsurg",
  "spec_related": r"^cardiology$|thoracicsurg|vascularsurg",
  "coreq": {"cardiac_OT": r"operation theat(re|er)|\bOT\b|operating (room|theat)", "cath_lab": r"cath[- ]?lab|catheteri[sz]ation lab",
            "cardiac_ICU": r"\bCTICU\b|\bICCU\b|\bCCU\b|cardiac (ICU|intensive)", "heart_lung_machine": r"heart[- ]lung (machine|bypass)|cardio[- ]?pulmonary bypass|\bCPB\b|perfusionist|perfusion (machine|pump|system)|\bECMO\b",
            "cardiac_surgeon": r"cardiac surgeon|heart surgeon|cardio[- ]?(thoracic|vascular)( (and|&) vascular)? surgeon|\bCT ?VS\b|\bM\.?\s?Ch\b\.?[\s,(\-]{0,4}(CT ?VS|CVTS|cardio|thoracic)", "blood_bank": r"blood (bank|storage|cent(re|er))", "ventilator": r"ventilat(?!ed\b)(?!ion system)"},
 },
 "mri": {
  "label": "MRI",
  "infra": [r"\bMRI (scanner|machine|system|unit|suite|cent(re|er)|facility|scan cent)", r"\b(0\.[2-5]|1\.5|3(\.0)?)\s?-?\s?T(esla)?\b.{0,25}\bMRI\b|\bMRI\b.{0,40}\b(0\.[2-5]|1\.5|3(\.0)?)\s?-?\s?T(esla)?\b",
            r"\b(1\.5|3(\.0)?)\s?-?\s?Tesla\b", r"\b(open|closed|silent|wide[- ]bore|high[- ]field|digital) MRI\b",
            r"(Siemens|GE|Philips|Hitachi|Toshiba|Canon|Magnetom|Signa|Ingenia|Achieva|Multiva|Skyra|Avanto|Aera|Essenza).{0,25}\bMRI\b|\bMRI\b.{0,25}(Siemens|\bGE\b|Philips|Hitachi|Toshiba|Magnetom|Signa|Ingenia)"],
  "proc": [r"\bMRI\b", r"magnetic resonance", r"\bfMRI\b|\bMRCP\b|\bMR (angiograph|spectroscop|venograph)"],
  "generic": [],
  "exclude": [r"\bEMRI\b", r"MRI[- ]compatible|MRI (safe|conditional)", r"(review|interpret|read|second opinion|bring|carry|previous|old|your|past|prior|based on|needs?|require[sd]?|advis\w+|prescrib\w+)\s+(an?\s+|the\s+|your\s+)?(X[- ]?ray[s,/ ]*|CT[s,/ ]*|and |or )*MRI"],
  "spec_direct": r"^mri|magneticres",
  "spec_related": r"^(diagnostic)?radiology$|neuroradiol|musculoskeletalradiol|clinicalimaging",
  "coreq": {"radiologist": r"radiolog", "CT": r"\bCT\b|computed tomograph", "tesla_spec": r"\b(1\.5|3(\.0)?)\s?-?\s?T(esla)?\b", "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?"},
 },
 "burn_care": {
  "label": "Burn care",
  "infra": [r"burns? (unit|ward|cent(re|er)|ICU|icu|department|care unit|theat|OT)", r"burns? (and|&) plastic surgery (unit|ward|department|cent)", r"\b\d+[- ]?bed(ded)? burns?"],
  "proc": [r"burns? (care|management|treatment|surgery|dressing|injur|reconstruction|rehabilitation|patients?|victims?|cases)", r"(treat(s|ment|ing)?|manag(es|ement|ing)|care) (of|for) (\w+ ){0,2}burns?\b",
           r"(skin graft|escharotomy|debridement).{0,40}burn|burn.{0,40}(skin graft|escharotomy|debridement)", r"post[- ]burn (contracture|deformit|reconstruct|scar)",
           r"(thermal|electric(al)?|chemical|acid|scald) burns?", r"burns? (and|&) (trauma|plastic|reconstructive)"],
  "generic": [r"\bburns?\b"],
  "exclude": [r"heart ?burn", r"burning (sensation|micturition|feet|mouth|pain|eyes?|urination|urine|stool)", r"burn[- ]?out", r"(fat|calorie)s?[- ]burn|burn(s|ing)? (fat|calories)", r"sun ?burn", r"no burns?|without burns?",
              r"(?-i:Burn(s|ham|ett|side|ley|pur|hall)\b)", r"burn (scar|mark)s? (removal|laser|revision)|razor burn|side ?burn"],
  "spec_direct": r"burn",
  "spec_related": r"^plasticsurgery$|reconstructiveplastic",
  "coreq": {"plastic_surgeon": r"plastic (surg|and recon)", "ICU": r"\bICU\b|intensive care", "skin_graft_or_bank": r"skin (graft|bank)", "operation_theatre": r"operation theat(re|er)|\bOT\b|operating (room|theat)",
            "blood_bank": r"blood (bank|storage|cent(re|er))", "round_the_clock_emergency": r"24\s?[x/*×]\s?7.{0,30}emergen|emergen.{0,30}24\s?[x/*×]\s?7|casualty"},
 },
 "trauma_emergency": {
  "label": "Trauma / emergency care",
  "infra": [r"trauma (cent(re|er)|unit|ward|ICU|bay|theat(re|er)|OT|care (cent(re|er)|unit|facility))", r"emergency (department|room|ward|unit|wing|block|OT|theat(re|er)|bay|beds?|cent(re|er))",
            r"\bcasualty\b", r"accident (and|&) emergency|\bA ?& ?E\b (department|unit)", r"\bER\b (department|beds?|bay)|\bED\b beds", r"level[- ]?(1|2|3|I|II|III)\b trauma", r"triage (area|bay|room|desk)|resuscitation (bay|room|area)",
            r"emergency (and|&) trauma (cent(re|er)|unit|department|care)", r"\b\d+[- ]?bed(ded)? (emergency|trauma|casualty)"],
  "proc": [r"24\s?[x/*×]\s?7 (emergency|trauma|casualty|accident)", r"(emergency|trauma|casualty|accident)[^.;]{0,40}24\s?[x/*×]\s?7", r"(24[- ]?h(ou)?rs?|round[- ]the[- ]clock) (emergency|trauma|casualty|accident)",
           r"(emergency|trauma)[^.;]{0,40}(24[- ]?h(ou)?rs?|round[- ]the[- ]clock)", r"poly[- ]?trauma", r"trauma (care|surgery|services?|management)", r"(road (traffic )?accident|\bRTA\b|accident) (cases|care|victims|trauma|and trauma|emergenc)",
           r"emergency (medicine|surger|care|services?|treatment|response)", r"head injur|neuro[- ]?trauma", r"\bATLS\b|\bACLS\b|\bBLS\b", r"medico[- ]?legal|\bMLC\b"],
  "generic": [r"\btrauma\b", r"\bemergenc(y|ies)\b"],
  "exclude": [r"(dental|tooth|teeth|eye|ocular|ophthalm\w*|psycholog\w*|emotional|childhood|birth|mental|sexual|relationship)\s+(trauma|emergenc\w+)", r"trauma[- ](informed|counsel+ing|therapy|focused)|trauma (counsel+ing|therapy|healing)|\bPTSD\b",
              r"emergency (contraception|contraceptive|pill|contact|number|exit|lighting|light|fund|loan|kit)", r"in case of (an? )?emergency|emergency (phone|call|helpline|no\.?)\b",
              r"emergency (dental|eye|dentist)|dental emergency|(?-i:EMRI)"],
  "spec_direct": r"^emergencymedic|accidentandemerg|polytrauma|orthopedictrauma|neurotrauma|^emergencymedicalserv|^pediatricemergency",
  "spec_related": r"criticalcare|^generalsurgery$|orthop(a)?edic(surgery|s)$",
  "coreq": {"round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?|round[- ]the[- ]clock", "ICU": r"\bICU\b|intensive care", "operation_theatre": r"operation theat(re|er)|\bOT\b|operating (room|theat)",
            "ambulance": r"ambulance", "blood_bank": r"blood (bank|storage|cent(re|er))", "CT": r"\bCT\b|computed tomograph", "ventilator": r"ventilat(?!ed\b)(?!ion system)",
            "surgeon_ortho_or_neuro": r"orthop|neurosurg|general surg"},
 },
 "ct_scan": {
  "label": "CT scan",
  "infra": [r"\bCT (scanner|machine|system|suite|unit|cent(re|er)|facility|scan (cent|machine|facility|unit))", r"\b(\d{1,3})[- ]?slice", r"multi[- ]?(slice|detector)|\bMDCT\b|dual[- ]source|spiral CT|helical CT",
            r"(Siemens|\bGE\b|Philips|Toshiba|Canon|Somatom|Revolution|Aquilion|Optima|Ingenuity|Brilliance).{0,25}\bCT\b|\bCT\b.{0,25}(Siemens|\bGE\b|Philips|Toshiba|Somatom|Aquilion)"],
  "proc": [r"\bCT\b(?![- ]?VS)", r"computed tomograph", r"\bHRCT\b|\bCECT\b|\bNCCT\b|\bCAT scan", r"\bCT[- ]guided", r"CT (angio|coronary|enterograph|urograph)"],
  "generic": [],
  "exclude": [r"\bPET[- /]?CT\b|\bSPECT[- /]?CT\b", r"cone[- ]beam (CT|computed tomograph\w*)|\bCBCT\b|dental CT|\bOCT\b|optical coherence tomograph\w*", r"\bCT ?VS\b|\bCT surg|\bCT ICU\b",
              r"(review|interpret|read|second opinion|bring|carry|previous|old|your|past|prior|based on|needs?|require[sd]?|advis\w+|prescrib\w+)\s+(an?\s+|the\s+|your\s+)?(X[- ]?ray[s,/ ]*|MRI[s,/ ]*|and |or )*CT\b",
              r"(?-i:\bCT\b (Road|Nagar|Colony|Scheme))|(?-i:\bCt\b)"],
  "spec_direct": r"^ctscan|computedtomog",
  "spec_related": r"^(diagnostic)?radiology$|neuroradiol|cardiothoracicradiol|clinicalimaging",
  "coreq": {"radiologist": r"radiolog", "MRI": r"\bMRI\b", "slice_spec": r"\b\d{1,3}[- ]?slice", "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?"},
 },
 "icu": {
  "label": "ICU / critical care",
  "infra": [r"\bICU\b", r"\bICCU\b|\bCCU\b|\bMICU\b|\bSICU\b|\bHDU\b|\bRICU\b|\bNSICU\b|\bCTICU\b|\bITU\b", r"intensive (care|cardiac care|coronary care|therapy) unit", r"critical care (unit|ward|beds?|department|block)",
            r"high dependency unit", r"\bICU beds?\b|intensive care beds?", r"multi[- ]?para(meter)? monitors?"],
  "proc": [r"intensive care", r"critical care (medicine|services?|management|support|team)", r"intensivist", r"life support"],
  "generic": [r"critical care", r"critically ill"],
  "exclude": [r"\bICU ambulance|ambulance.{0,15}\bICU\b|mobile ICU", r"basic life support|advanced (cardiac )?life support"],
  "spec_direct": r"criticalcare|intensivecare|^icu",
  "spec_related": r"^an(a)?esthesi|^emergencymedic|^pulmonology$",
  "coreq": {"ventilator": r"ventilat(?!ed\b)(?!ion system)", "monitors": r"(multi[- ]?para\w*|patient|bed[- ]?side|cardiac|vital\w*( signs?)?|ICU|central|ECG|invasive|h(a)?emodynamic)[- ]monitor|monitors? (in|for|at) (the |each |every )?(ICU|bed)|\bmonitors\b(?! for (blood|glucose))", "intensivist_or_anaesthetist": r"intensivist|ana?esthe|critical care (specialist|physician)", "oxygen": r"oxygen|\bO2\b",
            "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?|round[- ]the[- ]clock", "defibrillator": r"defibrillat", "bed_count": r"\d+[- ]?(bed(ded)?s?)? ?\bICU\b|\bICU\b beds?:? ?\d+"},
 },
 "blood_bank": {
  "label": "Blood bank / blood storage",
  "infra": [r"blood bank", r"blood (storage|cent(re|er)|component)( unit| facility| separation)?", r"component separat", r"blood bank refrigerator|apheresis (machine|unit)|\bSDP\b"],
  "proc": [r"blood transfusion", r"(platelet|plasma|packed (red )?cell|\bPRBC\b|\bFFP\b|cryoprecipitate)s? (transfusion|available|concentrate|apheresis|donation)", r"transfusion medicine", r"apheresis"],
  "generic": [r"blood donation", r"transfusion"],
  "exclude": [r"(cord|stem cell|eye|skin|milk|sperm|bone|tissue) bank", r"cord blood", r"no other blood banks", r"exchange transfusion"],
  "spec_direct": r"transfusionmed|bloodbank",
  "spec_related": r"^h(a)?ematology$|^pathology$|clinicalpathology",
  "coreq": {"licence": r"licen[cs]e|\bFDA\b|\bNACO\b|\bNBTC\b|\bSBTC\b", "component_separation": r"component", "pathologist": r"patholog|transfusion medicine", "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?|round[- ]the[- ]clock"},
 },
 "cataract_surgery": {
  "label": "Cataract surgery",
  "infra": [r"phaco(emulsification)? (machine|system|unit)", r"\b(Centurion|Infiniti|Stellaris|Catalys|LenSx|Victus|Oertli|WhiteStar)\b", r"femto(second)?[- ]?(laser)?[- ]?(assisted)? ?cataract|\bFLACS\b",
            r"\bIOL ?Master\b|optical biometer"],
  "proc": [r"cataract (surger|operation|extraction|removal|procedure)", r"phaco(emulsification)?\b", r"\bSICS\b|\bECCE\b|small incision cataract|\bMICS\b.{0,30}(cataract|phaco|IOL)|(cataract|phaco).{0,30}\bMICS\b", r"(intra[- ]?ocular lens|\bIOL\b)[^.;]{0,25}(implant|surgery|insertion)|(implant\w*)[^.;]{0,25}(intra[- ]?ocular lens|\bIOL\b)",
           r"(bladeless|laser|robotic|micro[- ]?incision|stitchless|no[- ]stitch) cataract", r"cataract surgeon"],
  "generic": [r"cataract", r"intra[- ]?ocular lens|\bIOLs?\b"],
  "exclude": [r"cataract (screening|detection|awareness) camp"],
  "spec_direct": r"cataract",
  "spec_related": r"^ophthalmology$|anteriorsegment",
  "coreq": {"ophthalmologist": r"ophthalm|eye (surgeon|specialist)", "operation_theatre": r"operation theat(re|er)|\bOT\b|operating (room|theat)", "biometry": r"biometr|A[- ]scan|IOL ?Master|keratomet", "operating_microscope": r"(operating|surgical|ophthalmic) microscope", "YAG_laser": r"\bYAG\b",
            "phaco_machine": r"phaco", "volume": r"\d[\d,]*\+?\s*(cataract )?(surger|operation)"},
 },
 "chemotherapy": {
  "label": "Chemotherapy / medical oncology day care",
  "infra": [r"chemo(therapy)? (day ?care|ward|unit|suite|lounge|beds?|chairs?|cent(re|er)|daycare)", r"day ?care chemo", r"(oncology|cancer) (day ?care|ward|wing|block|unit)", r"bio[- ]?safety cabinet|cytotoxic (drug )?(mixing|admixture|cabinet)|chemo ?port"],
  "proc": [r"chemo[- ]?therap", r"\bchemo\b", r"targeted therap|immunotherap(y|ies) (for|in) cancer|hormon(e|al) therapy for (breast|prostate|cancer)", r"chemo[- ]?(embolization|radiation|immuno)", r"\bHIPEC\b|\bPIPAC\b|intrathecal chemo"],
  "generic": [r"medical oncolog", r"cancer (care|treatment|hospital|cent(re|er)|institute)", r"oncolog", r"bone marrow transplant|\bBMT\b|stem cell transplant"],
  "exclude": [r"chemotherapy is not effective|not respon\w+ to chemo", r"chemo(nucleolysis|cautery|surgery|preventi)", r"chemical peel"],
  "spec_direct": r"chemotherap|^medicaloncolog|h(a)?emato?oncolog|clinicaloncolog",
  "spec_related": r"oncolog|cancercare",
  "coreq": {"medical_oncologist": r"medical oncolog|oncologist|\bDM\b.{0,12}oncolog", "day_care": r"day ?care", "pharmacy": r"pharmac", "lab": r"patholog|laborator", "tumour_board": r"tumou?r board", "radiotherapy": r"radiotherap|radiation (therap|oncolog)|\bLINAC\b"},
 },
 "radiotherapy": {
  "label": "Radiotherapy / radiation oncology",
  "infra": [r"linear accelerator|\bLINAC\b", r"\b(True ?Beam|Halcyon|Versa ?HD|Elekta|Varian|Novalis|Tomo ?therapy|Cyber ?Knife|Gamma ?Knife|Clinac|Bhabhatron)\b", r"cobalt[- ]?(60)?( tele(therapy)?)? (unit|machine)|telecobalt",
            r"brachytherapy (unit|suite|machine|system)|\bHDR\b brachy", r"radio(therapy)? (bunker|vault)|simulator.{0,20}radio|CT simulator|treatment planning system|\bTPS\b"],
  "proc": [r"radio[- ]?therap", r"radiation (therap|treatment|oncolog)", r"brachytherap", r"\bIMRT\b|\bIGRT\b|\bVMAT\b|\bSBRT\b|\bSRS\b|\bSRT\b|\b3D[- ]?CRT\b|rapid ?arc|stereotactic (radio|body)", r"proton (beam )?therap", r"tele(therapy|cobalt)"],
  "generic": [r"radiation oncolog", r"\bradiation\b"],
  "exclude": [r"radiation (safety|protection|dose|exposure|free|hazard|badge)|low[- ]radiation|less radiation|no radiation|radiation[- ]free|without radiation", r"radio ?frequency|radio ?surgery (for|of) (skin|piles|wart)|radio ?iodine"],
  "spec_direct": r"radiationoncolog|radiotherap",
  "spec_related": r"oncolog|nuclearmedicine",
  "coreq": {"radiation_oncologist": r"radiation oncolog|radiotherapist", "linac_or_cobalt": r"linear accelerator|\bLINAC\b|cobalt", "physicist": r"physicist", "AERB": r"\bAERB\b", "planning": r"treatment planning|simulator|\bTPS\b", "chemotherapy": r"chemo"},
 },
 "cath_lab": {
  "label": "Cath lab / angioplasty (PCI)",
  "infra": [r"cath[- ]?lab", r"cardiac cath(eteri[sz]ation)? ?(lab(oratory)?|suite|unit)", r"catheteri[sz]ation lab", r"(flat[- ]panel|bi[- ]?plane) (cath|angio)", r"(Artis|Allura|Azurion|Innova|Zee)\b.{0,25}(cath|angio)|\bDSA\b (lab|suite|machine)", r"hybrid (OT|cath|operating)"],
  "proc": [r"angioplast", r"\bPTCA\b|\bPCI\b|primary PCI|\bPAMI\b", r"(coronary|cardiac|peripheral|carotid|renal) (angiogra|stent)", r"(coronary|cardiac) catheteri[sz]ation", r"\bCAG\b", r"(pacemaker|\bICD\b|\bCRT[- ]?[DP]?\b) (implant|insertion)|permanent pacemaker|\bPPI\b|\bTPI\b",
           r"\bTAVI\b|\bTAVR\b|device closure|balloon (valvotomy|valvuloplasty|mitral)|\bBMV\b|\bPTMC\b", r"electrophysiolog\w+ (stud|lab)|radiofrequency ablation.{0,20}(arrhythm|cardiac)|\bEPS\b (and|&|/) \bRFA\b", r"\bFFR\b|\bIVUS\b|rotablat"],
  "generic": [r"angiogra", r"\bstent", r"interventional cardiolog", r"pacemaker"],
  "exclude": [r"(fundus|fluorescein|retinal|\bFFA\b|\bICG\b|indocyanine|\bOCT\b)[- ]?(fluorescein )?angiogra\w*|\bFFA\b", r"(CT|MR|MRI)[- ]?(coronary )?angiogra\w*", r"(ureter(ic|al)|biliary|\bDJ\b|double[- ]J|urethral|prostatic|esophageal|oesophageal|colonic|tracheal|airway|lacrimal|nasal|sinus|\bCBD\b|pancreatic) stent\w*|stent (removal|exchange)",
              r"without (angioplasty|stent)|avoid(ing)? (angioplasty|stent)"],
  "spec_direct": r"interventionalcardiolog|cathlab|electrophysiolog",
  "spec_related": r"^cardiology$|^pa?ediatriccardiology$|interventionalradiol",
  "coreq": {"cardiologist": r"cardiolog", "cardiac_ICU": r"\bICCU\b|\bCCU\b|cardiac (ICU|intensive|care unit)|coronary care", "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?|round[- ]the[- ]clock",
            "echo_TMT": r"\b(2D )?echo|\bTMT\b|treadmill", "cardiac_surgery_backup": r"\bCABG\b|bypass surg|cardiac surg|\bCTVS\b", "volume": r"\d[\d,]*\+?\s*(angioplast|angiograph|cardiac procedures|PCI)"},
 },
 "ventilator": {
  "label": "Ventilator / respiratory support",
  "infra": [r"\bventilators?\b", r"ventilator[- ]?(beds?|support|care|equipped)", r"\bECMO\b", r"\bBi[- ]?PAP\b|\bHFNC\b|high[- ]flow nasal|\bHFOV\b|high[- ]frequency oscillat", r"(invasive|non[- ]?invasive|mechanical) ventilat"],
  "proc": [r"mechanical ventilation|ventilatory support|ventilator management|respiratory support|\bNIV\b", r"\bCPAP\b"],
  "generic": [r"ventilat(ion|ed)\b", r"oxygen (support|therapy)"],
  "exclude": [r"(well|properly|naturally|cross)[- ]ventilated|ventilation system|air ventilation|ventilated (room|ward|space|hall|building|waiting)s?", r"\bCPAP\b.{0,30}(sleep|apnea|apnoea|titration)|(sleep|apnea|apnoea).{0,30}\bCPAP\b", r"ventilat\w+ ambulance|ambulance.{0,30}ventilat"],
  "spec_direct": r"^$a",
  "spec_related": r"criticalcare|^pulmonology$|^an(a)?esthesi",
  "coreq": {"ICU": r"\bICU\b|intensive care|\bNICU\b|\bPICU\b", "intensivist_or_anaesthetist": r"intensivist|ana?esthe|critical care", "oxygen": r"oxygen|\bO2\b", "ABG": r"\bABG\b|blood gas", "ventilator_count": r"\d+\s*(\w+\s){0,2}ventilators?"},
 },
 "stroke": {
  "label": "Stroke care / thrombolysis",
  "infra": [r"stroke (unit|cent(re|er)|ICU|ward|ready|team|clinic|program(me)?)", r"neuro[- ]?(ICU|intensive care|cath ?lab|critical care unit)|\bNSICU\b|\bNICU \(neuro", r"(comprehensive|primary|advanced) stroke"],
  "proc": [r"thromboly(sis|tic)", r"(mechanical )?thrombectomy", r"(?-i:\b(r?tPA|rt-PA|IV-tPA)\b)|alteplase|tenecteplase", r"(acute|brain|isch(a)?emic|ha?emorrhagic) stroke", r"stroke (care|management|treatment|intervention|emergenc)",
           r"golden hour.{0,30}stroke|stroke.{0,30}golden hour|\bcode stroke\b"],
  "generic": [r"\bstroke\b", r"paralysis", r"stroke rehab", r"(aneurysm|\bAVM\b) (coiling|clipping|embolization)|carotid (stenting|endarterectomy)"],
  "exclude": [r"heat[- ]?stroke|sun[- ]?stroke", r"stroke (rehabilitation|rehab|recovery|physiotherapy|exercises?)|post[- ]stroke", r"(?-i:\bTPA\b)|third[- ]party", r"deep vein thromb\w+.{0,30}thrombolys|catheter[- ]directed thrombolys|peripheral thrombolys"],
  "spec_direct": r"vascularneurolog|stroke|interventionalneuro|cerebrovascular",
  "spec_related": r"^neurology$|^neurosurgery$|neurocritical",
  "coreq": {"neurologist": r"neurolog", "CT_or_MRI": r"\bCT\b|\bMRI\b", "ICU": r"\bICU\b|intensive care", "round_the_clock": r"24\s?[x/*×]\s?7|24[- ]?h(ou)?rs?|round[- ]the[- ]clock", "neurosurgery": r"neuro[- ]?surg", "cath_lab_or_DSA": r"cath[- ]?lab|\bDSA\b|neuro[- ]?intervention"},
 },
 "snakebite_poison": {
  "label": "Snakebite / antivenom / poisoning",
  "infra": [r"anti[- ]?(snake[- ]?)?venom|\bASV\b|anti[- ]?venin", r"poison (cent(re|er)|control|information|unit|ward)", r"toxicology (unit|cent(re|er)|ward|ICU)"],
  "proc": [r"snake[- ]?bites?", r"envenom", r"scorpion (sting|bite)", r"(organophosph\w+|\bOP\b|pesticide|insecticide|rodenticide|paraquat|drug|alcohol|food|chemical|corrosive) poison", r"poisoning (cases|management|treatment|care|emergenc)|(treat\w*|manag\w*) (of |for )?poison", r"gastric lavage|stomach wash",
           r"(dog|animal|monkey)[- ]?bites?|anti[- ]?rabies|\bARV\b|rabies (vaccin|immunoglobulin|clinic)"],
  "generic": [r"poison", r"toxicolog", r"\bbites?\b", r"rabies"],
  "exclude": [r"food poisoning (test|prevention)", r"insect bite allerg|bite (correction|plane|block|guard|registration|problems?|alignment)|(deep|open|cross|over|under)[- ]?bite|\bbite[- ]?wing", r"blood poisoning"],
  "spec_direct": r"toxicolog",
  "spec_related": r"^emergencymedic|^internalmedicine$|criticalcare",
  "coreq": {"ICU": r"\bICU\b|intensive care", "ventilator": r"ventilat(?!ed\b)(?!ion system)", "round_the_clock_emergency": r"24\s?[x/*×]\s?7|casualty|emergency (department|room|ward)", "dialysis_backup": r"dialysis", "physician": r"general medicine|internal medicine|physician"},
 },
 "tb": {
  "label": "Tuberculosis diagnosis / treatment",
  "infra": [r"\bDOTS?\b (cent(re|er)|clinic|plus|site)|designated microscopy", r"\bCB[- ]?NAAT\b|gene ?xpert|true ?nat|line probe|\bLPA\b lab|\bMGIT\b|TB culture", r"\bTB\b (ward|hospital|clinic|unit|cent(re|er)|sanatorium)|tuberculosis (ward|hospital|clinic|unit|cent(re|er)|sanatorium)",
            r"\bDR[- ]?TB\b (cent(re|er)|ward)|\bNTEP\b|\bRNTCP\b|\bNikshay\b|\bART\b (cent(re|er)) .{0,20}\bTB\b"],
  "proc": [r"tubercul", r"(?-i:\bT\.?B\.?\b)(?! (Road|Nagar|Colony))", r"\bMDR[- ]?TB\b|\bXDR\b|anti[- ]?tubercul|\bATT\b|\bAKT\b", r"\bDOTS\b", r"sputum (AFB|microscopy|smear|test)|\bAFB\b|mantoux|\bIGRA\b|quantiferon", r"chest (and|&) TB|TB (and|&) chest|pulmonary koch"],
  "generic": [r"chest (medicine|physician|diseases)", r"pulmonolog", r"respiratory (medicine|diseases)"],
  "exclude": [r"(?-i:\bTb\b)|terabyte|\bTB (Road|Rd)\b", r"(?-i:T\.B\. (Road|Hospital Road))"],
  "spec_direct": r"tubercul|chestandtb|tbandchest",
  "spec_related": r"^pulmon|respiratorymed|chestmed|^infectiousdiseases$",
  "coreq": {"pulmonologist": r"pulmonolog|chest physician|respiratory medicine", "xray": r"x[- ]?ray|radiograph", "lab": r"patholog|laborator|microbiolog", "sputum_or_naat": r"sputum|\bAFB\b|\bCB[- ]?NAAT\b|gene ?xpert|true ?nat", "DOTS": r"\bDOTS?\b|\bNTEP\b|\bRNTCP\b"},
 },
 "ortho_surgery": {
  "label": "Orthopaedic surgery (joint replacement / fracture fixation)",
  "infra": [r"\bC[- ]?arm\b|image intensifier", r"ortho(p(a)?edic)? (OT|operation theat(re|er)|operating (room|theat))", r"laminar (air ?)?flow (OT|operation|theat)", r"arthroscop(y|ic) (system|tower|unit|equipment|set)", r"(robotic|navigation|computer[- ]?(assisted|navigated))[- ]?(arm|system)?.{0,25}(knee|hip|joint|replacement)|\b(MAKO|CORI|ROSA|Cuvis|Velys)\b",
            r"fracture table|traction table|bone bank"],
  "proc": [r"(total |partial |revision |uni\w* )?(knee|hip|joint|shoulder|elbow|ankle) (replacement|arthroplast)", r"\bTKR\b|\bTHR\b|\bTKA\b|\bTHA\b|\bUKR\b", r"arthroplast", r"arthroscop", r"fracture (fixation|surger|management|treatment|care)|(orthop(a)?edic )?trauma surger|\bORIF\b|\bCRIF\b|internal fixation|intramedullary|interlock(ing)? nail|\bplating\b|ilizarov|external fixat",
           r"spine surger|spinal (fusion|surger|instrumentation)|discectomy|laminectomy", r"\bACL\b (reconstruction|repair|surgery)|ligament (reconstruction|repair)|meniscus (repair|surgery)", r"orthop(a)?edic (surger|trauma|operations?)", r"hemiarthroplast"],
  "generic": [r"orthop(a)?edic", r"\bfractures?\b", r"bone (and|&) joint", r"\bortho\b"],
  "exclude": [r"orthodont\w*|dentofacial orthop\w*|orthognath\w*|ortho[- ]?k\b|orthoptic\w*|orthotic\w*|(jaw|TMJ|temporomandibular)[- ]?joint (replacement|surgery)?", r"(tooth|teeth|dental|root|crown|jaw|nasal|facial|orbital|mandib\w+|maxill\w+) fractures?|fractured (tooth|teeth)"],
  "spec_direct": r"jointrecon|jointreplac|arthroscop|orthop(a)?edic(spine|trauma|sports|surgery)|shoulderandelbow|footandankle|spinesurg|^orthop(a)?edics?$|orthop(a)?edicsand",
  "spec_related": r"^orthop|sportsmedicine|spineneurosurg",
  "coreq": {"operation_theatre": r"operation theat(re|er)|\bOT\b|operating (room|theat)", "C_arm": r"\bC[- ]?arm\b|image intensifier", "anaesthesia": r"ana?esthe", "physiotherapy": r"physio", "xray": r"x[- ]?ray", "orthopaedic_surgeon": r"orthop(a)?edic surgeon|\bMS\b.{0,8}orth|\bD\.? ?Ortho", "implants_volume": r"\d[\d,]*\+?\s*(joint|knee|hip) replacement|\d[\d,]*\+?\s*(orthop\w+ )?surger"},
 },
}

# ---- context flags (applied to the segment containing the hit; window = text around the hit) ----
CONTEXT = {
 # hit is denied: "No MRI facility", "NICU facility not available", "does not perform bypass surgery", "CT scan ... non-functional"
 "neg_pre":  r"(?:\b(?:no|not|without|lacks?|lacking|absence of|does ?n[o']?t|do(?:es)? not|cannot|can't|never|neither|nor|non[- ]availability of|shortage of|lack of)\b(?!\.)(?:[ ,/-]+[\w'&()-]+){0,6}[ ,/-]*)$",
 "neg_post": r"^(?:[ ,/-]*[\w'&()/-]+){0,6}[ ,:-]*\b(?:(?:is|are|was|were|has been|have been|remains?)\s+)?(?:not\s+(?:\w+\s+)?(?:available|functional|operational|working|provided|offered|performed|done|present|there|yet|in[- ]house|on[- ]site)|non[- ]?(?:functional|operational|existent)|unavailable|out of order|defunct|closed down|shut down|lying (?:idle|unused|defunct)|discontinued|suspended)",
 # false negation cues to ignore
 "neg_ignore": r"\bno\.? ?\d|\bno\. of|\bnumber|no[- ](?:stitch|injection|bandage|pad|patch|needle|pain|scar|blade|touch|cut|suture|incision|wait|cost|extra|additional|compromise|admission|hidden)|not only|not just|not limited|no other|no more|no longer need|not[- ]for[- ]profit|with or without|not (?:be )?(?:required|needed|necessary)|(?:no|not)\b[^.;]{0,60}\b(?:required|needed|necessary)|without (?:any )?(?:delay|compromis|pain|stitch|cut|incision|hassle|wait|scar|admission|extra)",
 # LLM extraction meta-statements: "No specific procedures ... listed in the provided content"
 "meta_absent": r"(?:not|no|none|n't)\b[^.;]{0,80}\b(?:listed|mentioned|specified|described|stated|indicated|found|provided|detailed|explicit\w*)\b|in the provided (?:content|text|source|information)|no explicit|not explicitly|unclear (?:whether|if)|wait no",
 # service is elsewhere / borrowed
 "referral": r"\brefer(?:s|red|ring)?\b[^.;]{0,50}\b(?:to|out|elsewhere|externally|higher)\b|\breferr?(?:ed|als?) (?:to|out)\b|outsourc|\btie[- ]?ups?\b|\btied[- ]up\b|\bvia [^.;]{0,25}(?:partner|tie|network)|referred externally|\bexternal (?:lab|provider|cent|facility|agency|diagnostic|vendor)|\bnearby\b|\bnearest\b|\bpartner (?:hospital|lab|diagnostic|cent)|arranged (?:from|through|on)|empanel+ed (?:lab|diagnostic|cent)",
 "referral_ignore": r"referral (?:cent(?:re|er)|hospital|unit|point|institut)|(?:tertiary|secondary|quaternary|apex|high(?:er)?|major|regional)[- ]referral|receives? [^.;]{0,30}referr|referred (?:by|from)|referrals? from|tie[- ]?up hospital",
 # not yet there
 "future": r"\bplans? to\b|\bplanned\b|\bplanning to\b|\bproposed\b|\bupcoming\b|\bcoming soon\b|\bunder construction\b|\bwill (?:be|soon|start|have|open|offer|provide|get|house|include)\b|\bto be (?:set up|commissioned|installed|started|established|built|added|opened|launched)\b|in the pipeline|foundation stone|\(future\)|\bfuture\b[^.;]{0,20}(?:cath|unit|wing|block|cent)|\bsoon\b|sanction(?:ed)? for|yet to (?:start|begin|be)|aims? to (?:start|set|open|establish|add)",
 # doctor biography sentence (person evidence, not infrastructure)
 "bio": r"\bDr\.?\s+[A-Z]|\bDoctor [A-Z]|\bMBBS\b|\bM\.B\.B\.S|\bfellowship\b|\btrained (?:at|in|under)\b|\bworked (?:at|as|in|with)\b|\byears? of (?:\w+ )?experience|\bexperience (?:in|of)\b|\bformerly\b|\bcompleted (?:his|her|their)\b|\b(?:he|she) (?:is|has|was|did|completed|specializ)|\bspecial interest|\bemployed\b|\bconsultant (?:at|in|with)\b|\balumn|\bpioneer(?:ed)? in\b|\bhis (?:area|expertise)|\bher (?:area|expertise)",
 # third-party directory / ranking boilerplate, or sentence about another facility
 "directory": r"\blisted (?:as|in|among|on|under|at)\b|\bis a nearby\b|\bnearby hospitals?\b|\b(?:top|best) \d*\s*(?:[\w&-]+\s){0,5}(?:hospitals?|clinics?|doctors?|cent(?:re|er)s?) (?:in|near|of)\b|\b(?:justdial|practo|lybrate|sulekha|quickobook|medindia|zmedhealth|credihealth|google) ?(?:'s|rating|review|listing|directory|profile)?\b|\brat(?:ed|ing)\b[^.;]{0,15}\d(?:\.\d)?|\breviews?\b|dropdown|search results?",
 # news / incident / allegation text
 "news": r"\balleg(?:ed|ation)|\bnegligen|\bfire\b|\bdied\b|\bdeath of\b|\bprotest|\bpolice\b|\bcourt\b|\bFIR\b|\binquiry\b|\bcriticis|\bshortage\b|\bNMC\b|reported (?:on|in) \d|\bstrike\b",
}

# quantity / specification that makes an infrastructure claim concrete
QUANTITY = r"(?:\b\d[\d,]*\+?\s*[- ]?(?:bed(?:ded|s)?|machines?|stations?|units?|ventilators?|warmers?|incubators?|slices?|tesla|T\b|OTs?|theat(?:re|er)s?|cath ?labs?|ambulances?|linacs?|accelerators?|monitors?|chairs?|cots?|dialys(?:is|es)|sessions?|surger(?:y|ies)|procedures?|operations?|deliver(?:y|ies)|transplants?|cases|patients|angioplast\w+|angiograph\w+|replacements?|implant\w*)\b)|(?:\b(?:beds?|machines?|ventilators?|capacity|strength)\s*[:=-]?\s*\d+)|(?:\blevel[- ]?(?:[1-3]|I{1,3}[AB]?)\b)|(?:\b\d{1,3}[- ]?slice\b)|(?:\b(?:0\.[2-5]|1\.5|3(?:\.0)?)\s?-?\s?T(?:esla)?\b)"

# a facility of this type claiming this service is implausible unless evidence is strong
IMPLAUSIBLE_TYPE = {
 "dentist": {"dialysis","emergency_obstetric","nicu","cardiac_surgery","burn_care","trauma_emergency","icu","blood_bank","cataract_surgery","chemotherapy","radiotherapy","cath_lab","ventilator","stroke","snakebite_poison","tb","ortho_surgery","mri"},
 "farmacy": "ALL", "pharmacy": "ALL",
}
''')

### `pipeline/location.py`
Where each facility really is: map point, postcode and stated city compared; the majority wins.

In [ ]:
write('pipeline/location.py', r'''
"""Location agreement, rebuilt from raw data only (facilities.csv + the two India Post PIN directory files).

Three independent location signals per facility:
    L  coordinates          -> nearest post offices (7-nearest majority for state and district)
    P  postcode             -> PIN lookup (robust centre, radius, state, district)
    C  stated city / state  -> a place found by name (other facilities stating the same city, or post offices
                               and districts carrying that name inside the stated state)
Two signals "agree" when they are within a tolerance (see AGREE_* below) or name the same district.

Outputs
    pipeline/out/pin_lookup.csv   one row per PIN
    pipeline/out/location.csv     one row per facility (uid) with best coordinates, class, distances and feature columns

Nothing here reads the facility name, facilityTypeId, specialties or any research/data_work table.
Run:  python -I pipeline/location.py
"""
import sys, os, re, difflib
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd
from sklearn.neighbors import BallTree
from common import ROOT, OUT, load_facilities, haversine

R_EARTH = 6371.0
BOX = dict(la0=6.0, la1=37.5, lo0=68.0, lo1=97.5)     # India bounding box
SHARED_PINS = 3          # an exact coordinate pair used by this many different PINs is a placeholder, not a location
K_STATE = 7              # nearest post offices used for the state / district vote
AGREE_LP_NEAR = 10.0     # coordinates within this many km of ANY post office of the stated PIN -> agree
AGREE_LP_CENTRE = 10.0   # or within PIN radius + this many km of the PIN centre
AGREE_CITY = 15.0        # slack added to the city radius (and the PIN radius) when comparing with the city
NEIGHBOUR_MIN = 2         # of the 10 nearest post offices from other PINs, this many must share the first 3 PIN digits
FAR_KM = 100.0           # "different part of the country" threshold used for the plain-language flag

STATES = ["ANDAMAN AND NICOBAR ISLANDS", "ANDHRA PRADESH", "ARUNACHAL PRADESH", "ASSAM", "BIHAR", "CHANDIGARH", "CHHATTISGARH", "DELHI", "GOA",
          "GUJARAT", "HARYANA", "HIMACHAL PRADESH", "JAMMU AND KASHMIR", "JHARKHAND", "KARNATAKA", "KERALA", "LADAKH", "LAKSHADWEEP",
          "MADHYA PRADESH", "MAHARASHTRA", "MANIPUR", "MEGHALAYA", "MIZORAM", "NAGALAND", "ODISHA", "PUDUCHERRY", "PUNJAB", "RAJASTHAN", "SIKKIM",
          "TAMIL NADU", "TELANGANA", "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "TRIPURA", "UTTAR PRADESH", "UTTARAKHAND", "WEST BENGAL"]
STATE_ALIAS = {"tamilnadu": "TAMIL NADU", "tn": "TAMIL NADU", "orissa": "ODISHA", "up": "UTTAR PRADESH", "uttarpradesh": "UTTAR PRADESH", "uttaranchal": "UTTARAKHAND",
               "uttranchal": "UTTARAKHAND", "uk": "UTTARAKHAND", "pondicherry": "PUDUCHERRY", "ut of puducherry": "PUDUCHERRY", "nct of delhi": "DELHI", "nct delhi": "DELHI",
               "nct": "DELHI", "new delhi": "DELHI", "dl": "DELHI", "west delhi": "DELHI", "east delhi": "DELHI",
               "south delhi": "DELHI", "north west delhi": "DELHI", "south east delhi area": "DELHI", "jammu kashmir": "JAMMU AND KASHMIR", "jk": "JAMMU AND KASHMIR",
               "j k": "JAMMU AND KASHMIR", "kashmir": "JAMMU AND KASHMIR", "srinagar kashmir": "JAMMU AND KASHMIR", "jammu": "JAMMU AND KASHMIR", "chattisgarh": "CHHATTISGARH",
               "cg": "CHHATTISGARH", "telengana": "TELANGANA", "ts": "TELANGANA", "tg": "TELANGANA", "mh": "MAHARASHTRA", "ms": "MAHARASHTRA", "gj": "GUJARAT", "mp": "MADHYA PRADESH",
               "madhyapradesh": "MADHYA PRADESH", "madhya": "MADHYA PRADESH", "br": "BIHAR", "ap": "ANDHRA PRADESH", "wb": "WEST BENGAL", "ka": "KARNATAKA", "kl": "KERALA",
               "hr": "HARYANA", "pb": "PUNJAB", "punjab region": "PUNJAB", "rj": "RAJASTHAN", "hp": "HIMACHAL PRADESH", "jh": "JHARKHAND", "chandigarh ut": "CHANDIGARH",
               "daman and diu": "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "dadra and nagar haveli and daman and diu": "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU",
               "dadra nagar haveli daman diu": "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "dadra and nagar haveli": "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU",
               "andaman and nicobar": "ANDAMAN AND NICOBAR ISLANDS", "andaman nicobar islands": "ANDAMAN AND NICOBAR ISLANDS"}
AMBIGUOUS_STATE = {"delhi ncr", "ncr delhi", "delhi ncr region", "ncr", "central india", "ut", "india"}     # the capital region spans Delhi, Haryana and Uttar Pradesh
# old / alternative spellings of places, applied to city strings, district names and post office names alike
PLACE_ALIAS = {"bangalore": "bengaluru", "bombay": "mumbai", "calcutta": "kolkata", "madras": "chennai", "gurgaon": "gurugram", "trivandrum": "thiruvananthapuram",
               "cochin": "kochi", "baroda": "vadodara", "mysore": "mysuru", "mangalore": "mangaluru", "belgaum": "belagavi", "pondicherry": "puducherry",
               "allahabad": "prayagraj", "vizag": "visakhapatnam", "vishakhapatnam": "visakhapatnam", "calicut": "kozhikode", "trichy": "tiruchirappalli",
               "tiruchirapalli": "tiruchirappalli", "tiruchy": "tiruchirappalli", "hubli": "hubballi", "gulbarga": "kalaburagi", "bellary": "ballari", "shimoga": "shivamogga",
               "tumkur": "tumakuru", "bijapur": "vijayapura", "poona": "pune", "benares": "varanasi", "banaras": "varanasi", "new delhi": "delhi", "ahmadabad": "ahmedabad",
               "trichur": "thrissur", "quilon": "kollam", "alleppey": "alappuzha", "palghat": "palakkad", "cannanore": "kannur", "tuticorin": "thoothukudi",
               "ooty": "udhagamandalam", "kanpur nagar": "kanpur", "bhubaneshwar": "bhubaneswar", "panjim": "panaji", "simla": "shimla", "faizabad": "ayodhya",
               "mohali": "sas nagar", "sahibzada ajit singh nagar": "sas nagar", "s a s nagar": "sas nagar", "noida": "gautam buddha nagar", "kochi": "ernakulam",
               "ernakulam": "ernakulam", "vizianagaram": "vizianagaram", "rajahmundry": "rajamahendravaram", "bezawada": "vijayawada", "mehsana": "mahesana", "north cachar hills": "dima hasao", "hoogly": "hooghly", "kutch": "kachchh"}
OFFICE_SUFFIX = re.compile(r"\s*\b(b\.?\s?o\.?|s\.?\s?o\.?|h\.?\s?o\.?|g\.?\s?p\.?\s?o\.?|p\.?\s?o\.?|h\.?\s?q\.?|ndc|edbo|ed|rs|r\.s\.?)\s*$", re.I)
TAIL_WORDS = re.compile(r"\s+(east|west|north|south|central|cantt|cant|cantonment|junction|jn|road|rural|urban|extension|extn|camp)$")
DROP_WORDS = re.compile(r"\b(district|dist|distt|city|town|taluk|taluka|tehsil|mandal|post)\b")


def nplace(s):
    """Normalised place key: lower case, letters only, generic words dropped, old spellings mapped."""
    s = re.sub(r"\(.*?\)", " ", str(s).lower())
    s = re.sub(r"[^a-z ]", " ", s.replace("-", " ").replace(".", " "))
    s = re.sub(r"\s+", " ", DROP_WORDS.sub(" ", s)).strip()
    return PLACE_ALIAS.get(s, s)


def hav(la1, lo1, la2, lo2):
    return haversine(np.asarray(la1, float), np.asarray(lo1, float), np.asarray(la2, float), np.asarray(lo2, float))


def robust_centre(la, lo, floor_km=20.0):
    """Median centre after dropping points far from a first median (further than max(floor, 4 x median distance))."""
    cla, clo = np.median(la), np.median(lo)
    d = hav(la, lo, cla, clo)
    keep = d <= max(floor_km, 4 * np.median(d))
    cla, clo = np.median(la[keep]), np.median(lo[keep])
    return cla, clo, keep, hav(la, lo, cla, clo)


# ---------------------------------------------------------------- 1. PIN lookup
def load_post_offices():
    d = pd.concat([pd.read_csv(os.path.join(ROOT, f), dtype=str, keep_default_na=False) for f in ("indiaPincodeDirectory1.csv", "indiaPincodeDirectory2.csv")], ignore_index=True)
    n_raw = len(d)
    d = d.drop_duplicates().reset_index(drop=True)
    for c in ("district", "statename"):
        d[c] = d[c].str.strip().str.upper().replace({"NA": "", "NAN": ""})
    la, lo = pd.to_numeric(d.latitude, errors="coerce"), pd.to_numeric(d.longitude, errors="coerce")
    inbox = la.between(BOX["la0"], BOX["la1"]) & lo.between(BOX["lo0"], BOX["lo1"])
    swapped = ~inbox & lo.between(BOX["la0"], BOX["la1"]) & la.between(BOX["lo0"], BOX["lo1"])     # latitude and longitude typed the wrong way round
    la2, lo2 = la.where(~swapped, lo), lo.where(~swapped, la)
    d["la"], d["lo"] = la2, lo2
    d["has_raw"] = la.notna() & lo.notna()
    d["ok"] = d.la.between(BOX["la0"], BOX["la1"]) & d.lo.between(BOX["lo0"], BOX["lo1"])
    d["swapped"] = swapped
    # placeholder coordinates (a division or district default, not a location):
    #   the very same point reused by several PINs, or one precise latitude / longitude value reused by 10+ PINs
    key = d.la.round(4).astype(str) + "," + d.lo.round(4).astype(str)
    npin = d[d.ok].groupby(key[d.ok]).pincode.nunique()
    shared = d.ok & key.map(npin).fillna(0).ge(SHARED_PINS)
    for ax in ("la", "lo"):
        v = d[ax].round(6)
        n_ax = v.map(d[d.ok].groupby(v[d.ok]).pincode.nunique()).fillna(0)
        precise = (v * 1000 - (v * 1000).round()).abs() > 1e-6          # more than 3 decimals, so a repeat is not a rounding coincidence
        shared |= d.ok & ((precise & n_ax.ge(10)) | n_ax.ge(25))
    d["shared"] = shared
    # state box test: a post office must lie inside the lat / lon range of the state on its own label
    # (10th to 90th percentile of that state's post offices, widened by half that range or 1 degree; skipped for the scattered union territories)
    lab = d[d.ok & (d.statename != "")]
    q = lab.groupby("statename")[["la", "lo"]].quantile([0.1, 0.9]).unstack()
    in_box = pd.Series(True, index=d.index)
    for ax in ("la", "lo"):
        lo_, hi_ = d.statename.map(q[(ax, 0.1)]), d.statename.map(q[(ax, 0.9)])
        m = np.maximum(1.0, 0.5 * (hi_ - lo_))
        in_box &= ~((d[ax] < lo_ - m) | (d[ax] > hi_ + m))
    in_box |= d.statename.isin(["PUDUCHERRY", "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "ANDAMAN AND NICOBAR ISLANDS", "LAKSHADWEEP", ""])
    d["in_state_box"] = d.ok & in_box
    # district box test, same idea one level down (districts with 20+ located post offices; margin half the range or 0.25 degree, about 28 km).
    # It removes systematic conversion errors that push a whole town's offices 100 km away, where they would otherwise support each other.
    lab = d[d.in_state_box & (d.district != "") & (d.statename != "")]
    dk = d.statename + "|" + d.district
    g = lab.groupby(dk[lab.index])
    q = g[["la", "lo"]].quantile([0.1, 0.9]).unstack(); big = g.size() >= 20
    in_dbox = pd.Series(True, index=d.index)
    for ax in ("la", "lo"):
        lo_, hi_ = dk.map(q[(ax, 0.1)][big]), dk.map(q[(ax, 0.9)][big])
        m = np.maximum(0.25, 0.5 * (hi_ - lo_))
        in_dbox &= ~((d[ax] < lo_ - m) | (d[ax] > hi_ + m))
    d["in_district_box"] = d.in_state_box & in_dbox
    # neighbour test: PIN numbers are handed out area by area, so a post office should have neighbours from its own 3-digit postal area.
    # Among the 10 nearest post offices that belong to OTHER PINs, at least 2 must share the first 3 digits; if not, the point sits in the wrong area.
    d["p3"] = d.pincode.str[:3]
    ref = d[d.in_district_box].drop_duplicates(["pincode", "la", "lo"])
    xy = np.radians(ref[["la", "lo"]].values)
    _, idx = BallTree(xy, metric="haversine").query(xy, k=min(81, len(ref)))
    pin, p3 = ref.pincode.to_numpy(dtype=object), ref.p3.to_numpy(dtype=object)
    other = pin[idx[:, 1:]] != pin[:, None]
    first10 = other & (np.cumsum(other, axis=1) <= 10)
    n_same = ((p3[idx[:, 1:]] == p3[:, None]) & first10).sum(1)
    ref = ref.assign(sup=(first10.sum(1) == 0) | (n_same >= NEIGHBOUR_MIN))
    d = d.merge(ref[["pincode", "la", "lo", "sup"]], on=["pincode", "la", "lo"], how="left")
    d["supported"] = d.in_district_box & d["sup"].fillna(False).astype(bool)
    d["clean"] = d.ok & ~d.shared & d.supported
    stats = dict(rows_raw=n_raw, rows_distinct=len(d), no_coord=int((~d.has_raw).sum()), swapped_fixed=int(swapped.sum()), out_of_india=int((d.has_raw & ~d.ok).sum()),
                 placeholder=int(d.shared.sum()), outside_own_state=int((d.ok & ~d.in_state_box).sum()), outside_own_district=int((d.in_state_box & ~d.in_district_box).sum()), no_neighbour_from_own_postal_area=int((d.in_district_box & ~d.supported).sum()), clean=int(d.clean.sum()))
    return d, stats


def build_pin_lookup(d):
    rows = []
    lab = d[d.statename != ""]
    st_mode = lab.groupby("pincode").statename.agg(lambda s: s.value_counts().index[0])
    st_all = lab.groupby("pincode").statename.agg(lambda s: "|".join(sorted(set(s))))
    di_mode = lab[lab.district != ""].groupby("pincode").district.agg(lambda s: s.value_counts().index[0])
    di_all = lab[lab.district != ""].groupby("pincode").district.agg(lambda s: "|".join(sorted(set(s))))
    n_off = d.groupby("pincode").size(); n_raw = d[d.has_raw].groupby("pincode").size()
    pts = {}
    for pinc, g in d[d.clean].groupby("pincode"):
        la, lo = g.la.values, g.lo.values
        # the sub / head offices mark the town the PIN is named after; branch offices are villages around it and are more often mis-placed
        core = np.array(g.officetype.isin(["PO", "HO"]), dtype=bool)
        if not core.any(): core[:] = True
        cla, clo, kc, dcore = robust_centre(la[core], lo[core])
        dist = hav(la, lo, cla, clo)
        keep = dist <= max(30.0, 4 * np.median(dcore[kc]))
        dk = dist[keep]
        p90_all = float(np.percentile(dist, 90)); p90 = float(np.percentile(dk, 90)) if len(dk) > 1 else 0.0
        rows.append(dict(pincode=pinc, lat=cla, lon=clo, coord_source="pin_median" if keep.sum() >= 2 else "pin_single", n_coord=int(keep.sum()), n_pin_outlier=int((~keep).sum()),
                         p90_km=p90, max_km=float(dk.max()), p90_all_km=p90_all))
        pts[pinc] = (la[keep], lo[keep])
    anypts = {k: (g.la.values, g.lo.values) for k, g in d[d.ok].groupby("pincode")}
    P = pd.DataFrame(rows).set_index("pincode").reindex(sorted(d.pincode.unique()))
    P["n_offices"] = n_off; P["n_raw_coord"] = n_raw.reindex(P.index).fillna(0).astype(int)
    # fallbacks for PINs left without a clean point:
    #   (1) placeholder points of that PIN that pass the region test (a town or division default: right area, low precision)
    #   (2) the median of the clean PIN centres sharing the first 4, then 3, then 2 digits
    sh = d[d.ok & d.shared & d.supported & ~d.pincode.isin(set(pts))].groupby("pincode")[["la", "lo"]].median()
    P.loc[sh.index, "lat"] = sh.la; P.loc[sh.index, "lon"] = sh.lo; P.loc[sh.index, "coord_source"] = "shared_point"; P.loc[sh.index, "p90_km"] = 20.0; P.loc[sh.index, "max_km"] = 20.0
    have = P.coord_source.isin(["pin_median", "pin_single"])
    for n, tag in ((4, "prefix4"), (3, "prefix3"), (2, "prefix2")):
        pre = P.index.str[:n]
        grp = P[have].groupby(pre[have])
        cen = grp[["lat", "lon"]].median(); cnt = grp.size()
        need = P.lat.isna() & pd.Series(pre, index=P.index).map(cnt).fillna(0).ge(2)
        if need.any():
            k = pre[need]
            P.loc[need, "lat"] = cen.lat.reindex(k).values; P.loc[need, "lon"] = cen.lon.reindex(k).values
            P.loc[need, "coord_source"] = tag
            # radius of a fallback = 90th percentile distance of the member PIN centres from the prefix centre
            dd = hav(P.lat[have].values, P.lon[have].values, cen.lat.reindex(pre[have]).values, cen.lon.reindex(pre[have]).values)
            r90 = pd.Series(dd, index=pre[have]).groupby(level=0).quantile(0.9)
            P.loc[need, "p90_km"] = r90.reindex(k).values; P.loc[need, "max_km"] = r90.reindex(k).values
    P["coord_source"] = P.coord_source.fillna("none")
    P[["n_coord", "n_pin_outlier"]] = P[["n_coord", "n_pin_outlier"]].fillna(0).astype(int)
    exact = P.coord_source.isin(["pin_median", "pin_single"])
    P["approx_radius_km"] = np.where(exact, P.p90_km.clip(3, 75), np.where(P.coord_source == "shared_point", 20.0, (P.p90_km.fillna(50) + 5).clip(10, 150))).round(1)
    P.loc[P.coord_source == "pin_single", "approx_radius_km"] = 5.0
    P["state"] = st_mode.reindex(P.index); P["district"] = di_mode.reindex(P.index)
    P["states"] = st_all.reindex(P.index).fillna(""); P["districts"] = di_all.reindex(P.index).fillna("")
    P["n_states"] = P.states.map(lambda s: len(s.split("|")) if s else 0); P["n_districts"] = P.districts.map(lambda s: len(s.split("|")) if s else 0)
    # "several states" only when the second state holds at least 20% of the PIN's labelled offices (a single stray label is a typing error)
    share2 = lab.groupby("pincode").statename.agg(lambda s: (s.value_counts(normalize=True).iloc[1] if s.nunique() > 1 else 0.0))
    P["multi_state"] = (share2.reindex(P.index).fillna(0) >= 0.2).astype(int); P["multi_district"] = (P.n_districts > 1).astype(int)
    P["scattered"] = ((P.p90_all_km > 25) | (P.n_pin_outlier >= np.maximum(2, 0.25 * (P.n_coord + P.n_pin_outlier)))).fillna(False).astype(int)
    # repair missing state / district labels from the nearest clean post offices of other PINs
    c = d[d.clean & (d.statename != "") & (d.district != "")]
    tree = BallTree(np.radians(c[["la", "lo"]].values), metric="haversine")
    miss = P.state.isna() & P.lat.notna()
    P["label_repaired"] = 0
    if miss.any():
        _, idx = tree.query(np.radians(P.loc[miss, ["lat", "lon"]].values), k=K_STATE)
        cs_, cd_ = c.statename.to_numpy(dtype=object), c.district.to_numpy(dtype=object)
        P.loc[miss, "state"] = [pd.Series(cs_[i]).value_counts().index[0] for i in idx]
        P.loc[miss, "district"] = [pd.Series(cd_[i]).value_counts().index[0] for i in idx]
        P.loc[miss, "label_repaired"] = 1
    P = P.reset_index().rename(columns={"index": "pincode"})
    cols = ["pincode", "state", "district", "lat", "lon", "coord_source", "approx_radius_km", "n_offices", "n_raw_coord", "n_coord", "n_pin_outlier", "p90_km", "max_km",
            "p90_all_km", "n_states", "states", "n_districts", "districts", "multi_state", "multi_district", "scattered", "label_repaired"]
    return P[cols].round({"lat": 6, "lon": 6, "p90_km": 2, "max_km": 2, "p90_all_km": 2}), anypts


# ---------------------------------------------------------------- 2. state names
def make_state_normaliser(d, F):
    canon = {re.sub(r"[^a-z ]", " ", s.lower()).strip(): s for s in STATES}
    canon.update({"dadra and nagar haveli and daman and diu": STATES[31]})
    dist_state = d[(d.district != "") & (d.statename != "")].groupby("district").statename.agg(lambda s: sorted(set(s)))
    dist_state = {nplace(k): v[0] for k, v in dist_state.items() if len(v) == 1}
    DIRECTION = {"uttar": "north", "dakshin": "south", "dakshina": "south", "purba": "east", "purbi": "east", "paschim": "west", "pashchim": "west", "pgs": "parganas", "kutch": "kachchh"}
    tok = lambda k: " ".join(sorted(DIRECTION.get(w, w) for w in k.split()))          # word order and Hindi / English direction words do not matter
    dist_tok = {tok(k): v for k, v in dist_state.items()}
    # a city written in the state field: use the state that other rows give for that city (3+ rows, 80%+ agreement)
    tmp = pd.DataFrame({"c": F.address_city.map(nplace), "s": F.address_stateOrRegion.map(lambda x: canon.get(re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", str(x).lower())).strip()))}).dropna()
    g = tmp.groupby("c").s.agg(lambda s: (s.value_counts().index[0], s.value_counts().iloc[0], len(s)))
    city_state = {k: v[0] for k, v in g.items() if v[2] >= 3 and v[1] / v[2] >= 0.8}

    def one(part):
        k = re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", part.lower().replace("&", " and ").replace(".", ""))).strip()
        if not k: return None, "unknown"
        if k in AMBIGUOUS_STATE: return None, "ambiguous"
        if k in canon: return canon[k], "state"
        if k in STATE_ALIAS: return STATE_ALIAS[k], "alias"
        k2 = k.replace(" ", "")
        for a, v in list(canon.items()) + list(STATE_ALIAS.items()):
            if len(k2) > 3 and a.replace(" ", "") == k2: return v, "alias"
        m = difflib.get_close_matches(k, list(canon), n=1, cutoff=0.88)
        if m: return canon[m[0]], "fuzzy"
        p = nplace(k)
        if p in dist_state: return dist_state[p], "district"
        if p in city_state: return city_state[p], "city"
        t = tok(p)
        if t in dist_tok: return dist_tok[t], "district"
        m = difflib.get_close_matches(t, list(dist_tok), n=1, cutoff=0.86)
        if m: return dist_tok[m[0]], "district"
        return None, "unknown"

    def norm(raw):
        if not isinstance(raw, str) or not raw.strip(): return None, "missing"
        res = one(raw)
        if res[0] or res[1] == "ambiguous": return res
        parts = [p for p in re.split(r"[,;/]", raw) if p.strip()]
        found = [one(p) for p in parts]
        strong = [f for f in found if f[0] and f[1] in ("state", "alias", "fuzzy")] or [f for f in found if f[0]]
        if strong:
            vc = pd.Series([f[0] for f in strong]).value_counts()
            return vc.index[0], strong[0][1]
        # last resort: a state name contained in a longer string ("Satara District Maharashtra")
        k = re.sub(r"[^a-z ]", " ", raw.lower())
        for a, v in canon.items():
            if re.search(r"\b" + a + r"\b", k): return v, "contains"
        return None, "unknown"
    return norm


# ---------------------------------------------------------------- 3. city gazetteer
def build_gazetteer(d, F):
    """name key -> list of candidate places (lat, lon, radius_km, state, source)."""
    c = d[d.in_district_box & (d.supported | d.shared)].copy()      # right district; a placeholder point is still the right town, only less precise
    c["key"] = c.officename.map(lambda s: nplace(OFFICE_SUFFIX.sub("", OFFICE_SUFFIX.sub("", str(s)))))
    gaz = {}

    def add(key, la, lo, r, st, src):
        gaz.setdefault(key, []).append((float(la), float(lo), float(r), st, src))
    # (a) post offices carrying exactly this name, one candidate per (state, ~20 km cell)
    c["cell"] = (c.la / 0.2).round().astype(int).astype(str) + "_" + (c.lo / 0.2).round().astype(int).astype(str)
    for (key, st, _), g in c[c.key.str.len() >= 3].groupby(["key", "statename", "cell"]):
        add(key, g.la.median(), g.lo.median(), 5.0 if (~g.shared).any() else 15.0, st, "office")
    # (b) districts: centre = head office of the district if one carries the district name, else median of the district's post offices
    for (dist, st), g in c[(c.district != "") & ~c.shared].groupby(["district", "statename"]):
        cla, clo, keep, dist_km = robust_centre(g.la.values, g.lo.values, floor_km=60.0)
        add(nplace(dist), cla, clo, float(np.clip(np.percentile(dist_km[keep], 75), 10, 60)), st, "district")
    # (c) head / sub offices whose name starts with the city name ("Pune City H.O", "Lucknow Chowk HO")
    ho = c[c.officetype.isin(["HO", "PO"]) & ~c.shared]
    first = {}
    for key, st, la, lo in zip(ho.key, ho.statename, ho.la, ho.lo):
        w = key.split(" ")
        for n in (1, 2):
            if len(w) > n: first.setdefault((" ".join(w[:n]), st), []).append((la, lo))
    for (key, st), pts in first.items():
        if len(key) < 4: continue
        a = np.array(pts); cla, clo, keep, dist_km = robust_centre(a[:, 0], a[:, 1])
        if keep.sum() >= 2 and np.percentile(dist_km[keep], 90) <= 25:
            add(PLACE_ALIAS.get(key, key), cla, clo, float(np.clip(np.percentile(dist_km[keep], 90), 5, 25)), st, "office_prefix")
    return gaz


def facility_consensus(F):
    """Where do the OTHER facilities that state the same city and state sit? (5+ rows, robust median of their PIN centres and coordinates)."""
    cons = {}
    for (key, st), g in F[F.city_key.str.len() >= 3].groupby(["city_key", F.st_state.fillna("")]):
        la = np.concatenate([g.lat[g.ll_ok].values, g.pc_lat[g.pc_lat.notna()].values]); lo = np.concatenate([g.lon[g.ll_ok].values, g.pc_lon[g.pc_lat.notna()].values])
        if len(g) < 5 or len(la) < 6: continue
        cla, clo, keep, dist = robust_centre(la, lo, floor_km=40.0)
        if keep.mean() < 0.6: continue                        # the rows do not agree among themselves
        cons[(key, st)] = (float(cla), float(clo), float(np.clip(np.percentile(dist[keep], 75), 5, 30)), st or None, "facility_consensus", len(g))
    return cons


# ---------------------------------------------------------------- 4. per facility
def main(write=True):
    os.makedirs(OUT, exist_ok=True)
    d, st = load_post_offices()
    print("post offices:", st)
    P, pin_pts = build_pin_lookup(d)
    print("PIN lookup:", len(P), "PINs | centre source", P.coord_source.value_counts().to_dict(), "| several states", int(P.multi_state.sum()), "| several districts", int(P.multi_district.sum()),
          "| scattered", int(P.scattered.sum()), "| label repaired", int(P.label_repaired.sum()), "| median radius km", P.approx_radius_km.median())
    Pi = P.set_index("pincode")

    F = load_facilities()
    F["lat"], F["lon"] = F.latitude_n, F.longitude_n
    F["ll_ok"] = F.lat.between(BOX["la0"], BOX["la1"]) & F.lon.between(BOX["lo0"], BOX["lo1"])
    norm = make_state_normaliser(d, F)
    ns = F.address_stateOrRegion.map(norm)
    F["st_state"], F["st_src"] = ns.map(lambda x: x[0]), ns.map(lambda x: x[1])
    print("stated state:", F.address_stateOrRegion.nunique(), "spellings ->", F.st_state.nunique(), "states | how resolved:", F.st_src.value_counts().to_dict())

    # postcode -> PIN
    z = F.address_zipOrPostcode.fillna("").astype(str).str.replace(r"\D", "", regex=True)
    F["pc"] = z.where(z.str.fullmatch(r"[1-9]\d{5}"))
    F["pc_status"] = np.where(F.pc.isna(), "no_postcode", np.where(F.pc.isin(Pi.index), "found", "not_in_directory"))
    look = F.pc.where(F.pc_status == "found")
    for a, b in (("pc_lat", "lat"), ("pc_lon", "lon"), ("pc_rad", "approx_radius_km"), ("pc_state", "state"), ("pc_district", "district"), ("pc_src", "coord_source"),
                 ("pin_multi_state", "multi_state"), ("pin_scattered", "scattered")):
        F[a] = look.map(Pi[b])
    # postcode not in the directory: use the PINs sharing the first 4 (then 3) digits
    ok_pin = P[P.coord_source.isin(["pin_median", "pin_single"])]
    for n in (4, 3):
        need = (F.pc_status == "not_in_directory") & F.pc_lat.isna()
        if not need.any(): break
        g = ok_pin.groupby(ok_pin.pincode.str[:n])
        cen = g[["lat", "lon"]].median(); stt = g.state.agg(lambda s: s.value_counts().index[0]); dis = g.district.agg(lambda s: s.value_counts().index[0])
        k = F.pc[need].str[:n]; hit = k.isin(cen.index); ii = k[hit].index
        F.loc[ii, "pc_lat"] = cen.lat.reindex(k[hit]).values; F.loc[ii, "pc_lon"] = cen.lon.reindex(k[hit]).values
        F.loc[ii, "pc_state"] = stt.reindex(k[hit]).values; F.loc[ii, "pc_district"] = dis.reindex(k[hit]).values
        F.loc[ii, "pc_rad"] = 25.0 if n == 4 else 60.0; F.loc[ii, "pc_src"] = f"unknown_pin_prefix{n}"
    F["pin_multi_state"] = F.pin_multi_state.fillna(0).astype(int); F["pin_scattered"] = F.pin_scattered.fillna(0).astype(int)
    print("postcode:", F.pc_status.value_counts().to_dict(), "| PIN centre source:", F.pc_src.value_counts(dropna=False).to_dict())

    # coordinates -> nearest post offices
    c = d[d.clean & (d.statename != "") & (d.district != "")]
    tree = BallTree(np.radians(c[["la", "lo"]].values), metric="haversine")
    cst, cdi = c.statename.to_numpy(dtype=object), c.district.to_numpy(dtype=object)

    def knn(la, lo):
        dist, idx = tree.query(np.radians(np.c_[la, lo]), k=K_STATE)
        s = [pd.Series(cst[i]).value_counts() for i in idx]; di = [pd.Series(cdi[i]).value_counts().index[0] for i in idx]
        return [x.index[0] for x in s], di, [x.iloc[0] / K_STATE for x in s], dist[:, 0] * R_EARTH
    m = F.ll_ok.values
    gs, gd, gshare, nn = knn(F.lat[m].values, F.lon[m].values)
    F.loc[m, "geo_state"] = gs; F.loc[m, "geo_district"] = gd; F.loc[m, "geo_state_share"] = gshare; F.loc[m, "nn_km"] = nn
    F["ll_usable"] = F.ll_ok & (F.nn_km <= 50)             # further than 50 km from every post office = at sea or abroad

    # distance from coordinates to the PIN: centre, and nearest post office of that PIN
    F["d_coord_pin_km"] = np.where(F.ll_usable & F.pc_lat.notna(), hav(F.lat, F.lon, F.pc_lat, F.pc_lon), np.nan)
    # generous on purpose: ANY post office recorded under that PIN counts, including the ones not trusted for the centre
    near = np.full(len(F), np.nan)
    for i, (ok, pc, la, lo) in enumerate(zip(F.ll_usable, F.pc, F.lat, F.lon)):
        if ok and pc in pin_pts: near[i] = hav(pin_pts[pc][0], pin_pts[pc][1], la, lo).min()
    F["d_coord_pin_nearest_km"] = np.fmin(near, F.d_coord_pin_km)

    # city
    F["city_key"] = F.address_city.map(nplace)
    gaz = build_gazetteer(d, F); cons = facility_consensus(F)
    cons_any = {}
    for (k, st_), v in cons.items():
        if k not in cons_any or v[5] > cons_any[k][5]: cons_any[k] = v
    names_by_state = {}
    for k, v in gaz.items():
        for x in v:
            if x[4] in ("office", "district") and len(k) >= 6: names_by_state.setdefault(x[3], set()).add(k)
    for (k, st_), v in cons.items():
        if st_: names_by_state.setdefault(st_, set()).add(k)
    names_by_state = {k: sorted(v) for k, v in names_by_state.items()}
    res = []
    for key, st_, lok, la, lo, pla, plo in zip(F.city_key, F.st_state, F.ll_usable, F.lat, F.lon, F.pc_lat, F.pc_lon):
        cand = []
        s = st_ if isinstance(st_, str) else ""
        if (key, s) in cons: cand.append(cons[(key, s)][:5])
        elif not s and key in cons_any: cand.append(cons_any[key][:5])      # stated state unreadable: use the largest group of rows naming this city
        g = [x for x in gaz.get(key, []) if (not s) or x[3] == s]
        exact = [x for x in g if x[4] in ("office", "district")]
        cand += exact if exact else g
        if not cand and len(key) >= 3:
            # retry: drop a trailing "east / west / cantt / junction ..." word, then allow a one-or-two letter spelling difference (Hissar / Hisar)
            k2 = nplace(TAIL_WORDS.sub("", key))
            if k2 != key and len(k2) >= 4:
                if (k2, s) in cons: cand.append(cons[(k2, s)][:5])
                cand += [x for x in gaz.get(k2, []) if ((not s) or x[3] == s) and x[4] in ("office", "district")]
            if not cand and s and len(key) >= 6:
                m = difflib.get_close_matches(key, names_by_state.get(s, []), n=1, cutoff=0.88)
                if m:
                    if (m[0], s) in cons: cand.append(cons[(m[0], s)][:5])
                    cand += [x for x in gaz.get(m[0], []) if x[3] == s and x[4] in ("office", "district")]
            cand = [(x[0], x[1], x[2], x[3], x[4] + "_approx") for x in cand]
        if not cand and len(key) >= 3:
            res.append((np.nan, np.nan, np.nan, "not_found", 0)); continue
        if len(key) < 3:
            res.append((np.nan, np.nan, np.nan, "no_city", 0)); continue
        a = np.array([(x[0], x[1], x[2]) for x in cand])
        cost = np.zeros(len(a))
        if lok: cost += np.maximum(hav(a[:, 0], a[:, 1], la, lo) - a[:, 2], 0)
        if pd.notna(pla): cost += np.maximum(hav(a[:, 0], a[:, 1], pla, plo) - a[:, 2], 0)
        j = int(np.argmin(cost))                             # benefit of the doubt: the same-named place that fits the other signals best
        res.append((a[j, 0], a[j, 1], a[j, 2], cand[j][4], len(cand)))
    F[["city_lat", "city_lon", "city_rad", "city_src", "city_n_cand"]] = pd.DataFrame(res, index=F.index)
    print("city located by:", F.city_src.value_counts().to_dict())
    F["d_coord_city_km"] = np.where(F.ll_usable & F.city_lat.notna(), hav(F.lat, F.lon, F.city_lat, F.city_lon), np.nan)
    F["d_pin_city_km"] = np.where(F.pc_lat.notna() & F.city_lat.notna(), hav(F.pc_lat, F.pc_lon, F.city_lat, F.city_lon), np.nan)

    # pairwise agreement
    has_L, has_P, has_C = F.ll_usable, F.pc_lat.notna(), F.city_lat.notna()
    same_d_LP = (F.geo_district == F.pc_district) & F.geo_district.notna()
    LP = (F.d_coord_pin_nearest_km <= AGREE_LP_NEAR) | (F.d_coord_pin_km <= F.pc_rad + AGREE_LP_CENTRE) | (same_d_LP & (F.d_coord_pin_km <= 40))
    ck = F.city_key
    LC = (F.d_coord_city_km <= F.city_rad + AGREE_CITY) | ((F.geo_district.fillna("").map(nplace) == ck) & has_L & (ck != ""))
    PC = (F.d_pin_city_km <= F.city_rad + F.pc_rad.fillna(0) + AGREE_CITY) | ((F.pc_district.fillna("").map(nplace) == ck) & has_P & (ck != ""))
    LP, LC, PC = LP & has_L & has_P, LC & has_L & has_C, PC & has_P & has_C
    # a place found only through a post-office name can confirm a location but cannot contradict one:
    # many villages and neighbourhoods share a name inside one state ("Aundh" is a suburb of Pune and a village in Satara)
    weak_city = has_C & ~F.city_src.isin(["facility_consensus", "district"]) & ~LC & ~PC
    F["city_unverifiable"] = weak_city.astype(int)
    has_C = has_C & ~weak_city
    F["agree_coord_pin"], F["agree_coord_city"], F["agree_pin_city"] = LP.astype(int), LC.astype(int), PC.astype(int)
    nsig = has_L.astype(int) + has_P.astype(int) + has_C.astype(int)
    npair = LP.astype(int) + LC.astype(int) + PC.astype(int)
    cls = np.full(len(F), "H_cannot_tell", dtype=object)
    three = (nsig == 3).values
    cls[three & (npair >= 2).values] = "A_all_agree"
    cls[three & (npair == 1).values & PC.values] = "B_coordinates_differ"
    cls[three & (npair == 1).values & LC.values] = "C_postcode_differs"
    cls[three & (npair == 1).values & LP.values] = "D_city_differs"
    cls[three & (npair == 0).values] = "E_all_differ"
    two = (nsig == 2).values
    cls[two & (npair == 1).values] = "F_two_agree_third_missing"
    cls[two & (npair == 0).values] = "G_two_differ_third_missing"
    F["loc_class"] = cls; F["n_signals"] = nsig

    # states named by the three signals
    def nstates(r):
        s = {x for x in (r.st_state, r.pc_state, r.geo_state if r.ll_usable else None) if isinstance(x, str) and x}
        return len(s)
    F["n_states"] = F.apply(nstates, axis=1)

    # best coordinates for distance ranking
    best_src = np.where(np.isin(cls, ["A_all_agree", "C_postcode_differs", "D_city_differs"]), "coordinates", "")
    best_src = np.where(cls == "B_coordinates_differ", "postcode", best_src)
    best_src = np.where((cls == "F_two_agree_third_missing") & has_L.values, "coordinates", best_src)
    best_src = np.where((cls == "F_two_agree_third_missing") & ~has_L.values, "postcode", best_src)
    # nothing agrees: prefer the signal backed by the stated state, in the order postcode, coordinates, city
    st = F.st_state.fillna("?")
    p_ok, l_ok = (has_P & (F.pc_state == st)).values, (has_L & (F.geo_state == st)).values
    unresolved = np.isin(cls, ["E_all_differ", "G_two_differ_third_missing", "H_cannot_tell"])
    pick = np.where(p_ok, "postcode", np.where(l_ok, "coordinates", np.where(has_C.values, "city", np.where(has_P.values, "postcode", np.where(has_L.values, "coordinates", "none")))))
    best_src = np.where(unresolved, pick, best_src)
    F["best_src"] = best_src
    F["best_lat"] = np.select([best_src == "coordinates", best_src == "postcode", best_src == "city"], [F.lat, F.pc_lat, F.city_lat], np.nan)
    F["best_lon"] = np.select([best_src == "coordinates", best_src == "postcode", best_src == "city"], [F.lon, F.pc_lon, F.city_lon], np.nan)
    F["best_radius_km"] = np.select([best_src == "coordinates", best_src == "postcode", best_src == "city"], [1.0, F.pc_rad, F.city_rad], np.nan)
    hb = F.best_lat.notna().values
    bs, bd, _, _ = knn(F.best_lat[hb].values, F.best_lon[hb].values)
    F.loc[hb, "best_state"] = bs; F.loc[hb, "best_district"] = bd
    usep = (best_src == "postcode") & F.pc_state.notna().values
    F.loc[usep, "best_state"] = F.pc_state[usep]; F.loc[usep, "best_district"] = F.pc_district[usep]

    base = pd.Series(cls).map({"A_all_agree": 0.95, "F_two_agree_third_missing": 0.85, "D_city_differs": 0.80, "C_postcode_differs": 0.75, "B_coordinates_differ": 0.60,
                               "G_two_differ_third_missing": 0.30, "E_all_differ": 0.15, "H_cannot_tell": 0.40}).values.astype(float)
    base = np.where((cls == "H_cannot_tell") & (best_src == "none"), 0.0, base)
    pen = 0.10 * ((best_src == "postcode") & ((F.pin_multi_state == 1) | (F.pin_scattered == 1) | ~F.pc_src.isin(["pin_median", "pin_single"]).values)) + 0.05 * (F.n_states.values >= 2)
    F["location_confidence"] = np.clip(base - pen, 0, 1).round(2)

    # feature columns for the yardstick (all numeric)
    F["geo_disagree"] = np.isin(cls, ["B_coordinates_differ", "C_postcode_differs", "D_city_differs", "E_all_differ", "G_two_differ_third_missing"]).astype(int)
    F["coord_differs"] = (cls == "B_coordinates_differ").astype(int); F["postcode_differs"] = (cls == "C_postcode_differs").astype(int)
    F["city_differs"] = (cls == "D_city_differs").astype(int); F["all_differ"] = np.isin(cls, ["E_all_differ", "G_two_differ_third_missing"]).astype(int)
    F["cannot_tell"] = (cls == "H_cannot_tell").astype(int)
    F["one_differs"] = np.isin(cls, ["B_coordinates_differ", "C_postcode_differs", "D_city_differs"]).astype(int)      # two signals agree, one is the odd one out
    F["three_places"] = (cls == "E_all_differ").astype(int)                                                           # all three located, no two agree
    min_pair = F[["d_coord_pin_nearest_km", "d_coord_city_km", "d_pin_city_km"]].min(axis=1)
    F["three_places_far"] = ((cls == "E_all_differ") & (min_pair > FAR_KM)).astype(int)                                # ... and every pair is more than 100 km apart
    F["n_pairs_differ"] = ((has_L & has_P & ~LP).astype(int) + (has_L & has_C & ~LC).astype(int) + (has_P & has_C & ~PC).astype(int))
    F["state_mismatch"] = (F.n_states >= 2).astype(int)
    F["coord_pin_far"] = (F.d_coord_pin_nearest_km > FAR_KM).astype(int)
    F["log_d_coord_pin"] = np.log1p(F.d_coord_pin_nearest_km.fillna(0))
    F["max_pair_km"] = F[["d_coord_pin_nearest_km", "d_coord_city_km", "d_pin_city_km"]].max(axis=1)
    F["log_max_pair_km"] = np.log1p(F.max_pair_km.fillna(0))
    F["any_pair_far"] = (F.max_pair_km > FAR_KM).astype(int)

    F = F.rename(columns={"unique_id": "uid"})
    # location.csv: every numeric column other than best_lat / best_lon / best_radius_km is a candidate feature for analysis/eval_feature.py.
    # Text columns explain the class. Intermediate numbers (PIN centre, city centre ...) are left out so the yardstick is not flooded.
    out = F[["uid", "best_lat", "best_lon", "best_state", "best_district", "loc_class", "d_coord_pin_km", "location_confidence",
             "geo_disagree", "one_differs", "all_differ", "three_places", "three_places_far", "coord_differs", "postcode_differs", "city_differs", "cannot_tell",
             "city_unverifiable", "n_pairs_differ", "n_states", "state_mismatch",
             "d_coord_pin_nearest_km", "log_d_coord_pin", "coord_pin_far", "log_max_pair_km", "any_pair_far", "pin_multi_state", "pin_scattered",
             "best_radius_km", "best_src", "st_state", "st_src", "pc_status", "pc_src", "pc_state", "pc_district", "geo_state", "geo_district", "city_src"]]
    out = out.round({"best_lat": 6, "best_lon": 6, "d_coord_pin_km": 2, "d_coord_pin_nearest_km": 2, "log_d_coord_pin": 3, "log_max_pair_km": 3, "best_radius_km": 1})
    if write:
        P.to_csv(os.path.join(OUT, "pin_lookup.csv"), index=False)
        out.to_csv(os.path.join(OUT, "location.csv"), index=False)
    print("\nfacilities:", len(out), "| class:", out.loc_class.value_counts().to_dict())
    print("signals usable: coordinates", int(has_L.sum()), "| postcode", int(has_P.sum()), "| city", int(has_C.sum()))
    print("states named by the signals:", out.n_states.value_counts().sort_index().to_dict(), "| best source:", out.best_src.value_counts().to_dict())
    print("coordinates vs postcode: within 10 km of PIN centre", int((F.d_coord_pin_km <= 10).sum()), "| within 10 km of a post office of that PIN", int((F.d_coord_pin_nearest_km <= 10).sum()),
          "| agree by the full rule", int(LP.sum()), "| more than 100 km apart", int(F.coord_pin_far.sum()))
    print("location confidence:", out.location_confidence.describe().round(2).to_dict())
    return P, F, out


if __name__ == "__main__":
    main()
''')

### `pipeline/claims.py`
Evidence per (facility, service): every matching sentence, caution flags (negation, referral, future, biography, directory, news), tier, best quote, items found.

In [ ]:
write('pipeline/claims.py', r'''
"""Step 2: find every sentence that mentions a service, and grade each (facility, service) claim.

Output
  pipeline/out/hits.pkl    one row per (facility, service, sentence) match
  pipeline/out/claims.csv  one row per (facility, service): tier 3 strong / 2 medium / 1 weak
Tier rule (no weights):
  strong  = an infrastructure/equipment sentence with no caution flag, AND (a quantity in it OR support in 2+ fields)
  medium  = a named procedure or an infrastructure sentence with no caution flag
  weak    = only loose topical words, or every matching sentence carries a caution flag
Caution flags: negation, referral elsewhere, future plan, doctor biography, directory listing, news, "not listed" meta text.
"""
import re, sys, os
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import load_facilities, OUT
from ontology import SERVICE_ONTOLOGY as ONT, CONTEXT, QUANTITY

I = re.I
CX = {k: re.compile(v, I) for k, v in CONTEXT.items()}
QTY = re.compile(QUANTITY, I)
SPLIT = re.compile(r";\s+|\s\|\s|\n+|(?<![A-Z]r)(?<!No)(?<!Mr)(?<!Mrs)(?<!St)(?<!vs)(?<!Prof)(?<!\b[A-Z])(?<=[a-z0-9\)\]])\.\s+(?=[A-Z0-9])")
comp = lambda lst: re.compile("|".join(f"(?:{p})" for p in lst), I) if lst else None
# False friends found by two independent readers of 227 suspect claims (2026-11-21): words that match a service but mean something else.
EXTRA_EXCLUDE = {
    "emergency_obstetric": [r"delivery (unit|arm|tray|cart|hose|tubing)", r"dental (chair )?(with |and )?(attached )?delivery", r"(timely|prompt|quick|fast|speedy|same[- ]day|on[- ]time|doorstep|free|express) delivery",
                            r"(denture|spectacle|eyewear|lens|crown|aligner|order|product|parcel|kit|report|result|sample|medicine|drug|oxygen|care|service|healthcare|treatment) delivery",
                            r"delivery of (dentures?|spectacles|eyewear|lenses|crowns?|aligners?|orders?|products?|reports?|results?|medicines?|care|services?|treatment|healthcare|quality)"],
    "cath_lab": [r"(retinal?|fundus|fluorescein|ophthalmic|ocular|eye|OCT|ICG|indocyanine|FFA|digital fundus)[ -]*(fluorescein )?angiograph\w*", r"angiograph\w* (of the )?(retina|eye|fundus)", r"(surgical|implant|dental|orthodontic|occlusal|radiographic) stents?"],
    "nicu": [r"(dry|co2|ivf|embryo|lab(oratory)?) incubators?", r"phototherapy (for|in) (skin|psoriasis|vitiligo|eczema)", r"(skin|derma\w*|psoriasis|vitiligo)[^.;]{0,40}phototherapy"],
    "cardiac_surgery": [r"\bEECP\b[^.;]*", r"(alternative|without|avoid\w*|non[- ]surgical)[^.;]{0,40}(bypass|heart surgery|cardiac surgery|open[- ]heart)"],
}
for _s, _extra in EXTRA_EXCLUDE.items():
    ONT[_s]["exclude"] = list(ONT[_s].get("exclude", [])) + _extra
C = {s: {k: comp(o.get(k, [])) for k in ["infra", "proc", "generic", "exclude"]} for s, o in ONT.items()}
ANY = {s: comp(o.get("infra", []) + o.get("proc", []) + o.get("generic", [])) for s, o in ONT.items()}
CQ = {s: {k: re.compile(v, I) for k, v in o["coreq"].items()} for s, o in ONT.items()}


def flags_for(seg, ms, me, own):
    f = []
    pre, post = seg[max(0, ms - 70):ms], seg[me:me + 70]
    m = CX["neg_pre"].search(pre)
    if m and not CX["neg_ignore"].search(pre[m.start():] + seg[ms:me]): f.append("neg")
    elif CX["neg_post"].search(post): f.append("neg")
    if CX["meta_absent"].search(seg): f.append("meta")
    loc = seg[max(0, ms - 80):me + 60]
    if CX["referral"].search(loc) and not CX["referral_ignore"].search(loc): f.append("referral")
    if CX["future"].search(seg[max(0, ms - 45):me + 30]): f.append("future")
    if CX["bio"].search(own.sub(" ", seg) if own else seg): f.append("bio")
    if CX["directory"].search(seg): f.append("directory")
    if CX["news"].search(seg): f.append("news")
    return f


def build_hits(df):
    rows = []
    for r in df.itertuples():
        nm = r.name.strip()
        own = re.compile(re.escape(nm), I) if len(nm) >= 4 else None
        units = []
        for fld in ["procedure", "equipment", "capability"]:
            seen = set()
            for t in getattr(r, fld + "_l"):
                t = t.strip(); k = re.sub(r"\W+", " ", t.lower()).strip()
                if t and k not in seen:
                    seen.add(k); units.append((fld, t))
        units.append(("description", r.description))
        for fld, t in units:
            cand = [s for s in ONT if ANY[s].search(t)]
            if not cand: continue
            for seg in SPLIT.split(t):
                if not seg or len(seg) < 3: continue
                for s in cand:
                    c = C[s]
                    mseg = c["exclude"].sub(lambda m: " " * len(m.group(0)), seg) if c["exclude"] else seg
                    if own: mseg = own.sub(lambda m: " " * len(m.group(0)), mseg)   # a service word inside the facility's own name is not evidence
                    hit = None
                    for cls in ("infra", "proc", "generic"):
                        if c[cls] is None: continue
                        m = c[cls].search(mseg)
                        if m: hit = (cls, m); break
                    if not hit: continue
                    cls, m = hit
                    fl = flags_for(seg, m.start(), m.end(), own)
                    qty = bool(QTY.search(seg[max(0, m.start() - 45):m.end() + 45]))
                    rows.append((r.unique_id, s, fld, cls, m.group(0), qty, "|".join(fl), len(seg), seg))
    return pd.DataFrame(rows, columns=["uid", "service", "field", "cls", "match", "qty", "flags", "seglen", "seg"])


def build_claims(H, df):
    H = H.assign(valid=H["flags"].eq(""))
    alltext = (df.description + " || " + df.procedure_l.map(" || ".join) + " || " + df.equipment_l.map(" || ".join) + " || " + df.capability_l.map(" || ".join))
    alltext = dict(zip(df.unique_id, alltext))
    rank = {"infra": 0, "proc": 1, "generic": 2}
    out = []
    for (uid, s), h in H.groupby(["uid", "service"]):
        v = h[h.valid]; vi = v[v.cls == "infra"]; vp = v[v.cls == "proc"]
        nf = v[v.cls.isin(["infra", "proc"])].field.nunique()
        tier = 3 if (len(vi) and (nf >= 2 or vi.qty.any())) else 2 if (len(vi) or len(vp)) else 1
        b = h.assign(o=(~h.valid) * 10 + h.cls.map(rank) - h.qty * 0.5, l=h.seglen.clip(upper=400)).sort_values(["o", "l"]).iloc[0]
        txt = alltext.get(uid, "")
        co = [k for k, rx in CQ[s].items() if rx.search(txt)]
        out.append((uid, s, tier, nf, len(h), int(h.valid.sum()), b.field, b.seg[:300], b["flags"], len(co), len(CQ[s]), "|".join(co)))
    return pd.DataFrame(out, columns=["uid", "service", "tier", "n_fields", "n_hits", "n_valid", "quote_field", "quote", "quote_flags", "coreq_found", "coreq_total", "coreq_list"])


if __name__ == "__main__":
    df = load_facilities()
    H = build_hits(df); H.to_pickle(os.path.join(OUT, "hits.pkl"))
    Cl = build_claims(H, df); Cl.to_csv(os.path.join(OUT, "claims.csv"), index=False)
    print("facilities", len(df), "| hits", len(H), "| claims", len(Cl))
    print(Cl.groupby("service").tier.value_counts().unstack(fill_value=0).to_string())
''')

### `pipeline/feat_attribution.py`
Warning sign: the text names towns far from the facility.

In [ ]:
write('pipeline/feat_attribution.py', r'''
"""Candidate features, family "who is each sentence about?"

For every sentence of the free-text fields (procedure, equipment, capability, description) decide whether it
  - names the record itself                      (own_name)
  - names ANOTHER organisation as subject/place  (other_org)   e.g. "... established at Lok Nayak Hospital"
  - names a place far from the record's address  (other_place) e.g. "Located in Surat" on an Ahmedabad record
  - states a total bed count                     (used for bed-count contradictions)
and roll that up
  (a) per facility            -> pipeline/out/feat_attribution_row.csv    (key uid)
  (b) per (facility, service) -> pipeline/out/feat_attribution_claim.csv  (key uid + service), using the supporting
      sentences in pipeline/out/hits.pkl: do THEY name another place/organisation, or sit next to sentences that do?
Also rebuilds two row signals whose code was lost: distinct landline area codes among phone_numbers, distinct source ids.

Leakage guard. The silver labels come from specialty words in the facility NAME. This script uses the name only to
tell "this sentence names the record" from "this sentence names someone else". Specialty words (trust.NEG / POS / BROAD
patterns, e.g. eye, dental, kidney, heart, maternity) are deleted from both sides before any comparison, so
"Asian Hospital" and "Asian Eye Hospital" compare as the same organisation. facilityTypeId and specialties are never read.

Second leak found while building this: a sentence that only repeats the facility's name ("X Kidney Hospital is listed in ...") is counted
as support for dialysis because the keyword sits inside the NAME. By construction only likely-real silver claims can have such support, so
any feature built on raw support counts or on "the supporting sentence names the record" looks far better than it is. The claim file therefore
carries only "clean" versions (name words masked before the service keyword is re-tested). The raw versions are written, prefixed leaky_,
only with --debug, for comparison; never feed them to the formula.

Run from the project root:  python -I pipeline/feat_attribution.py [--debug DIR]
Raw inputs only: facilities.csv, the two India Post PIN files, and hits.pkl (rebuilt with claims.build_hits if missing).

Main columns
  row file    n_other_places_city   distinct places named in the text that lie > 60 km from the record's city cell (towns < 40 km apart count once)
              n_other_places        same, but a place also counts as "own" when it is near the record's postcode
              n_sent_other_place / _org / _any (+ share_, any_, _city variants)   sentences about another place / organisation
              n_sent_own_name, share_sent_own_name   sentences naming the record itself
              bed_conflict          two total-bed statements (or one and the capacity cell) differ by 1.5x and 30+ beds
              n_phone_areas         distinct landline area codes (mobiles and directory call-forwarding numbers excluded); n_source_ids
  claim file  sup_other_any / sup_near_other_any   a supporting sentence names another place or organisation / sits within 2 sentences of one
              sup_own_name_clean / sup_near_own_clean   a supporting sentence (name masked) names the record / sits within 2 sentences of one that does
              n_sup_clean, sup_share_of_record   supporting sentences after masking the name, and their share of all sentences in the record
"""
import sys, os, re, difflib, collections
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd
from common import load_facilities, ROOT, OUT
import trust                                   # only for the name-specialty patterns that must be ignored
from claims import SPLIT, ANY

SAME_KM = 40         # two named places closer than this count as one place ("Chinchwad" and "Pune")
FAR_KM = 60          # "another place" = more than this from the record's own city AND own postcode. A district is ~100 km across.
NEAR_SENT = 2        # "next to" = within this many list positions in the same field
I = re.I

# ----------------------------------------------------------------------------------------------- place gazetteer
STATE_ALIAS = {"tamilnadu": "TAMIL NADU", "orissa": "ODISHA", "pondicherry": "PUDUCHERRY", "uttaranchal": "UTTARAKHAND", "uttranchal": "UTTARAKHAND",
               "nct of delhi": "DELHI", "nct delhi": "DELHI", "new delhi": "DELHI", "delhi ncr": "DELHI", "jammu & kashmir": "JAMMU AND KASHMIR",
               "jammu and kashmir": "JAMMU AND KASHMIR", "kashmir": "JAMMU AND KASHMIR", "chattisgarh": "CHHATTISGARH", "telengana": "TELANGANA",
               "uttarpradesh": "UTTAR PRADESH", "madhyapradesh": "MADHYA PRADESH", "andaman and nicobar": "ANDAMAN AND NICOBAR ISLANDS"}
STATE_CODE = {"up": "UTTAR PRADESH", "u p": "UTTAR PRADESH", "mp": "MADHYA PRADESH", "m p": "MADHYA PRADESH", "ts": "TELANGANA", "gj": "GUJARAT", "mh": "MAHARASHTRA",
              "ms": "MAHARASHTRA", "dl": "DELHI", "br": "BIHAR", "cg": "CHHATTISGARH", "u k": "UTTARAKHAND", "nct": "DELHI"}      # used for the address cell only, never in sentences
CITY_ALIAS = {"madras": "chennai", "bombay": "mumbai", "calcutta": "kolkata", "trivandrum": "thiruvananthapuram", "cochin": "kochi", "baroda": "vadodara",
              "poona": "pune", "prayagraj": "allahabad", "vizag": "visakhapatnam", "trichy": "tiruchirappalli", "banaras": "varanasi", "gurugram": "gurgaon",
              "bengaluru": "bangalore", "mysuru": "mysore", "mangaluru": "mangalore", "belagavi": "belgaum", "hubballi": "hubli", "kalaburagi": "gulbarga",
              "tuticorin": "thoothukudi", "calicut": "kozhikode", "secunderabad": "hyderabad", "bhatinda": "bathinda", "simla": "shimla", "panjim": "panaji",
              "chhatrapati sambhajinagar": "aurangabad", "vijaywada": "vijayawada", "pondicherry": "puducherry", "vijayapura": "bijapur",
              "delhi ncr": "delhi", "ncr": "delhi"}
CITY_STATES = {"delhi", "new delhi", "delhi ncr", "chandigarh", "puducherry", "pondicherry"}      # states small enough to be judged by distance, like a city
# place names that are also ordinary words, compass points or very common personal / organisation names
PLACE_STOP = set("""central north south east west new old north east north west south west south east east district north district south district west district
anand sagar krishna gaya dhar pali guna banka bid beed una mau dang tapi narmada mandi mon nadia saran hassan korea basti banda india city town
market colony civil lines main road camp model town gandhi nagar nehru nagar shastri nagar ashok nagar ram nagar medical college cantt fort station
ganga yamuna godavari kaveri sangam bharat hind raja rani bara chota patel sardar shivaji rajendra indira rajiv mahatma nehru ambedkar
kota champa gir sidhi seoni panna rewa satna durg bastar korba mahe diu daman leh doda samba reasi moga nuh jind hapur etah lohit siang
unity hope life care apex star sun moon lotus rose city centre global royal ideal modern national sanjeevani sanjivani jeevan jivan arogya
bishop mission sector block phase ward plot tower plaza complex mall park garden circle square junction cross bridge temple church mosque""".split()) | {
    "north east", "north west", "south west", "south east", "east district", "north district", "south district", "west district", "civil lines",
    "main road", "model town", "gandhi nagar", "nehru nagar", "shastri nagar", "ashok nagar", "ram nagar", "medical college"}
PLACE_KEEP = {"kota", "goa", "agra", "puri", "pune", "durg", "rewa", "satna", "korba", "moga", "jind", "hapur", "leh"}     # real towns worth keeping despite the list above
HONORIFIC = {"dr", "dr.", "shri", "smt", "mr", "mr.", "mrs", "mrs.", "ms", "late", "prof", "prof.", "sri", "shree", "sh", "sh.", "st", "st.", "saint", "lord", "maa", "baba"}
# a place word followed by one of these is part of a name ("Bombay Hospital", "Mysore Road", "Gujarat Ayurved University"), not a location statement
AFTER_PLACE = set("""hospital hospitals clinic clinics medical eye dental heart kidney institute nursing memorial road rd nagar marg street gate chowk colony bazar bazaar
market enclave vihar university college school scan scans diagnostics diagnostic labs lab laboratory pharmacy bank public council board ayurved ayurveda ayurvedic
healthcare health care cancer ortho orthopaedic orthopedic maternity children child ent skin netralaya nethralaya trust foundation society mission tower towers plaza
complex mall club hotel cinema talkies sweets stores store traders enterprises agencies pharma surgicals opticals optical fertility ivf neuro spine diabetes
x-ray imaging pathology test polyclinic dispensary homoeo homeo physiotherapy super multi speciality specialty multispeciality multispecialty pin
panchkarma panchakarma style massage therapy therapies treatment treatments technique techniques medicine medicines""".split())

SCHEME_STATE = {r"aa?rogya ?sri": {"ANDHRA PRADESH", "TELANGANA"}, r"mahatma (?:jyotiba|jyotirao) phule|mjpjay|rajiv gandhi jeevandayee": {"MAHARASHTRA"},
                r"chief minister'?s comprehensive health insurance|cmchis": {"TAMIL NADU"}, r"karunya|\bkasp\b|medisep": {"KERALA"}, r"swasthya sathi": {"WEST BENGAL"},
                r"biju swasthya|\bbsky\b": {"ODISHA"}, r"bhamashah|chiranjeevi|\brghs\b": {"RAJASTHAN"}, r"mukhyamantri amrutam|ma vatsalya|ma yojana": {"GUJARAT"},
                r"yeshasvini|arogya karnataka|suvarna arogya|vajpayee arogya": {"KARNATAKA"}, r"atal amrit": {"ASSAM"}, r"deen ?dayal swasthya seva": {"GOA"},
                r"himcare": {"HIMACHAL PRADESH"}, r"sarbat sehat|bhagat puran singh": {"PUNJAB"}, r"mohalla clinic|govt\.? of nct|government of nct|gnctd|delhi arogya": {"DELHI"},
                r"megha health": {"MEGHALAYA"}, r"atal ayushman uttarakhand": {"UTTARAKHAND"}}
SCHEME_RX = [(re.compile(k, I), v) for k, v in SCHEME_STATE.items()]

# sentence contexts in which another place / organisation is legitimately mentioned by the record's own page
BIO_RX = re.compile(r"\b(?:MBBS|M\.B\.B\.S|M\.?D|M\.?S|DNB|M\.?Ch|D\.?M|FRCS|MRCP|MRCS|BDS|MDS|BAMS|BHMS|DGO|DCH)\b|(?i:\b(?:fellowship|fellow|trained|training (?:at|from|in)|graduat\w*|alumn\w*|residency|internship|completed|studied|degree|worked (?:at|as|in|with)|formerly|previously|served (?:as|at|in)|experience (?:at|in|from)|visiting|born in|native of)\b)")
CATCH_RX = re.compile(r"patients? from|serv(?:es|ing)\b.{0,40}\b(?:districts?|region|areas?|states?|patients|cities|towns)|nearby (?:districts|cities|towns|areas)|neighbou?ring|catchment|across (?:india|the country)|all over|distance from|\bkm\b|kilomet|from across|largest|\bfirst\b|only (?:in|one)|best (?:in|of)|one of the|pioneer|\bserv(?:es|ing)\b|regist(?:ered|ration)|\bROC\b|universit|council|\bboard\b", I)
CHAIN_RX = re.compile(r"\bnetwork (?:includes|operates)\b|\b(?:cent(?:re|er)s|locations|outlets|units)\b.{0,30}\b(?:in|across|at|include[sd]?|located|established)\b|\bclinics are located\b|\blocations? include|\boperates (?:\w+ ){0,4}cent(?:re|er)s?\b|\bmajor (?:indian )?cities\b|\bbranch(?:es)?\b|\b(?:cent(?:re|er)s|locations|hospitals|clinics|units|outlets|presence|offices)\s+(?:in|across|at)\b|\bnetwork of\b|\bchain\b|\bpan[- ]india\b|\bheadquarter|\bhead office|\bregistered office|\bcorporate office", I)
AFFIL_RX = re.compile(r"affiliat|accredit|certified by|quality control|recogni[sz]ed by|approved by|\bunit of\b|venture of|\bpart of\b|managed by|run by|operated by|owned by|under the aegis|sister concern|subsidiary|collaborat|tie[- ]?up|association with|associated with|attached to|partner|founded by|established by|group of|parent", I)
LANDMARK_RX = re.compile(r"\bnear\b|\bnearby\b|opposite|\bopp\b|behind|next to|beside|adjacent|landmark|metres|meters|\bkm\b|in front of", I)
ORG_KEY = {"hospital", "hospitals", "clinic", "clinics", "netralaya", "nethralaya", "polyclinic", "institute", "sanatorium", "infirmary"}
ORG_KEY2 = {("nursing", "home"), ("maternity", "home"), ("medical", "college"), ("medical", "centre"), ("medical", "center"), ("health", "centre"), ("health", "center")}
CONNECT = {"of", "and", "&", "the", "for", "in"}
# words that never identify an organisation: legal forms, generic health words, insurers and web directories that the pages quote
ORG_GENERIC = set("""hospital hospitals clinic clinics centre center centres centers medical health healthcare care nursing home institute institutes research
pvt ltd limited private llp the and of for in at a an dr sri shri shree sree smt st new city life general district civil government govt public charitable trust
memorial mission foundation society super multi speciality specialty specialities multispeciality multispecialty superspeciality polyclinic top best leading
network empanelled empaneled cashless insurance listed star niva bupa hdfc ergo icici lombard bajaj allianz iffco tokio religare manipalcigna cigna aditya birla acko
tata aig reliance sbi oriental united national india indian future generali cholamandalam royal sundaram digit liberty magma kotak medi assist paramount vidal raksha
policybazaar practo justdial hexahealth lybrate medindia latlong sulekha credihealth docindia apollo247 google facebook ayushman bharat pmjay pm-jay cghs echs esic
referral related nearby society corporate tertiary teaching charitable community primary rural urban sub divisional taluk taluka area regional state
ent ivf icu nicu picu opd ipd covid nabh nabl iso jci unicef who sciences science wise dots stoma vitreo american armed forces yeshasvini aarogyasri arogyasri
policyx universal sompo capf ehs tpa ncd mtp tmd pcod pcos icl baby friendly municipal corporation cantonment railway esi phc chc uphc
tamil nadu uttar madhya andhra arunachal himachal pradesh bengal jammu kashmir east west north south navi nagar goverment
monday tuesday wednesday thursday friday saturday sunday brownfield catholic oncofertility obs gyn copd stabilisation stabilization""".split())
LEAK_RX = re.compile("|".join([trust.BROAD] + list(trust.NEG.values()) + list(trust.POS.values())), I)        # name-specialty words: always ignored
BED_RATIO, BED_DIFF = 1.5, 30     # two total-bed statements "contradict" when one is 1.5x the other and at least 30 beds apart
BED_RX = re.compile(r"(\d{1,2},\d{3}|\d{1,4})\s*\+?\s*-?\s*bed(?:ded|s)?\b", I)
BED_UNIT_RX = re.compile(r"\b(?:icu|iccu|nicu|picu|sicu|micu|ccu|hdu|sncu|wards?|emergency|casualty|dialysis|isolation|covid|ventilators?|oxygen|intensive|critical|trauma|burns?|day ?care|recovery|private rooms?|deluxe|cabins?|labou?r|neonatal|observation|triage|rooms?|hostel|college|sanctioned|proposed|planned|expan\w+|upcoming|will|originally|initially|started|began|founded|established|launched|functional|upgrad\w+|under development|historical|formerly|earlier|grew|grown)\b|\bin (?:19|20)\d\d\b", I)
TOK = re.compile(r"[A-Za-z0-9][A-Za-z0-9'’.\-]*|[,;:()&/]")
WORD = re.compile(r"[a-z]{3,}")


def nrm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()


def km(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = map(np.radians, [la1, lo1, la2, lo2])
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return float(6371 * 2 * np.arcsin(np.sqrt(h)))


def load_pin():
    P = pd.concat([pd.read_csv(os.path.join(ROOT, f), dtype=str) for f in ["indiaPincodeDirectory1.csv", "indiaPincodeDirectory2.csv"]]).drop_duplicates()
    P["lat"] = pd.to_numeric(P.latitude, errors="coerce"); P["lon"] = pd.to_numeric(P.longitude, errors="coerce")
    P.loc[~(P.lat.between(6, 37.5) & P.lon.between(68, 97.5)), ["lat", "lon"]] = np.nan
    P["statename"] = P.statename.str.upper().str.strip(); P["district"] = P.district.str.upper().str.strip()
    mode = lambda s: s.mode().iloc[0] if s.notna().any() else np.nan
    pin = P.groupby("pincode").agg(state=("statename", mode), district=("district", mode), lat=("lat", "median"), lon=("lon", "median"))
    dist = P.dropna(subset=["district", "statename"]).groupby(["statename", "district"]).agg(lat=("lat", "median"), lon=("lon", "median"), n=("lat", "size")).reset_index()
    return pin, dist


def build_gazetteer(F, pin, dist):
    """name -> list of (lat, lon, STATE). Sources: PIN-directory districts and states, and the facility table's own city names."""
    states = {s.lower(): s for s in pin.state.dropna().unique()}
    states.update(STATE_ALIAS)
    gaz = collections.defaultdict(list)
    for r in dist.itertuples():
        if np.isnan(r.lat): continue
        name = re.sub(r"\(.*?\)", " ", r.district.lower()); name = nrm(re.sub(r"\b(urban|rural|metro|nagar district|district)\b", " ", name)) if len(name.split()) > 1 else nrm(name)
        if name: gaz[name].append((r.lat, r.lon, r.statename))
    pc = F.address_zipOrPostcode.astype(str).str.replace(r"\D", "", regex=True).str[:6]
    C = pd.DataFrame({"city": F.address_city.map(nrm), "pc": pc}).join(pin, on="pc")
    C = C[(C.city.str.len() >= 3) & C.lat.notna()]
    for city, g in C.groupby("city"):
        if len(g) < 2: continue
        for st, h in g.groupby("state"):
            if len(h) >= 2 and len(h) >= 0.2 * len(g):          # keep each state cluster that is not a stray contaminated row
                gaz[city].append((float(h.lat.median()), float(h.lon.median()), st))
    for a, b in CITY_ALIAS.items():
        if b in gaz and a not in gaz: gaz[a] = list(gaz[b])
        elif a in gaz and b not in gaz: gaz[b] = list(gaz[a])
    for k in list(gaz):
        if (k in PLACE_STOP and k not in PLACE_KEEP) or (k in states and k not in CITY_STATES) or (len(k) < 4 and k not in PLACE_KEEP) or k.isdigit(): del gaz[k]
    return dict(gaz), states


# ----------------------------------------------------------------------------------------------- phones and sources
METRO2 = {"11", "20", "22", "33", "40", "44", "79", "80"}
STD3_678 = set("612 621 631 641 651 657 661 663 671 674 680 712 721 724 731 733 734 744 747 751 755 761 771 788 816 820 821 824 831 832 836 861 863 866 870 877 878 883 884 891".split())


def phone_area(p):
    """Landline area key for an Indian number, or None for mobiles / toll-free / unparseable.
    National numbers are 10 digits. First digit 9 = mobile. First digit 1-5 = landline (2-digit metro code or first 3 digits of the STD code).
    First digit 6-8 is ambiguous: counted as landline only for well-known city codes followed by a subscriber digit 2-6."""
    d = re.sub(r"\D", "", str(p))
    if len(d) == 12 and d.startswith("91"): d = d[2:]
    elif len(d) == 11 and d.startswith("0"): d = d[1:]
    if len(d) != 10 or d[:4] in ("1800", "1860"): return None
    if d[0] in "12345":
        return d[:2] if d[:2] in METRO2 else d[:3]
    if d[0] in "678":
        if d[:2] in ("79", "80") and d[2] in "23456": return d[:2]
        if d[:3] in STD3_678 and d[3] in "23456": return d[:3]
    return None


def phone_prefix(p):
    d = re.sub(r"\D", "", str(p))
    return d[-10:-6] if len(d) >= 10 else ""


def phone_features(F):
    own = F.address_stateOrRegion.map(nrm)
    # Call-forwarding numbers used by web directories (e.g. 07947..., 08048...) look like landlines but belong to no place, and mobile series
    # overlap the Ahmedabad (79) and Bengaluru (80) codes. Learn them from the table: a 4-digit number prefix whose rows are NOT concentrated
    # in the state where its area code is mostly seen is dropped.
    area_state, pre_state = collections.defaultdict(collections.Counter), collections.defaultdict(collections.Counter)
    for L, s_ in zip(F.phone_numbers_l, own):
        if not s_: continue
        for a, pre in {(phone_area(p), phone_prefix(p)) for p in L}:
            if a: area_state[a][s_] += 1; pre_state[pre][s_] += 1
    home = {a: c.most_common(1)[0][0] for a, c in area_state.items()}
    bad = set()
    for pre, c in pre_state.items():
        a = pre[:2] if pre[:2] in METRO2 else pre[:3]; n = sum(c.values()); share = c[home.get(a, "")] / n
        if (n >= 8 and share < 0.5) or (pre[:2] in ("79", "80") and pre[2] != "2" and (n < 8 or share < 0.8)): bad.add(pre)
    areas = F.phone_numbers_l.map(lambda L: sorted({phone_area(p) for p in L if phone_area(p) and phone_prefix(p) not in bad}))
    vote = collections.defaultdict(collections.Counter)          # learn area code -> state from the table itself (rows with exactly one area code)
    for a, s in zip(areas, own):
        if len(a) == 1 and s: vote[a[0]][s] += 1
    a2s = {a: c.most_common(1)[0][0] for a, c in vote.items() if sum(c.values()) >= 3 and c.most_common(1)[0][1] >= 0.6 * sum(c.values())}
    out = pd.DataFrame({"uid": F.unique_id})
    out["n_phone_areas"] = areas.map(len).values
    out["phone_areas_3plus"] = (out.n_phone_areas >= 3).astype(int)
    out["n_phone_states"] = areas.map(lambda a: len({a2s[x] for x in a if x in a2s})).values
    out["n_phone_areas_other_state"] = [sum(1 for x in a if x in a2s and s and a2s[x] != s) for a, s in zip(areas, own)]
    out["n_phones_distinct"] = F.phone_numbers_l.map(lambda L: len({re.sub(r"\D", "", x)[-10:] for x in L})).values
    out["n_source_ids"] = F.source_ids_l.map(lambda L: len(set(L))).values
    out["source_ids_6plus"] = (out.n_source_ids >= 6).astype(int)
    return out, areas, a2s, sorted(bad)


# ----------------------------------------------------------------------------------------------- sentence attribution
def units_of(r):
    """Sentences of one record: every list item of procedure / equipment / capability, and the description split into sentences."""
    U = []
    for fld in ["procedure", "equipment", "capability"]:
        U += [(fld, i, t.strip()) for i, t in enumerate(getattr(r, fld + "_l")) if t and t.strip()]
    U += [("description", i, s.strip()) for i, s in enumerate(x for x in SPLIT.split(r.description) if x and len(x.strip()) >= 3)]
    return U


def clean_tok(t):
    return t.strip(".'’-").lower()


class Attributor:
    def __init__(self, F, gaz, states, pin):
        self.gaz, self.states, self.pin = gaz, states, pin
        self.maxn = max(len(k.split()) for k in list(gaz) + list(states))
        low, cap = collections.Counter(), collections.Counter()      # how often a word is written in lower case inside sentences:
        for r in F.itertuples():                                     # proper names almost never are, ordinary words often are
            for _, _, t in units_of(r):
                for j, w in enumerate(TOK.findall(t)):
                    if j == 0 or not w[0].isalpha(): continue
                    (cap if w[0].isupper() else low)[clean_tok(w)] += 1
        self.common = {w for w in low if low[w] >= 5 and low[w] >= 0.1 * (low[w] + cap[w])}
        self.in_names = {w for nm in F.name for w in re.findall(r"[a-z]+", nm.lower())}      # words seen in any facility name of the table
        self._fz = {}

    def is_name_word(self, w, raw=""):
        if raw.isupper() and len(raw) <= 5 and w not in self.in_names: return False       # stray acronym (OPD, TPA, NABH ...) rather than a hospital's short name
        return len(w) >= 3 and w.isalpha() and w not in ORG_GENERIC and w not in self.common and not LEAK_RX.search(w) and w not in self.gaz and w not in self.states

    def same(self, a, b):
        if a == b: return True
        k = (a, b)
        if k not in self._fz:
            self._fz[k] = (min(len(a), len(b)) >= 4 and (a.startswith(b) or b.startswith(a))) or difflib.SequenceMatcher(None, a, b).ratio() >= 0.82
        return self._fz[k]

    def profile(self, r):
        """The record's own identity: name words (specialty words removed), own city / state / postcode place."""
        raw = [w for w in re.findall(r"[a-z]+", r.name.lower()) if len(w) >= 3]
        loose = [w for w in raw if w not in ORG_GENERIC and not LEAK_RX.search(w)]
        own_words = set(loose); core = [w for w in loose if w not in self.gaz and w not in self.states][:2]
        addr = nrm(" ".join(str(getattr(r, c)) for c in ["address_line1", "address_line2", "address_line3"] if isinstance(getattr(r, c), str)))
        name_n = nrm(r.name)
        st = nrm(r.address_stateOrRegion); own_states = set(); refs = []
        if st in self.states: own_states.add(self.states[st])
        elif st in STATE_CODE: own_states.add(STATE_CODE[st])
        elif st in self.gaz: own_states |= {c[2] for c in self.gaz[st]}; refs += [(c[0], c[1]) for c in self.gaz[st]]
        addr_states = set(own_states); city_refs = []
        pc = re.sub(r"\D", "", str(r.address_zipOrPostcode))[:6]
        if pc in self.pin.index:
            p = self.pin.loc[pc]
            if isinstance(p.state, str): own_states.add(p.state)
            if not np.isnan(p.lat): refs.append((p.lat, p.lon))
        city = nrm(r.address_city); city = CITY_ALIAS.get(city, city) if city not in self.gaz else city
        if city in self.gaz:
            cands = [c for c in self.gaz[city] if not own_states or c[2] in own_states] or self.gaz[city]
            refs += [(c[0], c[1]) for c in cands]; own_states |= {c[2] for c in cands} if not own_states else set()
            cc = [c for c in self.gaz[city] if not addr_states or c[2] in addr_states] or self.gaz[city]
            city_refs = [(c[0], c[1]) for c in cc]; addr_states = addr_states or {c[2] for c in cc}
        elif city in self.states: own_states.add(self.states[city]); addr_states.add(self.states[city])
        # "strict" view = judged against the city and state cells only (what a reader sees); falls back to the lenient view when the city is unknown
        strict = dict(refs=city_refs or refs, own_states=addr_states or own_states, cache={})
        return dict(own_words=own_words, core=core, addr=" " + addr + " ", name_n=" " + name_n + " ", own_states=own_states, refs=refs, city=nrm(r.address_city), cache={}, strict=strict)

    def place_class(self, g, P, V=None):
        """'own' / 'other' / None for a gazetteer or state name seen in a sentence of record P. V = P (lenient: city + postcode) or P['strict'] (city only)."""
        V = V or P; c = V["cache"]
        if g in c: return c[g]
        res = None
        if g == P["city"] or f" {g} " in P["name_n"] or f" {g} " in P["addr"]: res = ("own", None, None, None)
        elif g in self.states and not (g in self.gaz and V["refs"]):
            s = self.states[g]
            if V["own_states"]: res = ("own", s, None, None) if s in V["own_states"] else ("other", s, None, None)
        else:
            cands = self.gaz[g]
            if V["refs"]:
                d, s, la, lo = min((km(a[0], a[1], b[0], b[1]), a[2], a[0], a[1]) for a in cands for b in V["refs"])
                res = ("own", s, la, lo) if d <= FAR_KM else ("other", s, la, lo)
            elif V["own_states"]:
                res = ("own", cands[0][2], None, None) if any(a[2] in V["own_states"] for a in cands) else ("other", cands[0][2], cands[0][0], cands[0][1])
        c[g] = res
        return res

    def sentence(self, t, P):
        toks = TOK.findall(t); n = len(toks); low = [clean_tok(w) for w in toks]
        isw = [w[0].isalnum() for w in toks]
        bio, catch, chain = bool(BIO_RX.search(t)), bool(CATCH_RX.search(t)), bool(CHAIN_RX.search(t))
        # ---- places
        oplace, ostate, own_place, oplace_city = {}, set(), False, {}
        i = 0
        while i < n:
            hit = None
            if isw[i] and toks[i][0].isupper():
                for k in range(min(self.maxn, n - i), 0, -1):
                    if not all(isw[i:i + k]): continue
                    g = " ".join(low[i:i + k])
                    if g in self.gaz or g in self.states: hit = (g, k); break
            if not hit: i += 1; continue
            g, k = hit
            prev = low[i - 1] if i > 0 else ""; nxt = low[i + k] if i + k < n else ""
            if prev not in HONORIFIC and nxt not in AFTER_PLACE:
                pcl = self.place_class(g, P)
                if pcl:
                    if pcl[0] == "own": own_place = True
                    else: oplace[g] = pcl[1:]; ostate.add(pcl[1])
                pcs = self.place_class(g, P, P["strict"])
                if pcs and pcs[0] == "other": oplace_city[g] = pcs[1:]
            i += k
        for rx, sts in SCHEME_RX:                                     # state-run schemes are tied to a place
            if P["own_states"] and rx.search(t) and not (sts & P["own_states"]):
                oplace["scheme:" + "/".join(sorted(sts))] = oplace_city["scheme:" + "/".join(sorted(sts))] = (sorted(sts)[0], None, None); ostate |= sts
        if bio or catch or chain or AFFIL_RX.search(t): oplace, ostate, oplace_city = {}, set(), {}
        # ---- organisations
        oorg, own_name = set(), False
        skip_org = bio or bool(AFFIL_RX.search(t)) or bool(LANDMARK_RX.search(t)) or chain
        for j in range(n):
            if not isw[j]: continue
            two = j + 1 < n and (low[j], low[j + 1]) in ORG_KEY2
            if not (low[j] in ORG_KEY or two): continue
            k = j - 1; phrase = []
            while k >= 0 and len(phrase) < 6 and isw[k] and (toks[k][0].isupper() or low[k] in CONNECT):
                phrase.append((low[k], toks[k].strip(".'’-"))); k -= 1
            words = [(w, raw) for w, raw in phrase if w.isalpha() and len(w) >= 3]
            if any(self.same(w, o) for w, _ in words for o in P["own_words"]): own_name = True; continue
            names = [w for w, raw in words if self.is_name_word(w, raw)]
            if names and P["own_words"] and not skip_org: oorg.add(" ".join(reversed(names)))
        if not own_name and P["core"]:
            caps = {low[j] for j in range(n) if isw[j] and toks[j][0].isupper()}
            own_name = all(w in caps for w in P["core"])
        # ---- total bed statements
        beds = None
        if not BED_UNIT_RX.search(t):
            m = BED_RX.search(t)
            if m:
                v = int(m.group(1).replace(",", ""))
                if v >= 5: beds = v
        return dict(oplace=oplace, ostate=ostate, oorg=oorg, own_place=own_place, own_name=own_name, beds=beds, oplace_city=oplace_city)


def n_places(info):
    """Distinct places among {name: (state, lat, lon)}: names closer than SAME_KM are one place; a bare state counts only if no named town lies in it."""
    reps, seen = [], set()
    for st, la, lo in info.values():
        if la is None: continue
        seen.add(st)
        if not any(km(la, lo, a, b) <= SAME_KM for a, b in reps): reps.append((la, lo))
    return len(reps) + len({st for st, la, lo in info.values() if la is None and st not in seen})


def bed_conflict(vals, cap):
    v = sorted(set(vals)); txt = int(len(v) >= 2 and v[-1] >= BED_RATIO * v[0] and v[-1] - v[0] >= BED_DIFF)
    capc = int(bool(v) and cap == cap and cap >= 1 and any(max(x, cap) >= BED_RATIO * min(x, cap) and abs(x - cap) >= BED_DIFF for x in v))
    return txt, capc


def main(debug=None):
    F = load_facilities()
    pin, dist = load_pin()
    gaz, states = build_gazetteer(F, pin, dist)
    A = Attributor(F, gaz, states, pin)
    hp = os.path.join(OUT, "hits.pkl")
    if os.path.exists(hp): H = pd.read_pickle(hp)
    else:
        from claims import build_hits
        H = build_hits(F)
    hits_by_uid = {u: g for u, g in H.groupby("uid")}
    rows, claims, dbg = [], [], []
    for r in F.itertuples():
        P = A.profile(r); U = units_of(r)
        S = [A.sentence(t, P) for _, _, t in U]
        n = len(U)
        op = np.array([bool(s["oplace"]) for s in S], bool); oo = np.array([bool(s["oorg"]) for s in S], bool)
        opc = np.array([bool(s["oplace_city"]) for s in S], bool); othc = opc | oo
        oth = op | oo; ownn = np.array([s["own_name"] for s in S], bool); ownp = np.array([s["own_place"] for s in S], bool)
        places, places_c = {}, {}
        for s in S: places.update(s["oplace"]); places_c.update(s["oplace_city"])
        ostates = set().union(*[s["ostate"] for s in S]) if S else set()
        orgs = set().union(*[s["oorg"] for s in S]) if S else set()
        beds = [s["beds"] for s in S if s["beds"]]; btxt, bcap = bed_conflict(beds, r.capacity_n)
        d = max(n, 1)
        rows.append(dict(uid=r.unique_id, n_sent=n,
                         n_sent_other_place=int(op.sum()), n_sent_other_org=int(oo.sum()), n_sent_other_any=int(oth.sum()),
                         share_sent_other_place=op.sum() / d, share_sent_other_org=oo.sum() / d, share_sent_other_any=oth.sum() / d,
                         any_other_place=int(op.any()), any_other_org=int(oo.any()), any_other=int(oth.any()), other_any_2plus=int(oth.sum() >= 2),
                         n_other_places=n_places(places), n_other_states=len(ostates), n_other_orgs=len(orgs), other_places_2plus=int(n_places(places) >= 2),
                         n_sent_other_place_city=int(opc.sum()), share_sent_other_place_city=opc.sum() / d, n_sent_other_any_city=int(othc.sum()),
                         share_sent_other_any_city=othc.sum() / d, other_any_city_2plus=int(othc.sum() >= 2), other_any_city_3plus=int(othc.sum() >= 3),
                         n_other_places_city=n_places(places_c), other_places_city_2plus=int(n_places(places_c) >= 2),
                         n_sent_own_name=int(ownn.sum()), share_sent_own_name=ownn.sum() / d, n_sent_own_place=int(ownp.sum()),
                         other_minus_own=int(oth.sum()) - int((ownn | ownp).sum()),
                         other_vs_own_share=oth.sum() / max(oth.sum() + (ownn | ownp).sum(), 1),
                         n_bed_totals=len(set(beds)), bed_conflict_text=btxt, bed_conflict_capacity=bcap, bed_conflict=int(btxt or bcap),
                         name_has_words=int(bool(P["own_words"])), place_known=int(bool(P["refs"] or P["own_states"]))))
        if debug is not None:
            dbg += [(r.unique_id, f, i, t, "|".join(sorted(s["oplace"])), "|".join(sorted(s["oorg"])), s["own_name"], s["own_place"], s["beds"], "|".join(sorted(s["oplace_city"]))) for (f, i, t), s in zip(U, S)]
        h = hits_by_uid.get(r.unique_id)
        if h is None: continue
        # position of every sentence inside its field, and distance to the nearest "other" / "own" sentence of that field
        byf = collections.defaultdict(list)
        for k, (f, i, t) in enumerate(U): byf[f].append(k)
        dist_o, dist_w, fshare, dist_c = np.full(n, 99.0), np.full(n, 99.0), np.zeros(n), np.full(n, 99.0)
        own_anchor = ownn | ownp
        for f, ks in byf.items():
            ks = np.array(ks); o = np.where(oth[ks])[0]; w = np.where(own_anchor[ks])[0]; pos = np.arange(len(ks))
            if len(o): dist_o[ks] = np.abs(pos[:, None] - o[None, :]).min(1)
            oc = np.where(othc[ks])[0]
            if len(oc): dist_c[ks] = np.abs(pos[:, None] - oc[None, :]).min(1)
            if len(w): dist_w[ks] = np.abs(pos[:, None] - w[None, :]).min(1)
            fshare[ks] = oth[ks].mean()
        # A sentence that merely repeats the facility's name ("X Kidney Hospital is listed in ...") matches the service keyword through the NAME.
        # Such name-induced support is circular (and would leak the name-based silver label), so "clean" features mask the name words first.
        nw = sorted({w for w in re.findall(r"[a-z]+", r.name.lower()) if len(w) >= 3 and w not in ORG_GENERIC}, key=len, reverse=True)
        mask = re.compile(r"\b(?:" + "|".join(map(re.escape, nw)) + r")(?:'?s)?\b", I) if nw else None
        for svc, g in h.groupby("service"):
            idx, clean = [], set()
            for f, seg in zip(g.field, g.seg):
                seg = seg.strip(); k = next((k for k in byf.get(f, []) if seg in U[k][2]), None)
                if k is None and f == "description": k = next((k for k in byf.get(f, []) if U[k][2][:40] in seg or seg[:40] in U[k][2]), None)
                if k is not None:
                    idx.append(k)
                    if mask is None or ANY[svc].search(mask.sub(" ", seg)): clean.add(k)
            if not idx:
                claims.append(dict(uid=r.unique_id, service=svc, sup_found=0)); continue
            idx = np.array(sorted(set(idx))); do, dw, dc = dist_o[idx], dist_w[idx], dist_c[idx]
            cl = np.array(sorted(clean), int); isclean = np.isin(idx, cl)
            closer = (do < dw) & (do <= 5)
            claims.append(dict(uid=r.unique_id, service=svc, sup_found=1, n_sup=len(idx),
                               sup_other_any=int(oth[idx].any()), sup_other_share=float(oth[idx].mean()), sup_other_all=int(oth[idx].all()),
                               sup_other_place=int(op[idx].any()), sup_other_org=int(oo[idx].any()),
                               sup_near_other_any=int((do <= NEAR_SENT).any()), sup_near_other_share=float((do <= NEAR_SENT).mean()),
                               sup_near_other_all=int((do <= NEAR_SENT).all()),
                               sup_closeness_other=float((1 / (1 + do)).max()), sup_closeness_other_mean=float((1 / (1 + do)).mean()),
                               sup_other_closer_any=int(closer.any()), sup_other_closer_share=float(closer.mean()),
                               sup_own_name=int(ownn[idx].any()), sup_near_own_any=int((dw <= NEAR_SENT).any()), sup_near_own_share=float((dw <= NEAR_SENT).mean()),
                               n_sup_clean=len(cl), sup_share_of_record=len(cl) / max(n, 1), sup_name_only=int(len(cl) == 0), share_sup_name_induced=float(1 - isclean.mean()),
                               sup_own_name_clean=int(ownn[cl].any()) if len(cl) else 0, sup_near_own_clean=int((dist_w[cl] <= NEAR_SENT).any()) if len(cl) else 0,
                               sup_other_any_clean=int(oth[cl].any()) if len(cl) else 0, sup_near_other_clean=int((dist_o[cl] <= NEAR_SENT).any()) if len(cl) else 0,
                               sup_field_other_share=float(fshare[idx].mean()),
                               sup_other_any_city=int(othc[idx].any()), sup_near_other_any_city=int((dc <= NEAR_SENT).any()), sup_closeness_other_city=float((1 / (1 + dc)).max()),
                               sup_other_and_row2=int(oth[idx].any() or ((do <= NEAR_SENT).any() and oth.sum() >= 2))))
    R = pd.DataFrame(rows)
    ph, areas, a2s, bad_pre = phone_features(F)
    R = R.merge(ph, on="uid")
    Cl = pd.DataFrame(claims).fillna(0)
    Cl = Cl.merge(R[["uid", "share_sent_other_any", "n_other_places"]].rename(columns={"share_sent_other_any": "row_share_other", "n_other_places": "row_n_other_places"}), on="uid")
    LEAKY = ["n_sup", "sup_own_name", "sup_near_own_any", "sup_near_own_share", "sup_name_only", "share_sup_name_induced"]      # see module docstring
    R.round(4).to_csv(os.path.join(OUT, "feat_attribution_row.csv"), index=False)
    Cl.drop(columns=LEAKY).round(4).to_csv(os.path.join(OUT, "feat_attribution_claim.csv"), index=False)
    print(f"facilities {len(R)} | sentences {int(R.n_sent.sum())} | gazetteer names {len(gaz)} | claims {len(Cl)} (supporting sentence located for {int(Cl.sup_found.sum())})")
    print(f"rows with a sentence about another place {int(R.any_other_place.sum())} | another organisation {int(R.any_other_org.sum())} | either {int(R.any_other.sum())} | 2+ such sentences {int(R.other_any_2plus.sum())}")
    print(f"rows with bed-count conflict {int(R.bed_conflict.sum())} | 3+ landline area codes {int(R.phone_areas_3plus.sum())} | 6+ source ids {int(R.source_ids_6plus.sum())}")
    print(f"claims whose supporting sentence names another place/organisation {int(Cl.sup_other_any.sum())} | or sits within {NEAR_SENT} sentences of one {int(Cl.sup_near_other_any.sum())}")
    print(f"claims supported ONLY by sentences that repeat the facility's own name {int(Cl.sup_name_only.sum())} | claims with at least one such sentence {int((Cl.share_sup_name_induced > 0).sum())}")
    if debug is not None:
        os.makedirs(debug, exist_ok=True)
        Cl.rename(columns={c: "leaky_" + c for c in LEAKY}).round(4).to_csv(os.path.join(debug, "feat_attribution_claim_with_leaky.csv"), index=False)
        pd.DataFrame(dbg, columns=["uid", "field", "i", "text", "oplace", "oorg", "own_name", "own_place", "beds", "oplace_city"]).to_pickle(os.path.join(debug, "attr_sentences.pkl"))
        pd.Series({k: str(v) for k, v in gaz.items()}).to_csv(os.path.join(debug, "attr_gazetteer.csv"))
        pd.DataFrame({"uid": F.unique_id, "areas": areas.map("|".join)}).to_csv(os.path.join(debug, "attr_phone_areas.csv"), index=False)
        pd.Series(a2s).to_csv(os.path.join(debug, "attr_area_state.csv")); open(os.path.join(debug, "attr_bad_prefix.txt"), "w").write(" ".join(bad_pre))


if __name__ == "__main__":
    main(sys.argv[sys.argv.index("--debug") + 1] if "--debug" in sys.argv else None)
''')

### `pipeline/feat_sources.py`
Warning sign: the row was built from other organisations' websites; and whether the facility's own site has a page for the service.

In [ ]:
write('pipeline/feat_sources.py', r'''
"""Feature family: WHERE DID THE ROW'S INFORMATION COME FROM?

Reads only raw data (facilities.csv through common.load_facilities, the India Post PIN directory) and
pipeline/out/claims.csv (the list of facility x service claims). No dependence on research/data_work.

    python -I pipeline/feat_sources.py            # writes pipeline/out/feat_sources_row.csv and feat_sources_claim.csv
    python -I pipeline/feat_sources.py --show ID  # print the parsed URLs of one record (unique_id prefix)

Every web address attached to a row (source_urls + websites, de-duplicated) is put in one of these boxes:
    own      on the record's official website domain, or the address spells the record's own name
    social   facebook / instagram / linkedin / maps ... (says nothing either way, ignored)
    foreign  everything else: a directory page, a list, a journal article, or ANOTHER facility's page
Row features count the boxes; place features compare the places written in the addresses with the record's state.
Claim features ask whether the service word itself appears in an address path (own vs foreign).

Leakage guard: the facility name is used only to recognise "this address is about this record". Specialty and
generic words (eye, dental, kidney, hospital ...) are dropped from the name before matching, and for the claim
features every word of the record's own name is removed from the address before the service word is looked for,
so "…/abc-kidney-dialysis-centre" cannot vote for dialysis on "ABC Kidney Dialysis Centre".
facilityTypeId and the specialties column are never read.

What the yardstick said (2026-11-21, 744 silver claims; current formula AUC 0.799, hospitals only 0.741):
    other_sites_2plus   +0.007 overall, +0.017 within hospitals (bootstrap range includes 0)   <- only row sign worth adding
    svc_in_own_url      +0.004 / +0.006, fires on 25 silver claims of which 23 are likely real  <- small positive evidence
    own_share, n_foreign_log, every "other state" and e-mail / Facebook feature: gain 0 or negative. They repeat what the
    existing 'sources' sign (6+ source ids) already measures (rank correlation of n_foreign_log with it: 0.75).
    has_official_site and no_own_url show gains but in a direction with no plain reason; treat them as profile proxies.
"""
import sys, os, re
from collections import Counter
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import ROOT, OUT, load_facilities, norm_name

# ---------------------------------------------------------------- web address parsing
SLD = {"gov.in", "nic.in", "co.in", "org.in", "ac.in", "edu.in", "net.in", "res.in", "gen.in", "firm.in", "ind.in", "co.uk", "org.uk", "ac.uk", "com.au", "com.sg", "co.za", "com.bd", "com.np", "business.site", "com.in"}
SOCIAL = {"facebook.com", "instagram.com", "linkedin.com", "twitter.com", "x.com", "youtube.com", "youtu.be", "goo.gl", "google.com", "google.co.in", "g.page", "mappls.com", "wa.me", "whatsapp.com",
          "t.me", "pinterest.com", "bit.ly", "g.co", "latlong.net", "mapcarta.com", "wikimapia.org", "mapsofindia.com", "threads.net", "linktr.ee", "threads.com"}
JOURNAL = {"nih.gov", "researchgate.net", "lww.com", "sciencedirect.com", "springer.com", "wiley.com", "biomedcentral.com", "scribd.com", "academia.edu", "semanticscholar.org", "tandfonline.com",
           "sagepub.com", "bmj.com", "nature.com", "plos.org", "mdpi.com", "frontiersin.org", "cureus.com", "jcdr.net", "ijcmr.com", "slideshare.net", "doi.org", "elsevier.com", "thelancet.com", "oup.com", "medknow.com"}
FREEMAIL = {"gmail.com", "yahoo.com", "yahoo.co.in", "yahoo.in", "hotmail.com", "rediffmail.com", "ymail.com", "outlook.com", "live.com", "icloud.com", "aol.com", "rocketmail.com", "live.in", "msn.com", "protonmail.com"}


def split_url(u):
    u = str(u).strip().lower(); u = re.sub(r"^[a-z]+://", "", u)
    m = re.split(r"[/?#]", u, maxsplit=1); h = m[0].split("@")[-1].split(":")[0]
    h = re.sub(r"^(www\d?|m|en|in)\.", "", re.sub(r"^(www\d?|m)\.", "", h))
    return h, (m[1] if len(m) > 1 else "")


def reg_domain(h):
    p = h.split(".")
    if len(p) >= 3 and ".".join(p[-2:]) in SLD: return ".".join(p[-3:])
    return ".".join(p[-2:]) if len(p) >= 2 else h


def toks(s):
    return [t for t in re.split(r"[^a-z0-9]+", s.lower()) if t]


# ---------------------------------------------------------------- the record's own name (identity only)
GENERIC = set("""hospital hospitals clinic clinics centre center centres centers care health healthcare medical medicare medicals nursing home homes multi multispeciality multispecialty
speciality specialty specialities super superspeciality dr doctor doctors the and of for pvt ltd private limited trust foundation institute research memorial charitable general society
eye eyes dental dentist dentistry skin hair kidney heart cardiac children child childrens maternity women womens ortho orthopaedic orthopedic ent diagnostic diagnostics lab labs laboratory
pathology scan scans imaging physiotherapy physio ayurveda ayurvedic homeo homeopathy homoeopathy homeopathic surgical surgery fertility ivf test tube baby cancer neuro dialysis sri shri shree
new india indian city unit services service group llp inc mission govt government district civil community primary sub polyclinic poly dispensary pharmacy medicine medicines laser cosmetic
vision netralaya nethralaya smile tooth teeth derma dermatology retina lasik trauma emergency icu institute sciences science college university school department wellness life line plus
mother mothers kids ladies gynae gynec stone urology uro gastro liver brain spine bone joint nephro renal piles diabetes diabetic thyroid chest lung clinic's www com org
plastic aesthetic aesthetics reconstructive cardiology cardio cardiothoracic oncology ophthalmic ophthalmology orthodontic orthodontics implant implants nephrology neurology gastroenterology
diabetology pediatric paediatric pediatrics paediatrics gynecology gynaecology obstetrics psychiatry psychiatric rehabilitation medicity nursinghome multispecialityhospital superspecialty
maternal neonatal newborn infertility andrology oncology radiology sonography ultrasound xray pharma chemist opticals optical optics dentals vascular endocrine ortho""".split())


def name_keys(name, city):
    t = toks(norm_name(name)); place = set(toks(norm_name(city)))
    return [x for x in dict.fromkeys(t) if len(x) >= 4 and not x.isdigit() and x not in GENERIC and x not in place], set(t)


def mentions(sq, keys):
    """Does the squashed address spell the record's name?  Two name words, one long one, or the leading name word (5+ letters)."""
    if not keys: return False
    hit = [k for k in keys if k in sq]
    if len(keys) == 1: return bool(hit)
    return len(hit) >= 2 or any(len(k) >= 7 for k in hit) or (len(keys[0]) >= 5 and keys[0] in sq)


# ---------------------------------------------------------------- places written inside addresses
STATES = {"andhra pradesh": ["andhrapradesh", "andhra"], "arunachal pradesh": ["arunachal", "arunachalpradesh"], "assam": ["assam"], "bihar": ["bihar"], "chhattisgarh": ["chhattisgarh", "chattisgarh"],
          "goa": ["goa"], "gujarat": ["gujarat", "gujrat"], "haryana": ["haryana"], "himachal pradesh": ["himachal", "himachalpradesh"], "jharkhand": ["jharkhand"], "karnataka": ["karnataka"],
          "kerala": ["kerala"], "madhya pradesh": ["madhyapradesh", "mpdoctors"], "maharashtra": ["maharashtra"], "manipur": ["manipur"], "meghalaya": ["meghalaya"], "mizoram": ["mizoram"],
          "nagaland": ["nagaland"], "odisha": ["odisha", "orissa"], "punjab": ["punjab"], "rajasthan": ["rajasthan"], "sikkim": ["sikkim"], "tamil nadu": ["tamilnadu"], "telangana": ["telangana"],
          "tripura": ["tripura"], "uttar pradesh": ["uttarpradesh"], "uttarakhand": ["uttarakhand", "uttaranchal"], "west bengal": ["westbengal", "bengal"], "delhi": ["delhi", "newdelhi"],
          "jammu and kashmir": ["jammu", "kashmir"], "chandigarh": ["chandigarh"], "puducherry": ["puducherry", "pondicherry"], "ladakh": ["ladakh"]}
GOV_CODE = {"tn": "tamil nadu", "up": "uttar pradesh", "mp": "madhya pradesh", "ap": "andhra pradesh", "wb": "west bengal", "hp": "himachal pradesh", "uk": "uttarakhand", "cg": "chhattisgarh", "jk": "jammu and kashmir",
            "ts": "telangana", "tg": "telangana", "pb": "punjab", "hr": "haryana", "ka": "karnataka", "mh": "maharashtra", "gj": "gujarat", "rj": "rajasthan", "br": "bihar", "od": "odisha", "as": "assam"}
STATE_FIX = {"tamilnadu": "tamil nadu", "u p": "uttar pradesh", "up": "uttar pradesh", "orissa": "odisha", "pondicherry": "puducherry", "chattisgarh": "chhattisgarh", "new delhi": "delhi", "jammu kashmir": "jammu and kashmir",
             "punjab region": "punjab", "nct of delhi": "delhi", "uttaranchal": "uttarakhand", "the government of nct of delhi": "delhi", "jammu  kashmir": "jammu and kashmir", "telengana": "telangana"}
# place words that are also everyday hospital / person names: never read as a place
AMBIG = set("""krishna ganga salem bharat north south east west central india indian nagar road city hospital rama ram sagar unity vijaya vijay chandra raja anand shanti janta seva
hope star medical kota karur durg mandi banda bid pali una mau daman diu washim patan morbi nadia doda reasi""".split())


def build_places(F):
    """place word -> state.  Sources: state names, big cities of the facility table (10+ rows, 90% in one state), PIN-directory districts."""
    P = {}
    try:
        pin = pd.concat([pd.read_csv(os.path.join(ROOT, f), dtype=str, usecols=["district", "statename"]) for f in ("indiaPincodeDirectory1.csv", "indiaPincodeDirectory2.csv")]).dropna()
        pin["d"] = pin.district.map(norm_name); pin["s"] = pin.statename.map(norm_name).replace(STATE_FIX)
        g = pin.groupby("d").s.agg(lambda x: float(x.value_counts(normalize=True).iloc[0])); gs = pin.groupby("d").s.agg(lambda x: x.value_counts().index[0])
        for d in g.index:
            if g[d] >= 0.95 and gs[d] in STATES and len(d.replace(" ", "")) >= 5: P[d.replace(" ", "")] = gs[d]
    except Exception as e:
        print("PIN directory not used:", e)
    c = F[F.state_n.isin(STATES)].assign(c=F.address_city.map(norm_name)); c = c[c.c.str.len() >= 4]
    g = c.groupby("c").state_n.agg(["size", lambda x: x.value_counts(normalize=True).iloc[0], lambda x: x.value_counts().index[0]]); g.columns = ["n", "share", "st"]
    for d, r in g[(g.n >= 10) & (g.share >= 0.9)].iterrows(): P[d.replace(" ", "")] = r.st
    for st, al in STATES.items():
        for a in al: P[a] = st
    P.update({"bangalore": "karnataka", "bengaluru": "karnataka", "bombay": "maharashtra", "calcutta": "west bengal", "madras": "tamil nadu", "gurgaon": "haryana", "gurugram": "haryana", "noida": "uttar pradesh",
              "vizag": "andhra pradesh", "trivandrum": "kerala", "cochin": "kerala", "kochi": "kerala", "mysore": "karnataka", "baroda": "gujarat", "pune": "maharashtra", "agra": "uttar pradesh", "goa": "goa",
              "dwarka": "delhi", "secunderabad": "telangana", "navimumbai": "maharashtra", "thane": "maharashtra"})
    for a in AMBIG: P.pop(a, None)
    return P


BIGRAM_OK = {a for al in STATES.values() for a in al} | {"newdelhi", "navimumbai", "tamilnadu", "andhrapradesh", "madhyapradesh", "uttarpradesh", "himachalpradesh", "westbengal", "arunachalpradesh"}


def url_places(host, path, P, skip=frozenset()):
    """States named in an address: whole words of the path, word pairs, and labels of the host (nashik.gov.in, health.delhi.gov.in)."""
    t = toks(path); out = set()
    for i, x in enumerate(t):
        if x in P and x not in skip: out.add(P[x])
        if i + 1 < len(t) and (x + t[i + 1]) in BIGRAM_OK and (x + t[i + 1]) in P and x not in skip: out.add(P[x + t[i + 1]])
    lab = host.split(".")
    for x in lab[:-1]:
        for y in toks(x):
            if y in P and y not in skip: out.add(P[y])
        if x in P and x not in skip: out.add(P[x])
    if host.endswith((".gov.in", ".nic.in")):
        for x in lab[:-2]:
            for y in toks(x):
                if y in GOV_CODE: out.add(GOV_CODE[y])
                for code, st in GOV_CODE.items():        # tnhealth, upnrhm, mphealth, cghealth, wbhealth
                    if re.fullmatch(code + r"(health|nrhm|nhm|medical|hfw|govt|police|esic)\w*", y): out.add(st)
    return out


# ---------------------------------------------------------------- service words inside address paths (claim level)
SVC_URL = {
 "dialysis": r"dialysis|hemodialysis|haemodialysis|nephrolog", "emergency_obstetric": r"obstetric|maternity|caesarean|cesarean|c section|lscs|labour room|labor room|high risk pregnancy|normal delivery|delivery",
 "nicu": r"nicu|neonat|newborn|sncu", "cardiac_surgery": r"cardiac surg|cardio ?thoracic|ctvs|cabg|bypass|heart surg|open heart|cardiovascular surg", "cath_lab": r"cath ?lab|angioplast|angiograph|interventional cardiolog|catheteri",
 "burn_care": r"burns?\b", "blood_bank": r"blood ?bank|blood cent|transfusion", "mri": r"\bmri\b|magnetic resonance", "ct_scan": r"\bct ?scan|computed tomograph|\bct\b", "icu": r"\bicu\b|intensive care|critical care",
 "trauma_emergency": r"trauma|emergency|casualty", "cataract_surgery": r"cataract|phaco", "chemotherapy": r"chemo|oncolog", "radiotherapy": r"radiotherap|radiation oncolog|linac", "ventilator": r"ventilat",
 "stroke": r"stroke|thrombolys", "snakebite_poison": r"snake|poison|venom", "tb": r"\btb\b|tubercul|dots", "ortho_surgery": r"joint replace|knee replace|hip replace|arthroplast|arthroscop|orthop"}


SVC_DEPT = {"dialysis": r"nephrolog|kidney|renal", "emergency_obstetric": r"gyna?ecolog|obstetric|\bobg\b|obgyn|maternity|women", "nicu": r"neonat|pa?ediatric|child", "cardiac_surgery": r"cardiac|cardiolog|heart|ctvs|cardio",
            "cath_lab": r"cardiac|cardiolog|heart|cardio", "burn_care": r"plastic surg|burn", "blood_bank": r"blood", "mri": r"radiolog|imaging|\bmri\b", "ct_scan": r"radiolog|imaging|\bct\b", "icu": r"critical care|intensive|\bicu\b|emergency",
            "trauma_emergency": r"trauma|emergency|casualty|accident", "cataract_surgery": r"ophthalm|cataract|eye", "chemotherapy": r"oncolog|cancer|chemo", "radiotherapy": r"oncolog|cancer|radiation|radiotherap",
            "ventilator": r"critical care|intensive|\bicu\b|ventilat", "stroke": r"neurolog|stroke|neuro", "snakebite_poison": r"toxicolog|poison|snake|emergency", "tb": r"pulmonolog|chest|\btb\b|tubercul|respiratory",
            "ortho_surgery": r"orthop|joint|arthro"}


# ---------------------------------------------------------------- per record
def parse_record(r, P):
    keys, alln = name_keys(r["name"], r.address_city)
    oh, _ = split_url(r.officialWebsite) if r.officialWebsite else ("", "")
    od = reg_domain(oh) if oh else ""
    if od in SOCIAL or od in JOURNAL: od = ""
    urls = list(dict.fromkeys([u for u in list(r.source_urls_l) + list(r.websites_l) if isinstance(u, str) and "." in u]))
    rows = []
    for u in urls:
        h, p = split_url(u); d = reg_domain(h); sq = re.sub(r"[^a-z0-9]", "", h + p)
        kind = "social" if d in SOCIAL else ("own" if (od and d == od) or mentions(sq, keys) else "foreign")
        rows.append({"url": u, "host": h, "path": p, "dom": d, "kind": kind, "on_official": int(bool(od) and d == od), "journal": int(d in JOURNAL), "gov": int(h.endswith((".gov.in", ".nic.in"))),
                     "states": url_places(h, p, P, alln) if kind != "social" else set(), "ptoks": toks(p)})
    return keys, alln, od, rows


LISTY = re.compile(r"\b(hospitals|clinics|doctors|list|lists|listing|directory|directories|network|empanelled|empaneled|cashless|best|top|near|nearby|category|search)\b")
PROFILE = re.compile(r"\b(clinic|hospital|doctor|dr|profile|centre|center)\b")


OTHER_SITE_MAX = 10
ROW_COLS = ["own_share", "n_foreign", "n_foreign_log", "n_domains_log", "n_other_sites", "other_sites_2plus", "other_sites_strict_2plus", "n_other_hosts_log", "other_profiles_3plus", "n_list_pages_log",
            "directory_share", "journal_share", "gov_share", "n_url_other_state", "url_other_state_share", "url_other_state_2plus", "most_urls_other_state", "gov_other_state",
            "email_mismatch", "fb_slug_other", "site_not_name", "site_shared_by_records", "contact_conflicts", "has_official_site", "no_own_url", "n_source_ids_log", "web_page_share", "n_source_types"]
CLAIM_COLS = ["svc_in_own_url", "svc_on_official_site", "dept_on_official_site", "svc_in_foreign_url", "svc_only_in_foreign_url", "svc_on_other_facility_page", "svc_on_doctor_page", "svc_on_list_page", "svc_in_any_url"]


def row_features(r, P, shared, offn=None):
    keys, alln, od, U = parse_record(r, P)
    st = r.state_n if r.state_n in STATES else P.get(str(r.state_n).replace(" ", ""), P.get(norm_name(r.address_city).replace(" ", "")))   # state cell sometimes holds a district
    ns = [u for u in U if u["kind"] != "social"]; own = [u for u in ns if u["kind"] == "own"]; fo = [u for u in ns if u["kind"] == "foreign"]
    n = len(ns); f = {"uid": r.unique_id}
    f["own_share"] = len(own) / n if n else np.nan                       # share of non-social addresses that are the record's own
    f["no_own_url"] = int(n > 0 and not own)
    f["n_foreign"] = len(fo)                                              # distinct addresses about something else
    f["n_foreign_log"] = np.log1p(len(fo))
    f["n_domains_log"] = np.log1p(len({u["dom"] for u in ns}))
    f["n_foreign_domains_log"] = np.log1p(len({u["dom"] for u in fo}))
    # OTHER ORGANISATIONS' WEBSITES: foreign addresses on a site that at most 10 of the 9,989 records carry.
    # A directory (practo, justdial ...) is on hundreds of records; a site on a handful of records is somebody's own website.
    oth = {u["dom"] for u in fo if shared.get(u["dom"], 0) <= OTHER_SITE_MAX and not u["journal"]}
    f["n_other_sites"] = len(oth); f["other_sites_2plus"] = int(len(oth) >= 2)
    f["other_sites_strict_2plus"] = int(len({u["dom"] for u in fo if shared.get(u["dom"], 0) <= 3 and not u["gov"] and not u["journal"]}) >= 2)   # stricter: on 3 records at most, no government sites
    f["n_other_hosts_log"] = np.log1p(len({u["host"] for u in fo if shared.get(u["dom"], 0) < 30 and not u["journal"]}))                            # wider: distinct hosts, sites on < 30 records
    # kinds of foreign address: a list of many facilities / one other provider's profile on a directory
    fdir = [u for u in fo if shared.get(u["dom"], 0) >= 30 and not u["gov"] and not u["journal"]]
    lst = [u for u in fo if LISTY.search(" ".join(u["ptoks"]))]
    prof = [u for u in fdir if u not in lst and PROFILE.search(" ".join(u["ptoks"]))]
    f["n_list_pages_log"] = np.log1p(len(lst)); f["n_other_profiles_log"] = np.log1p(len(prof)); f["other_profiles_3plus"] = int(len(prof) >= 3)
    # a second website that spells the record's name but is not the official one (sister organisation / copy)
    f["own_alt_site"] = int(any(u["kind"] == "own" and not u["on_official"] and shared.get(u["dom"], 0) <= 3 for u in ns))
    f["journal_share"] = np.mean([u["journal"] for u in ns]) if n else np.nan
    f["gov_share"] = np.mean([u["gov"] for u in ns]) if n else np.nan
    f["directory_share"] = np.mean([int(shared.get(u["dom"], 0) >= 30 and not u["gov"] and not u["journal"]) for u in ns]) if n else np.nan
    # places
    pl = [u for u in ns if u["states"]]
    other = [u for u in pl if st and st not in u["states"]]
    f["n_url_other_state"] = len(other) if st else np.nan
    f["url_other_state_share"] = (len(other) / len(pl)) if (st and pl) else (0.0 if st else np.nan)
    f["url_other_state_2plus"] = int(len(other) >= 2) if st else np.nan
    f["n_url_states"] = len(set().union(*[u["states"] for u in pl])) if pl else 0
    govo = [u for u in other if u["gov"]]
    f["gov_other_state"] = int(len(govo) >= 1) if st else np.nan
    f["n_gov_other_state"] = len(govo) if st else np.nan
    f["most_urls_other_state"] = int(len(other) >= 3 and len(other) / len(pl) >= 0.5) if st else np.nan
    f["foreign_other_state"] = len([u for u in other if u["kind"] == "foreign"]) if st else np.nan
    # contact identity
    sqn = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
    em = str(r.email).lower() if isinstance(r.email, str) else ""
    em = em if "@" in em and "protected" not in em else ""
    if em:
        loc, ed = em.split("@")[0], re.sub(r"[^a-z0-9.\-]", "", em.split("@")[-1])
        ok = (od and reg_domain(ed) == od) or mentions(sqn(em), keys) or any(k[:5] in sqn(loc) for k in keys)
        f["email_matches"] = int(bool(ok)); f["email_mismatch"] = int(not ok and bool(keys))
        f["email_own_domain"] = int(bool(od) and reg_domain(ed) == od)
    else:
        f["email_matches"] = 0; f["email_mismatch"] = 0; f["email_own_domain"] = 0
    fb = str(r.facebookLink).lower() if isinstance(r.facebookLink, str) else ""
    slug = re.sub(r"\d{6,}", "", split_url(fb)[1]) if "facebook" in fb else ""
    slug_words = [t for t in toks(slug) if len(t) >= 4 and t not in ("pages", "people", "profile")]
    f["fb_slug_other"] = int(bool(slug_words) and bool(keys) and not mentions(sqn(slug), keys) and not any(k in sqn(slug) for k in keys))
    f["site_matches_name"] = int(bool(od) and bool(keys) and any(k[:5] in sqn(od) for k in keys))
    f["has_official_site"] = int(bool(od))
    f["site_not_name"] = int(bool(od) and bool(keys) and not f["site_matches_name"])                 # official website spells some other name
    f["site_shared_by_records"] = int(bool(od) and offn is not None and offn.get(od, 0) >= 2)          # same official website on 2+ records (chain or host hospital)
    f["contact_conflicts"] = f["email_mismatch"] + f["fb_slug_other"] + int(bool(od) and bool(keys) and not f["site_matches_name"] and shared.get(od, 0) <= 3)
    # source ids / types
    ids = list(r.source_ids_l); ty = Counter(r.source_types_l); nt = sum(ty.values())
    f["n_source_ids_log"] = np.log1p(len(set(ids)))
    f["web_page_share"] = ty.get("dynamic", 0) / nt if nt else np.nan      # 'dynamic' = scraped web pages; 'overture' = map listing
    f["n_source_types"] = len(ty)
    f["has_name_key"] = int(bool(keys))
    return f, U, alln


LISTPAT = re.compile(r"\b(in|near|nct|doctors|hospitals|clinics|categories|category|best|top|list|listing|search|network|cashless)\b")
FACPAT = re.compile(r"\b(hospital|clinic|nursing|home|centre|center|institute)\b")
DOCPAT = re.compile(r"\b(dr|doctor)\b")


def page_kind(u):
    """A foreign address is a LIST of many providers, ONE other facility's page, or ONE doctor's page."""
    t = re.sub(r"^directories (hospitals|doctors) ", "", " ".join(u["ptoks"]))
    if LISTPAT.search(t): return "list"
    if DOCPAT.search(t): return "doctor"
    return "facility" if FACPAT.search(t) else "other"


def claim_features(uid, services, U, alln):
    out = []
    ns = [u for u in U if u["kind"] != "social"]
    # address text with every word of the record's own name removed, and every word that starts like one (nephro / nephrology) -- leak guard
    stems = {n[:5] for n in alln if len(n) >= 4}
    txt = [(u, " ".join(t for t in u["ptoks"] if t not in alln and t[:5] not in stems)) for u in ns]
    for s in services:
        rx = re.compile(SVC_URL[s]) if s in SVC_URL else None
        hit = [u for u, t in txt if rx and rx.search(t)]
        o = [u for u in hit if u["kind"] == "own"]; fo = [u for u in hit if u["kind"] == "foreign"]
        rd = re.compile(SVC_DEPT[s]) if s in SVC_DEPT else None
        dep = [u for u, t in txt if rd and rd.search(t) and u["on_official"]]
        out.append({"uid": uid, "service": s, "dept_on_official_site": int(bool(dep)), "svc_on_official_site": int(any(u["on_official"] for u in o)), "svc_in_own_url": int(bool(o)), "svc_in_foreign_url": int(bool(fo)), "svc_in_any_url": int(bool(hit)),
                    "svc_only_in_foreign_url": int(bool(fo) and not o),
                    "svc_on_other_facility_page": int(any(page_kind(u) == "facility" for u in fo)), "svc_on_doctor_page": int(any(page_kind(u) == "doctor" for u in fo)),
                    "svc_on_list_page": int(any(page_kind(u) == "list" for u in fo)), "svc_only_on_other_facility_page": int(any(page_kind(u) == "facility" for u in fo) and not o),
                    "_own_hit": o[0]["url"] if o else "", "_foreign_hit": fo[0]["url"] if fo else ""})
    return out


def main():
    F = load_facilities()
    F["state_n"] = F.address_stateOrRegion.map(norm_name).replace(STATE_FIX)
    P = build_places(F)
    # how many records carry each domain (directory = shared by many rows)
    shared = Counter()
    for a, b in zip(F.source_urls_l, F.websites_l):
        for d in {reg_domain(split_url(u)[0]) for u in set(a) | set(b) if isinstance(u, str) and "." in u}: shared[d] += 1
    offn = Counter()
    for w in F.officialWebsite:
        if w:
            d = reg_domain(split_url(w)[0])
            if d not in SOCIAL and d not in JOURNAL: offn[d] += 1
    if "--show" in sys.argv:
        r = F[F.unique_id.str.startswith(sys.argv[sys.argv.index("--show") + 1])].iloc[0]; f, U, _ = row_features(r, P, shared, offn)
        print(r["name"], "|", r.address_city, "|", r.state_n, "| official:", r.officialWebsite, "| email:", r.email, "| fb:", r.facebookLink)
        for u in U: print(f"  {u['kind']:8s} {','.join(sorted(u['states'])) or '-':22s} {u['url'][:110]}")
        print({k: round(float(f[k]), 3) for k in ROW_COLS})
        for c in claim_features(r.unique_id, list(SVC_URL), U, _): print("  ", c["service"], {k: c[k] for k in CLAIM_COLS if c[k]}, c["_own_hit"][:80], c["_foreign_hit"][:80]) if any(c[k] for k in CLAIM_COLS) else None
        return
    C = pd.read_csv(os.path.join(OUT, "claims.csv"), usecols=["uid", "service"]); svc = C.groupby("uid").service.apply(list).to_dict()
    R, K = [], []
    for _, r in F.iterrows():
        f, U, alln = row_features(r, P, shared, offn); R.append(f)
        if r.unique_id in svc: K += claim_features(r.unique_id, svc[r.unique_id], U, alln)
    R = pd.DataFrame(R); K = pd.DataFrame(K)
    R = R[["uid"] + ROW_COLS]; K = K[["uid", "service"] + CLAIM_COLS]
    R.round(4).to_csv(os.path.join(OUT, "feat_sources_row.csv"), index=False); K.to_csv(os.path.join(OUT, "feat_sources_claim.csv"), index=False)
    print("rows", len(R), "| claims", len(K), "| place words", len(P))
    print(R.drop(columns="uid").describe().T[["count", "mean", "50%", "max"]].round(3).to_string())
    print(K.drop(columns=["uid", "service"]).mean().round(4).to_string())


if __name__ == "__main__":
    main()
''')

### `pipeline/trust.py`
Labels, logistic-regression fit, grouped cross-validation, bootstrap ranges.

In [ ]:
write('pipeline/trust.py', r'''
"""Step 4: trust score for every (facility, service) claim.

    odds  = BASE x E x C          trust = 100 * odds / (1 + odds)
    E (evidence)  = tier multiplier x coreq multiplier ^ (share of co-requirements found) x own-site-page multiplier
    C (coherence) = product of one multiplier (< 1) per warning sign on the row

Inputs (all rebuilt from raw data by other pipeline steps): claims.csv, location.csv, feat_attribution_row.csv,
feat_sources_row.csv, feat_sources_claim.csv.
Weights are logistic-regression coefficients. Labels: name-based control groups (likely real = the name states the service;
likely false = single-specialty facility claiming acute care), corrected by two independent readers of the likely-false group
and one reader of a sample of the likely-real group (analysis/labels/). Names are never a feature.
Trust is an index for ranking and banding, not a calibrated probability.
"""
import sys, os, json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import ROOT, OUT, load_facilities

BROAD = r"multi|general|maternity|nursing home|medical college|trauma|heart|cardiac|kidney|ortho|surgical|children|institute of medical|super ?special|poly ?clinic|hospitals? and|& ?hospital|research cent"
NEG = {"eye": r"\beye\b|netralaya|nethralaya|ophthal|\bvision\b|retina|lasik|drishti|\bnetra\b", "dental": r"dental|dentist|\btooth|teeth|smile|orthodont",
       "skin_hair": r"\bskin\b|derma|\bhair\b|cosmet|aesthetic", "physio": r"physio", "alt_med": r"ayurved|homoeo|homeo|unani|\bsiddha\b|naturopath|panchakarma"}
POS = {"dialysis": r"dialysis|kidney|nephro|\brenal", "emergency_obstetric": r"maternity|matern|women|obstet|gyna?ec|prasuti|zenana", "nicu": r"children|child ?care|pa?ediatric|neonat|newborn|nicu",
       "cardiac_surgery": r"heart|cardiac|cardio", "cath_lab": r"heart|cardiac|cardio", "burn_care": r"\bburn", "blood_bank": r"blood ?bank|blood cent"}
FIT_SERVICES = list(POS)
EVID = {"t2": "medium evidence: a named procedure or unit", "t3": "strong evidence: equipment or a unit, with a quantity or in two fields",
        "coreq": "share of supporting items found (e.g. operating theatre, anaesthesia and blood for a C-section)", "ownurl": "the facility's own website has a page named after this service"}
SIGNS = {"all_differ": "map point, postcode and city all point to different places", "far2": "the text names two or more towns over 60 km away",
         "other_sites_2plus": "the row was built from the websites of two or more other organisations"}
FEATS = list(EVID) + list(SIGNS)


def features():
    o = lambda f: pd.read_csv(os.path.join(OUT, f))
    F = load_facilities(); nm = F.name.str.lower(); F["neg"] = ""
    for k, rx in NEG.items(): F.loc[nm.str.contains(rx, regex=True) & (F.neg == ""), "neg"] = k
    F.loc[(F.facilityTypeId == "dentist") & (F.neg == ""), "neg"] = "dental"; F.loc[nm.str.contains(BROAD, regex=True), "neg"] = ""
    C = o("claims.csv").merge(F[["unique_id", "name", "address_city", "facilityTypeId", "operatorTypeId", "neg"]], left_on="uid", right_on="unique_id")
    C = C.merge(o("location.csv")[["uid", "all_differ"]], on="uid", how="left").merge(o("feat_attribution_row.csv")[["uid", "n_other_places_city"]], on="uid", how="left")
    C = C.merge(o("feat_sources_row.csv")[["uid", "other_sites_2plus"]], on="uid", how="left").merge(o("feat_sources_claim.csv")[["uid", "service", "svc_in_own_url"]], on=["uid", "service"], how="left")
    C["t2"] = (C.tier == 2).astype(int); C["t3"] = (C.tier == 3).astype(int); C["coreq"] = C.coreq_found / C.coreq_total
    C["far2"] = (C.n_other_places_city.fillna(0) >= 2).astype(int); C["ownurl"] = C.svc_in_own_url.fillna(0)
    for c in ["all_differ", "other_sites_2plus"]: C[c] = C[c].fillna(0)
    nml = C.name.str.lower(); C["y"] = np.nan
    for s, rx in POS.items(): C.loc[(C.service == s) & nml.str.contains(rx, regex=True) & (C.neg == ""), "y"] = 1
    C.loc[(C.neg != "") & C.service.isin(FIT_SERVICES) & ~((C.neg == "skin_hair") & (C.service == "burn_care")), "y"] = 0
    C["claim_id"] = C.uid.str[:8] + ":" + C.service; L = os.path.join(ROOT, "analysis", "labels")
    if os.path.exists(os.path.join(L, "labels_likely_false_A.csv")):
        a = pd.read_csv(os.path.join(L, "labels_likely_false_A.csv")).merge(pd.read_csv(os.path.join(L, "labels_likely_false_B.csv")), on="claim_id", suffixes=("_a", "_b"))
        a["v"] = np.where((a.verdict_a == "false") & (a.verdict_b == "false"), 0.0, np.where((a.verdict_a == "real") & (a.verdict_b == "real"), 1.0, np.nan))
        p = pd.read_csv(os.path.join(L, "labels_likely_real.csv")); p["v"] = p.verdict.map({"real": 1.0, "false": 0.0})
        rd = pd.concat([a[["claim_id", "v"]], p[["claim_id", "v"]]]).assign(was_read=1)
        C = C.merge(rd, on="claim_id", how="left"); C.loc[C.was_read == 1, "y"] = C.loc[C.was_read == 1, "v"]
    return C


def fit(S):
    return LogisticRegression(max_iter=2000).fit(S[FEATS].values, S.y.values)


def main():
    C = features(); S = C[C.y.notna()].reset_index(drop=True)
    X, y, g = S[FEATS].values, S.y.values, S.uid.values; hosp = (S.facilityTypeId == "hospital").values
    oof = np.zeros(len(S))
    for tr, te in GroupKFold(5).split(X, y, g): oof[te] = LogisticRegression(max_iter=2000).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    m = fit(S); w = dict(zip(FEATS, m.coef_[0])); b0 = float(m.intercept_[0])
    rng = np.random.default_rng(3); ug = np.unique(g); B = []
    for _ in range(300):
        idx = np.concatenate([np.where(g == u)[0] for u in rng.choice(ug, len(ug))])
        if len(set(y[idx])) == 2: B.append(LogisticRegression(max_iter=2000).fit(X[idx], y[idx]).coef_[0])
    lo, hi = np.percentile(B, 5, 0), np.percentile(B, 95, 0)
    print("labelled claims:", len(S), "| real", int(y.sum()), "| false", int((y == 0).sum()))
    print("cross-validated AUC: all", round(roc_auc_score(y, oof), 3), "| hospitals only", round(roc_auc_score(y[hosp], oof[hosp]), 3))
    print("multiplier on the odds (90% bootstrap range):")
    for i, f in enumerate(FEATS): print(f"  {f:18s} x{np.exp(w[f]):6.2f}  ({np.exp(lo[i]):.2f} to {np.exp(hi[i]):.2f})")
    print(f"  BASE               x{np.exp(b0):6.2f}")
    # honesty check: refit without the known bad record, then score it
    AV = "b8a5401f-42f1-422a-8cd9-686a15b4cb76"; m2 = fit(S[S.uid != AV]); av = C[(C.uid == AV) & (C.service == "dialysis")]
    C["E"] = np.exp(sum(w[k] * C[k] for k in EVID)); C["C_row"] = np.exp(sum(w[k] * C[k] for k in SIGNS))
    C["trust"] = (100 * m.predict_proba(C[FEATS].values)[:, 1]).round(1)
    if len(av): print(f"known bad record, dialysis claim: trust {float(C.loc[av.index, 'trust'].iloc[0])} | when the record is left out of fitting: {100 * m2.predict_proba(av[FEATS].values)[0, 1]:.1f}")
    so = pd.DataFrame({"s": 100 * oof, "y": y}).sort_values("s", ascending=False); so["prec"] = so.y.expanding().mean()
    hi_thr = float(so[so.prec >= 0.95].s.min()); sa = so.sort_values("s"); sa["fal"] = (1 - sa.y).expanding().mean(); lo_thr = float(sa[sa.fal >= 0.5].s.max())
    C["band"] = np.where(C.trust >= hi_thr, "high", np.where(C.trust < lo_thr, "low", "medium"))
    print(f"bands: high >= {hi_thr:.0f} | low < {lo_thr:.0f} | all claims: {C.band.value_counts().to_dict()}")
    def reasons(r):
        out = [f"evidence tier {int(r.tier)} of 3", f"{int(r.coreq_found)} of {int(r.coreq_total)} supporting items found" + (f" ({r.coreq_list})" if isinstance(r.coreq_list, str) and r.coreq_list else "")]
        if r.ownurl == 1: out.append(EVID["ownurl"])
        return " | ".join(out + [v for k, v in SIGNS.items() if r[k] == 1])
    C["reasons"] = C.apply(reasons, axis=1)
    keep = ["uid", "name", "address_city", "facilityTypeId", "operatorTypeId", "service", "trust", "band", "E", "C_row", "tier", "coreq_found", "coreq_total", "coreq_list", "ownurl"] + list(SIGNS) + ["quote_field", "quote", "reasons"]
    C[keep].round(3).to_csv(os.path.join(OUT, "trust.csv"), index=False)
    json.dump({"base": b0, "weights": w, "features": FEATS, "high_threshold": hi_thr, "low_threshold": lo_thr, "auc_cv": float(roc_auc_score(y, oof)), "auc_cv_hospitals": float(roc_auc_score(y[hosp], oof[hosp])), "n_labelled": int(len(S))}, open(os.path.join(OUT, "weights.json"), "w"), indent=1)
    ACUTE = ["dialysis", "emergency_obstetric", "nicu", "cardiac_surgery", "burn_care", "cath_lab", "icu", "ventilator", "blood_bank", "stroke", "chemotherapy", "radiotherapy"]
    ng = C[(C.neg != "") & C.service.isin(ACUTE)]; print("single-specialty facilities, acute-care claims:", len(ng), "| high", int((ng.band == "high").sum()), "| medium", int((ng.band == "medium").sum()), "| low", int((ng.band == "low").sum()))
    L = os.path.join(ROOT, "analysis", "labels", "labels_general.csv")
    if os.path.exists(L):
        G = pd.read_csv(L).merge(C[["claim_id", "trust", "band"]], on="claim_id"); print("general-hospital packet read by an independent reader (not used for fitting):"); print(pd.crosstab(G.band, G.verdict).to_string())
        print("  AUC, real vs unclear-or-false:", round(roc_auc_score((G.verdict == "real").astype(int), G.trust), 3))


if __name__ == "__main__":
    main()
''')

### `pipeline/musthave.py`
Must-have items per service, each kept only if a medical refutation check and a data check both passed.

In [ ]:
write('pipeline/musthave.py', r'''
"""Must-have items per service.

A must-have is something the service cannot be delivered without. It is not a sign of quality.
Each entry is a group of co-requirement keys from ontology.py. Finding ANY key of the group in the record satisfies it.

Rule applied in export_app.py: if a must-have group is not found in the record, the claim cannot be shown as
"strong evidence" (trust stops at CAP, just under the strong threshold) and the card tells the user to confirm
the item by phone first. The facility is never removed for this reason: a silent record is not a "no".

An item is on this list only if it passed two tests:
  (a) medical: a web-research pass proposed it from Indian and international standards, and a second pass that
      tried to refute it ("is there a common, legitimate way to deliver the service without it?") failed to;
  (b) data: at least one in three strong-evidence records for the service mention it. If almost no record
      mentions an item (oxygen for an ICU, the water plant for dialysis), silence is the norm and tells us nothing.

Items that passed (a) but not (b) are listed in DROPPED so the decision can be audited.
The shares were re-measured after the co-requirement regexes were tightened (2026-11-22): bedside monitors for an ICU
fell to 26% once Holter, fetal and glucose monitors stopped counting, and heart surgeon to 30% once the bare "M.Ch"
match was removed, so both left the list.
Not reviewed by a clinician.
"""

MUST = {
    "emergency_obstetric": [
        {"keys": ["operation_theatre"],
         "why": "A C-section cannot be done outside an operating theatre. Staff and blood can be brought in; a theatre cannot.",
         "src": "https://www.unfpa.org/resources/setting-standards-emergency-obstetric-and-newborn-care"},
    ],
    "cardiac_surgery": [
        {"keys": ["cardiac_OT", "heart_lung_machine"],
         "why": "Heart surgery needs an operating theatre, with a heart-lung machine ready even when the surgeon plans not to use one.",
         "src": "https://cdnbbsr.s3waas.gov.in/s3169779d3852b32ce8b1a1724dbf5217d/uploads/2024/08/20240828153687657.pdf"},
        {"keys": ["cardiac_ICU", "ventilator"],
         "why": "After heart surgery every patient needs intensive care with a breathing machine ready.",
         "src": "https://cdnbbsr.s3waas.gov.in/s3169779d3852b32ce8b1a1724dbf5217d/uploads/2024/08/20240828153687657.pdf"},
    ],
    "burn_care": [
        {"keys": ["operation_theatre"],
         "why": "Deep burns need surgery under anaesthesia; the national burn guideline tells hospitals without a theatre to refer the patient on.",
         "src": "https://www.dghs.mohfw.gov.in/uploads/assets/S1Ipab4025VceQpFs7Qa6THwh07NLmggStsg1cDD.pdf"},
    ],
    "trauma_emergency": [
        {"keys": ["operation_theatre"],
         "why": "A badly bleeding patient can only be saved by immediate surgery; without a theatre a hospital can only stabilise and transfer.",
         "src": "https://www.dghs.mohfw.gov.in/uploads/assets/SwQ7pJMiJepMtqfUQNUJrGOROd584XHIgipqf7gI.pdf"},
        {"keys": ["surgeon_ortho_or_neuro"],
         "why": "Injuries need a general, bone or brain surgeon to operate; a visiting or on-call surgeon counts.",
         "src": "https://www.dghs.mohfw.gov.in/uploads/assets/SwQ7pJMiJepMtqfUQNUJrGOROd584XHIgipqf7gI.pdf"},
    ],
    "cath_lab": [
        {"keys": ["cardiologist"],
         "why": "Angiography and angioplasty are done by a heart doctor in person; a visiting cardiologist counts.",
         "src": "https://pmc.ncbi.nlm.nih.gov/articles/PMC11307489"},
    ],
}

# passed the medical test, failed the data test (share of strong-evidence records that mention it)
DROPPED = {
    "icu": {"oxygen": 0.14, "monitors": 0.26},
    "nicu": {"warmer_or_incubator": 0.17},
    "cardiac_surgery": {"cardiac_surgeon": 0.30},
}

# share of strong-evidence records that mention each must-have in use (the data test, threshold 0.33)
SHARE = {"emergency_obstetric": [0.66], "cardiac_surgery": [0.87, 0.56], "burn_care": [0.81], "trauma_emergency": [0.66, 0.86], "cath_lab": [0.89]}

# proposed, then refuted by the second pass
REFUTED = {
    "dialysis": "RO water plant: peritoneal and bedside dialysis need none. Nephrologist: many public and chain units run with trained doctors and a visiting specialist",
    "emergency_obstetric": "anaesthesia and blood bank: commonly a visiting anaesthetist and blood from a linked bank",
    "nicu": "ventilator and specialist: Level II newborn units legitimately run without them and refer",
    "mri": "radiologist: scans are commonly reported off-site by tele-radiology",
    "burn_care": "skin graft or plastic surgeon: general surgeons graft skin in an ordinary theatre",
    "trauma_emergency": "blood bank: many hospitals operate with blood fetched from a nearby licensed bank",
    "icu": "ventilator: a Level I ICU (ISCCM 2020) may run on non-invasive support and transfer",
}

CAP = 89.9   # highest trust a claim can show when a must-have is not on the record
''')

### `pipeline/export_app.py`
Bands, rules outside the score, ladder quotes, doctors, district context: everything the app shows.

In [ ]:
write('pipeline/export_app.py', r'''
"""Step 5: pack everything the app needs into one JSON file (app/data.json)."""
import sys, os, re, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import ROOT, OUT, load_facilities
from ontology import SERVICE_ONTOLOGY as ONT
import trust as T
from musthave import MUST, CAP

SERVICES = list(ONT)   # all 19 services
# acute services a single-specialty facility (eye, dental, skin, physio, ayurveda/homeopathy) should not offer
OUT_OF_SCOPE = {"dialysis", "emergency_obstetric", "nicu", "cardiac_surgery", "cath_lab", "burn_care", "icu", "ventilator", "blood_bank", "stroke", "radiotherapy", "snakebite_poison"}
HIGH, LOW = 90.0, 70.0
nz = lambda s: re.sub(r"[^A-Z]", "", str(s).upper().replace("&", "AND"))
t = lambda x, n: (re.sub(r"\s+", " ", x).strip()[:n] if isinstance(x, str) else "")

C = T.features(); S = C[C.y.notna()]; m = T.fit(S)
C["trust"] = 100 * m.predict_proba(C[T.FEATS].values)[:, 1]
C["band"] = np.where(C.trust >= HIGH, 2, np.where(C.trust >= LOW, 1, 0))
# honest validation numbers for the app: everything out-of-fold (each claim scored by a model that never saw its facility)
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.model_selection import GroupKFold as _GK
from sklearn.metrics import roc_auc_score as _auc
_S = S.reset_index(drop=True); _X, _y, _g = _S[T.FEATS].values, _S.y.values, _S.uid.values; _o = np.zeros(len(_S))
for _tr, _te in _GK(5).split(_X, _y, _g): _o[_te] = _LR(max_iter=2000).fit(_X[_tr], _y[_tr]).predict_proba(_X[_te])[:, 1]
_b = np.where(_o >= HIGH / 100, 2, np.where(_o >= LOW / 100, 1, 0))
_svc = {s_: round(float(_auc(_y[(_S.service == s_).values], _o[(_S.service == s_).values])), 2) for s_ in _S.service.unique() if (_S.service == s_).sum() >= 50}
VAL = {"auc": round(float(_auc(_y, _o)), 2), "real": [round(100 * float(_y[_b == k].mean())) for k in (0, 1, 2)], "n": [int((_b == k).sum()) for k in (0, 1, 2)],
       "n_read": int((_S.was_read == 1).sum()) if "was_read" in _S else 0, "n_name": int((_S.was_read != 1).sum()) if "was_read" in _S else len(_S),
       "svc_min": min(_svc, key=_svc.get), "svc_min_auc": min(_svc.values()), "svc_max_auc": max(_svc.values())}
_rng = np.random.default_rng(3); _ug = np.unique(_g); _ix = {u: np.where(_g == u)[0] for u in _ug}; _E = C[C.trust >= LOW][T.FEATS].drop_duplicates().values; _P = []
for _ in range(200):
    _i = np.concatenate([_ix[u] for u in _rng.choice(_ug, len(_ug))])
    if len(set(_y[_i])) == 2: _P.append(100 * _LR(max_iter=2000).fit(_X[_i], _y[_i]).predict_proba(_E)[:, 1])
_sd = np.repeat(np.array(_P).std(0), C[C.trust >= LOW].groupby(T.FEATS, sort=False).size().reindex(pd.MultiIndex.from_arrays(_E.T, names=T.FEATS)).values)
TIE_PTS = round(float(np.median(_sd)), 1)
print("score noise (bootstrap SD of trust over refits, shown claims): median", TIE_PTS, "points -> tie window for planned ranking")
print("out-of-fold validation for the app:", VAL, _svc)
lab = C[C.y.notna()]; print("labelled claims by band (share real):", lab.groupby("band").y.agg(["size", "mean"]).round(2).to_dict("index"))
gp = os.path.join(ROOT, "analysis/labels/labels_general.csv")
if os.path.exists(gp):
    G = pd.read_csv(gp).merge(C[["claim_id", "band"]], on="claim_id"); print("independent reader, general hospitals:"); print(pd.crosstab(G.band, G.verdict).to_string())
C["excluded"] = ((C.neg != "") & C.service.isin(OUT_OF_SCOPE) & ~((C.neg == "skin_hair") & (C.service == "burn_care"))).astype(int)
C.loc[(C.service == "cataract_surgery") & ~C.neg.isin(["", "eye"]), "excluded"] = 1
# wider wrong-specialty rule (found by the stress test): the fitted labels use a narrow name rule; for display we also
# leave out dental, eye, skin, physio and ayurveda/homeopathy facilities for every service outside their own field.
_nm = C.name.str.lower().str.split(r" - |,", regex=True).str[0]
_PAT = {"dental": r"dental|dentist|orthodont|\bteeth|\btooth", "alt_med": r"ayurved|homoeo|homeo|unani|\bsiddha\b|naturopath|panchakarma", "eye": r"\beye\b|netralaya|nethralaya|ophthal|retina|lasik", "physio": r"physio", "skin_hair": r"\bskin\b|derma|\bhair\b|cosmet"}
_GEN = r"multi ?special(?!ity dental)|general|super ?special|nursing home|maternity|women|trauma|heart|cardiac|kidney|children|child|institute of medical|poly ?clinic|research cent|medical college"
_COLL = r"(homoeo\w*|homeo\w*|ayurved\w*|unani|dental) (medical )?college"
_ALLOW = {"eye": {"cataract_surgery"}, "skin_hair": {"burn_care"}, "dental": set(), "alt_med": set(), "physio": set()}
_kinds = pd.DataFrame({k: _nm.str.contains(rx, regex=True) for k, rx in _PAT.items()})
_single = _kinds.any(axis=1) & (~_nm.str.contains(_GEN, regex=True) | _nm.str.contains(_COLL, regex=True))
def _out(i):
    ks = [k for k in _PAT if _kinds.at[i, k]]; s_ = C.at[i, "service"]
    return not (any(s_ in _ALLOW[k] for k in ks) or (s_ == "ortho_surgery" and "ortho" in _nm.at[i]))
_new = pd.Series([bool(_single.at[i]) and _out(i) for i in C.index], index=C.index) & (C.excluded == 0)
C.loc[_new, "neg"] = [next(k for k in _PAT if _kinds.at[i, k]) for i in C.index[_new]]; C.loc[_new, "excluded"] = 1
print("wider wrong-specialty rule: extra claims left out", int(_new.sum()), "| of them previously shown (band >= 1)", int((_new & (C.band >= 1)).sum()), "| labelled real among them", int((C.y[_new] == 1).sum()))
C = C[C.service.isin(SERVICES)]
# must-have rule: a claim whose record lacks an indispensable item cannot be "strong"
def missing(r):
    found = set(r.coreq_list.split("|")) if isinstance(r.coreq_list, str) and r.coreq_list else set()
    return [j for j, g in enumerate(MUST.get(r.service, [])) if not (set(g["keys"]) & found)]
C["miss"] = [missing(r) for r in C.itertuples()]; C["raw"] = C.trust
cap = C.miss.map(len).gt(0) & (C.trust > CAP)
print("must-have rule: strong claims before", int((C.band == 2).sum()), "| capped", int(cap.sum()), "| by service:", C[cap].service.value_counts().to_dict())
lab2 = C[C.y.notna() & (C.raw >= HIGH)]; print("  labelled strong claims, share real: must-haves all found", lab2[lab2.miss.map(len) == 0].y.agg(["size", "mean"]).round(3).tolist(), "| a must-have missing", lab2[lab2.miss.map(len) > 0].y.agg(["size", "mean"]).round(3).tolist())
C.loc[cap, "trust"] = CAP; C.loc[cap, "band"] = 1
print("claims exported:", len(C), "| bands:", C.band.value_counts().to_dict(), "| excluded by specialty rule:", int(C.excluded.sum()))

F = load_facilities(); L = pd.read_csv(os.path.join(OUT, "location.csv"))
F = F.merge(L[["uid", "best_lat", "best_lon", "best_state", "best_district", "location_confidence", "loc_class"]], left_on="unique_id", right_on="uid", how="left")
F = F[F.unique_id.isin(C.uid) & F.best_lat.notna()].reset_index(drop=True)
DOC = re.compile(r"\bDr\.?\s+[A-Z][a-zA-Z.]+(?:\s+[A-Z][a-zA-Z.]+){0,3}")
def doctor(r):
    """Up to 4 short sentences on the record that name a doctor (shown as unverified information, never scored)."""
    seen, out = set(), []
    for s in sorted(r.capability_l + r.procedure_l, key=len):
        m = DOC.search(s)
        if m and 25 <= len(s) <= 160 and m.group(0) not in seen:
            seen.add(m.group(0)); out.append(t(s, 160))
        if len(out) == 4: break
    return out
# distance noise: for rows whose three location signals agree, how far the map point sits from the postcode centre.
# Two places whose distances differ by less than the 75th percentile of this cannot be ordered by distance with confidence.
_d = L[L.loc_class.astype(str).str.contains("agree")].d_coord_pin_km.dropna(); TIE = round(float(_d.quantile(.75)), 1)
print("location noise (map point vs postcode centre, agreeing rows): median", round(float(_d.median()), 2), "| 75th percentile", TIE, "km -> tie window for emergency ranking")
places, pidx = [], {}
def place(st, di):
    k = (str(st).title() if isinstance(st, str) else "", str(di).title() if isinstance(di, str) else "")
    if k not in pidx: pidx[k] = len(places); places.append(list(k))
    return pidx[k]
fac = []; fid = {}
for i, r in enumerate(F.itertuples()):
    fid[r.unique_id] = i
    fac.append([t(r.name, 70), t(r.address_city, 30), place(r.best_state, r.best_district), round(r.best_lat, 4), round(r.best_lon, 4), round(float(r.location_confidence), 2) if pd.notna(r.location_confidence) else None,
                r.facilityTypeId or "", "public" if r.operatorTypeId in ("public", "government") else (r.operatorTypeId or ""), t(r.officialPhone, 20), t(r.officialWebsite, 60), doctor(r),
                [int(r.capacity_n) if pd.notna(r.capacity_n) and 1 <= r.capacity_n <= 5000 else None, int(r.numberDoctors_n) if pd.notna(r.numberDoctors_n) and 1 <= r.numberDoctors_n <= 5000 else None,
                 int(r.yearEstablished_n) if pd.notna(r.yearEstablished_n) and 1800 <= r.yearEstablished_n <= 2026 else None]])
# ladder: department -> procedure -> equipment, each with the record's own words
H = pd.read_pickle(os.path.join(OUT, "hits.pkl")); H = H[H["flags"].eq("")]
def win(seg, match, n=120):
    """Cut a sentence to n characters around the matched words."""
    seg = re.sub(r"\s+", " ", seg).strip()
    if len(seg) <= n: return seg
    i = seg.lower().find(match.lower()); a = max(0, min(i - 40, len(seg) - n)) if i >= 0 else 0
    if a > 0: a = seg.find(" ", a) + 1
    cut = seg[a:a + n]; cut = cut[:cut.rfind(" ")] if a + n < len(seg) and " " in cut else cut
    return ("… " if a > 0 else "") + cut + (" …" if a + n < len(seg) else "")
def best(cls):
    d = H[H.cls == cls].assign(q=lambda x: ~x.qty, sh=lambda x: x.seglen < 14, l=lambda x: x.seglen.clip(upper=400)).sort_values(["q", "sh", "l"]).drop_duplicates(["uid", "service"])
    return {(r.uid, r.service): [win(r.seg, r.match), r.field] for r in d.itertuples()}
BP, BI = best("proc"), best("infra")
CQ = {s: {k: re.compile(rx, re.I) for k, rx in ONT[s]["coreq"].items()} for s in ONT}
SEGS = {r.unique_id: [(f, x) for f, l in (("equipment", r.equipment_l), ("capability", r.capability_l), ("procedure", r.procedure_l)) for x in l] + [("description", x) for x in re.split(r"(?<=[.!?])\s+", r.description or "")] for r in F.itertuples()}
NOT_EQUIP = re.compile(r"ologist|surgeon|ian$|obstetric|anaesth|intensivist|round_the_clock|volume|count|backup|ambulance", re.I)
OTHER_ICU = re.compile(r"neonat|\bNICU\b|p(a)?ediatric|\bPICU\b|\bSNCU\b", re.I)
def needquote(uid, s, found):
    """When no sentence describes the unit itself: the record's sentence for one needed item (must-haves first)."""
    order = [k for g in MUST.get(s, []) for k in g["keys"]] + list(CQ[s])
    for k in dict.fromkeys(order):
        if k not in found or NOT_EQUIP.search(k): continue      # this step is about rooms and machines, not staff or opening hours
        for f, x in SEGS.get(uid, []):
            m = CQ[s][k].search(x)
            if m and len(x) >= 14 and not (k == "ICU" and OTHER_ICU.search(x)): return [win(x, m.group(0)), f, list(CQ[s]).index(k)]
    return ["", "", -1]
nsp = lambda x: re.sub(r"[^a-z]", "", str(x).lower())
pretty = lambda x: re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(x)).lower()[:40]
SPEC = dict(zip(F.unique_id, F.specialties_l))
def dept(uid, s):
    """Matching department on the specialties list: 2 = the service's own department, 1 = its parent department."""
    for lvl, key in ((2, "spec_direct"), (1, "spec_related")):
        rx = ONT[s].get(key)
        if not rx: continue
        for x in SPEC.get(uid, []):
            if re.search(rx, nsp(x)): return [pretty(x), lvl]
    return ["", 0]
# doctors read ahead of time from facilities' own websites (pipeline/scrape_doctors.py); demo cities only
_dw = os.path.join(ROOT, "pipeline", "external", "doctors_web.json"); DW = json.load(open(_dw)) if os.path.exists(_dw) else {}
for i, r in enumerate(F.itertuples()):
    w = DW.get(r.unique_id)
    # drop names cut by the reader (a short last word glued to a degree, e.g. "Dr. Shamsul Ho" + "DA" from "HODA"); fix ALL-CAPS
    cap = lambda x_: re.sub(r"\b([A-Z])([A-Z]{2,})\b", lambda m_: m_.group(1) + m_.group(2).lower(), x_)
    # clean-ups found on the demo check: "ent" matched inside words (department, patient); street or hospital words read as names
    spec_ = lambda x_: " / ".join(t_ for t_ in x_.split(" / ") if t_ and t_.lower() != "ent")
    NOTNAME = re.compile(r"\b(the|icu|ward|unit|road|nagar|street|marg|colony|hospital|clinic|cent(re|er)|chowk|lane|sector|block|phase)\b", re.I)
    def deg_(x_):
        """Keep whole degrees only: never end inside an open bracket such as 'FNB ( Vitreo retina'."""
        x_ = re.sub(r"\(\s+", "(", x_).strip(" ,")
        while x_.count("(") > x_.count(")"): x_ = x_[:x_.rfind(",")].strip(" ,") if "," in x_ else re.sub(r"\s*\([^)]*$", "", x_)
        return x_
    ds = [[cap(d_[0]), deg_(d_[1]), spec_(d_[2])[:40], d_[3] if len(d_) > 3 else None, (d_[4][:50] if len(d_) > 4 else "")] for d_ in (w or {}).get("doctors", [])
          if len(d_[0].split()[-1].strip(".")) >= 3 and not NOTNAME.search(d_[0])]
    ds = [d_ for d_ in ds if d_[1] or d_[2]]
    fac[i].append([w["url"], ds[:10], w["fetched"]] if ds else None)
print("doctors from websites: facilities", sum(1 for f in fac if f[12]), "| doctors", sum(len(f[12][1]) for f in fac if f[12]))
claims = {}
for s in SERVICES:
    keys = list(ONT[s]["coreq"].keys()); d = {}
    for r in C[C.service == s].itertuples():
        if r.uid not in fid: continue
        found = set(r.coreq_list.split("|")) if isinstance(r.coreq_list, str) and r.coreq_list else set()
        signs = int(r.all_differ) | (int(r.far2) << 1) | (int(r.other_sites_2plus) << 2)
        row = [round(r.trust, 1), int(r.band), int(r.tier), [keys.index(k) for k in keys if k in found], t(r.quote, 190), r.quote_field, signs, int(r.ownurl), int(r.excluded), r.neg or ""]
        if r.band >= 1 and not r.excluded:   # shown on a card: add the ladder, the missing must-haves and the uncapped score
            row += [dept(r.uid, s) + BP.get((r.uid, s), ["", ""]) + (BI[(r.uid, s)] + [-1] if (r.uid, s) in BI else needquote(r.uid, s, found)), r.miss, round(r.raw, 1)]
        d[fid[r.uid]] = row
    claims[s] = {"coreq": [k.replace("_", " ") for k in keys], "c": d,
                 "must": [{"k": [keys.index(k) for k in g["keys"]], "why": g["why"], "src": g["src"]} for g in MUST.get(s, [])]}
P = pd.read_csv(os.path.join(OUT, "pin_lookup.csv"), dtype={"pincode": str}); P = P[P.lat.notna()]
pins = {r.pincode: [round(r.lat, 3), round(r.lon, 3), place(r.state, r.district)] for r in P.itertuples()}
cities = {}
for (c, st, di), g in F.groupby([F.address_city.str.strip().str.title(), "best_state", "best_district"]):
    if c and len(c) >= 3 and (c not in cities or len(g) > cities[c][3]): cities[c] = [round(g.best_lat.median(), 3), round(g.best_lon.median(), 3), place(st, di), len(g)]
for (st, di), g in P.groupby(["state", "district"]):
    c = str(di).title()
    if c not in cities and len(g) >= 3: cities[c] = [round(g.lat.median(), 3), round(g.lon.median(), 3), place(st, di), 0]
cities = {k: v[:3] for k, v in cities.items()}
N = pd.read_csv(os.path.join(ROOT, "nfhs_5_district_health_indicators.csv")); D = pd.read_csv(os.path.join(ROOT, "analysis/out_nfhs_district.csv"))
N = N.merge(D[["district_name", "state_ut", "st", "di", "fdi"]], on=["district_name", "state_ut"])
_acc = ["institutional_birth_5y_pct", "mothers_who_had_at_least_4_anc_visits_lb5y_pct", "births_attended_by_skilled_hp_5y_10_pct", "child_12_23m_fully_vaccinated_based_on_information_from_eit_pct"]
_idx = N[_acc].apply(pd.to_numeric, errors="coerce").rank(pct=True).mean(axis=1)      # how much basic care actually reaches people here
N["access_group"] = pd.qcut(_idx.rank(method="first"), 3, labels=[0, 1, 2]).astype(int)   # 0 = least reached third of districts
print("access groups (districts):", N.access_group.value_counts().sort_index().to_dict(), "| median facility births by group:", N.assign(v=pd.to_numeric(N.institutional_birth_5y_pct, errors="coerce")).groupby("access_group").v.median().to_dict())
g_ = lambda r, c: (None if pd.isna(pd.to_numeric(r[c], errors="coerce")) else round(float(pd.to_numeric(r[c], errors="coerce")), 1))
nfhs = {}
for _, r in N.iterrows():
    v = [g_(r, "institutional_birth_5y_pct"), g_(r, "institutional_birth_in_public_facility_5y_pct"), g_(r, "births_in_a_public_fac_that_were_delivered_by_csection_5y_pct"), g_(r, "births_in_a_private_fac_that_were_delivered_by_csection_5y_pct"),
         g_(r, "hh_member_covered_health_insurance_pct"), g_(r, "women_age_15_49_who_are_literate_pct"), g_(r, "all_w15_49_who_are_anaemic_pct"), r.district_name.strip(), int(r.access_group),
         g_(r, "w15_19_who_were_already_mothers_or_pregnant_at_the_time_of_pct"), g_(r, "average_out_of_pocket_expenditure_per_delivery_in_a_public_fac"), g_(r, "w20_24_married_before_age_18_years_pct")]
    for k in {r.st + "|" + (r.fdi if isinstance(r.fdi, str) else r.di), r.st + "|" + r.di}: nfhs[k] = v
# renamed districts: facility-side name -> name used by the 2019-21 survey (found by the data check: 11.6% of facilities had no district context)
ALIAS = {"KARNATAKA|BENGALURUURBAN": "KARNATAKA|BANGALORE", "HARYANA|GURUGRAM": "HARYANA|GURGAON", "UTTARPRADESH|PRAYAGRAJ": "UTTARPRADESH|ALLAHABAD", "KARNATAKA|BELAGAVI": "KARNATAKA|BELGAUM",
         "KARNATAKA|MYSURU": "KARNATAKA|MYSORE", "KARNATAKA|KALABURAGI": "KARNATAKA|GULBARGA", "KARNATAKA|BALLARI": "KARNATAKA|BELLARY", "ASSAM|KAMRUPMETRO": "ASSAM|KAMRUPMETROPOLITAN",
         "WESTBENGAL|PARAGANASNORTH": "WESTBENGAL|NORTHTWENTYFOURPARGANA", "WESTBENGAL|PARAGANASSOUTH": "WESTBENGAL|SOUTHTWENTYFOURPARGANA", "MAHARASHTRA|RAIGAD": "MAHARASHTRA|RAIGARH",
         "UTTARPRADESH|AYODHYA": "UTTARPRADESH|FAIZABAD", "HARYANA|NUH": "HARYANA|MEWAT", "KARNATAKA|SHIVAMOGGA": "KARNATAKA|SHIMOGA", "KARNATAKA|TUMAKURU": "KARNATAKA|TUMKUR"}
_n0 = len(nfhs)
for k, v in ALIAS.items():
    if v in nfhs and k not in nfhs: nfhs[k] = nfhs[v]
_fk = pd.Series([nz(places[f[2]][0]) + "|" + nz(places[f[2]][1]) for f in fac])
print("district context: aliases added", len(nfhs) - _n0, "| facilities with survey context", int(_fk.isin(set(nfhs)).sum()), "of", len(fac))
pm = pd.read_csv(os.path.join(ROOT, "pipeline/external/pmndp_centres.csv")); pmndp = {}
for r in pm.itertuples(): pmndp.setdefault(nz(r.state) + "|" + nz(r.district), []).append(t(r.facility, 60))
w = dict(zip(T.FEATS, np.exp(m.coef_[0]).round(2).tolist()))
out = {"fac": fac, "places": places, "claims": claims, "pins": pins, "cities": cities, "nfhs": nfhs, "pmndp": pmndp,
       "meta": {"tie_pts": TIE_PTS, "tie_km": TIE, "val": VAL, "cap": CAP, "weights": w, "base": round(float(np.exp(m.intercept_[0])), 2), "high": HIGH, "low": LOW, "n_fac_table": 9989, "public_share_table": 5.0, "n_labelled": int(len(S))}}
js = json.dumps(out, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
open(os.path.join(ROOT, "app", "data.json"), "w").write(js)
print("facilities:", len(fac), "| pins:", len(pins), "| cities:", len(cities), "| nfhs keys:", len(nfhs), "| pmndp districts:", len(pmndp), "| bytes:", len(js.encode()))
for q in ["Araria", "Patna", "Nashik", "Gadchiroli", "Jaipur"]: print(q, cities.get(q), places[cities[q][2]] if q in cities else None, nfhs.get(nz(places[cities[q][2]][0]) + "|" + nz(places[cities[q][2]][1]))[:5] if q in cities and nfhs.get(nz(places[cities[q][2]][0]) + "|" + nz(places[cities[q][2]][1])) else None, len(pmndp.get(nz(places[cities[q][2]][0]) + "|" + nz(places[cities[q][2]][1]), [])) if q in cities else None)
''')

### `pipeline/scrape_doctors.py`
Optional, not run here (Free Edition blocks most outbound web traffic): how doctors were read from hospitals' own websites.

In [ ]:
write('pipeline/scrape_doctors.py', r'''
"""Optional step: read doctors' names, degrees, specialty, years of experience and college from facilities' own websites.

The app cannot fetch other websites when a user clicks (browser security and Databricks Free Edition's outbound
limits), so we fetch once, offline, and store what we find. Scope: every facility in the app that lists a website,
highest-scoring first, so a run stopped early still covers the most relevant places.
Politeness: robots.txt is respected, a generic user agent, at most 5 pages per site (home page plus links whose text
mentions doctors or team), 6-second timeout. Directory sites (Practo, Justdial and similar) are NOT read: their terms
forbid automated collection.

Output: pipeline/external/doctors_web.json
  {facility unique_id: {"url": page, "doctors": [[name, degrees, specialty, years, college], ...], "fetched": date}}
Shown in the app as "read from the hospital's own website, not verified". Never used in a score.
"""
import sys, os, re, json, html, time, socket, threading, concurrent.futures as cf, urllib.request, urllib.parse, urllib.robotparser
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import ROOT

socket.setdefaulttimeout(6)   # robots.txt reads have no timeout of their own; without this one slow site stalls the run
UA = "Mozilla/5.0 (compatible; health-facility-research/0.2)"
OUTF = os.path.join(ROOT, "pipeline", "external", "doctors_web.json")
SKIP = re.compile(r"practo|justdial|lybrate|sulekha|facebook|instagram|google|wikipedia|indiamart|credihealth|medifee|quikr", re.I)
NAME = re.compile(r"\bDr\.?\s+((?:[A-Z][a-zA-Z.']+|[A-Z]\.)(?:\s+(?:[A-Z][a-zA-Z.']+|[A-Z]\.)){0,3})(?=[\s,|(\-–:]|$)")
QUAL = re.compile(r"\b(MBBS|M\.B\.B\.S\.?|MD|M\.D\.|MS|M\.S\.|DM|D\.M\.|M\.?\s?Ch\.?|DNB|FRCS\w*|MRCP\w*|FRCP\w*|DGO|DCH|D\.?\s?Ortho|DA|FNB|FICS|FACS|FACC|FESC|MRCOG|FRCOG|MDS|BDS|DOMS|DLO|DPM|DMRD|DMRT|PhD|FIAGES|FMAS)\b(\s*\([A-Za-z &.,-]{2,30}\))?")
SPEC = re.compile(r"(orthopa?edic\w*|joint replacement|spine|trauma|neuro ?surg\w*|neurolog\w*|nephrolog\w*|urolog\w*|gyn(a)?ecolog\w*|obstetric\w*|"
                  r"p(a)?ediatric\w*|neonatolog\w*|cardiac surg\w*|cardio ?thoracic|ctvs|cardiolog\w*|interventional cardiolog\w*|oncolog\w*|radiation oncolog\w*|"
                  r"radiolog\w*|ophthalmolog\w*|plastic surg\w*|burns?|pulmonolog\w*|chest physician|respiratory|critical care|intensivist|ana?esthe\w*|"
                  r"emergency medicine|general surg\w*|laparoscopic|internal medicine|general physician|physician|patholog\w*|dermatolog\w*|ent|psychiatr\w*|gastro\w*|endocrin\w*|dentist|dental)", re.I)
YEARS = re.compile(r"(\d{1,2})\+?\s*(?:years|yrs)\b", re.I)
COLL = re.compile(r"((?:[A-Z][A-Za-z.&']+\s){0,5}(?:Medical College|University|Institute of Medical Sciences|AIIMS|PGIMER|JIPMER|CMC|KEM|Maulana Azad|Grant Medical)[A-Za-z ,.'&]{0,30})")
LINK = re.compile(r'<a[^>]+href=["\']([^"\'#]+)["\'][^>]*>(.*?)</a>', re.I | re.S)


def get(url, rp):
    if rp is not None and not rp.can_fetch(UA, url): return ""
    try:
        req = urllib.request.Request(url, headers={"User-Agent": UA, "Accept": "text/html"})
        with urllib.request.urlopen(req, timeout=6) as r:
            if "html" not in (r.headers.get("Content-Type") or "html"): return ""
            return r.read(1_500_000).decode("utf-8", "ignore")
    except Exception:
        return ""


def text(h):
    h = re.sub(r"(?is)<(script|style|noscript)[^>]*>.*?</\1>", " ", h)
    h = re.sub(r"(?i)<br\s*/?>|</(p|div|li|h\d|td|span|a)>", " | ", h)
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", h)))


def doctors(t):
    out, seen = [], set()
    ms = list(NAME.finditer(t))
    for i, m in enumerate(ms):
        toks = re.sub(r"\s+", " ", m.group(1)).strip(" .").split(" ")
        while toks and QUAL.fullmatch(toks[-1].strip(",.")): toks.pop()      # "Dr. Rakesh Sharma MBBS" -> name ends before the degree
        name = " ".join(toks).strip(" .")
        if not name: continue
        if len(name.split()[-1].strip(".")) < 3 or len(name) < 6 or name.lower() in seen: continue
        end = ms[i + 1].start() if i + 1 < len(ms) else len(t)
        w = t[m.start() + 4 + len(name): min(end, m.end() + 320)]
        q = list(dict.fromkeys(re.sub(r"\s+", " ", "".join(x)).strip() for x in QUAL.findall(w)))[:5]
        s = list(dict.fromkeys(x.group(0).lower() for x in SPEC.finditer(w[:200])))[:2]
        if not q and not s: continue
        y = YEARS.search(w); c = COLL.search(w)
        seen.add(name.lower())
        out.append(["Dr. " + name, ", ".join(q)[:60], " / ".join(s)[:40], int(y.group(1)) if y and 1 <= int(y.group(1)) <= 60 else None, c.group(1).strip(" ,.")[:60] if c else ""])
        if len(out) >= 12: break
    return out


def site(uid, raw):
    base = raw if raw.startswith("http") else "https://" + raw.strip("/")
    if SKIP.search(base): return uid, None
    p = urllib.parse.urlparse(base); dom = p.netloc
    rp = urllib.robotparser.RobotFileParser(); rp.set_url(f"{p.scheme}://{dom}/robots.txt")
    try: rp.read()
    except Exception: rp = None
    home = get(base, rp) or (get(base.replace("https://", "http://"), rp) if base.startswith("https://") else "")
    if not home: return uid, None
    pages = [(base, home)]
    for href, lab in LINK.findall(home):
        if re.search(r"doctor|team|consultant|specialist|faculty|our[- ]?experts|physician|surgeon", href + " " + lab, re.I):
            u = urllib.parse.urljoin(base, href)
            if urllib.parse.urlparse(u).netloc.replace("www.", "") == dom.replace("www.", "") and u not in [x[0] for x in pages]:
                h = get(u, rp)
                if h: pages.append((u, h))
        if len(pages) >= 5: break
    docs, src, seen = [], base, set()
    for u, h in pages:
        for d in doctors(text(h)):
            if d[0].lower() in seen: continue
            seen.add(d[0].lower()); docs.append(d)
            if src == base and u != base: src = u
    return uid, {"url": src, "doctors": docs[:12]}


if __name__ == "__main__":
    from common import load_facilities
    D = json.load(open(os.path.join(ROOT, "app", "data.json")))
    F = load_facilities(); ids = dict(zip(F.name.str.slice(0, 70).str.strip(), F.unique_id))
    best = {}
    for s, v in D["claims"].items():
        for k, c in v["c"].items():
            if c[1] >= 1 and not c[8]: best[int(k)] = max(best.get(int(k), 0), c[0])
    jobs = {}
    for i in sorted(best, key=lambda i: -best[i]):
        f = D["fac"][i]; uid = ids.get(re.sub(r"\s+", " ", f[0]).strip())
        if f[9] and uid: jobs[uid] = f[9]
    out = json.load(open(OUTF)) if os.path.exists(OUTF) else {}
    todo = [kv for kv in jobs.items() if kv[0] not in out or (out[kv[0]].get("doctors") and len(out[kv[0]]["doctors"][0]) < 5)]
    print("sites to read:", len(todo), "of", len(jobs), flush=True)
    stop = time.time() + float(os.environ.get("MINUTES", "1e6")) * 60; n = 0
    ex = cf.ThreadPoolExecutor(64); futs = [ex.submit(site, *kv) for kv in todo]
    try:
        for fu in cf.as_completed(futs, timeout=max(1, stop - time.time())):
            uid, res = fu.result(); n += 1
            out[uid] = {**(res or {"url": "", "doctors": []}), "fetched": "2026-11-22"}
            if n % 250 == 0:
                json.dump(out, open(OUTF, "w"), ensure_ascii=False)
                print(n, "read |", sum(1 for v in out.values() if v["doctors"]), "sites with doctors", flush=True)
    except cf.TimeoutError:
        print("time limit reached after", n, "sites; the rest are left for a later run", flush=True)
    json.dump(out, open(OUTF, "w"), ensure_ascii=False)
    print("finished | sites", len(out), "| with doctors", sum(1 for v in out.values() if v["doctors"]), "| doctors", sum(len(v["doctors"]) for v in out.values()), flush=True)
    os._exit(0)   # do not wait for threads still reading slow sites
''')

### `app/build.py`
Assembles the single-page app from the template, the data and the icons.

In [ ]:
write('app/build.py', r'''
"""Assemble the single-page app from template.html, data.json and icons/. Run from the app folder."""
import re, json, glob, os
ic = {}
for f in glob.glob("icons/*.svg"):
    s = open(f).read(); s = re.sub(r'\s(width|height)="48"', "", s, count=2); ic[os.path.basename(f)[:-4]] = re.sub(r"\s+", " ", s).strip()
t = open("template.html").read(); d = open("data.json").read()
assert t.count("/*__DATA__*/null") == 1 and t.count("/*__ICONS__*/{}") == 1
frag = t.replace("/*__ICONS__*/{}", json.dumps(ic).replace("</", "<\\/")).replace("/*__DATA__*/null", d)
open("sahi-ilaj.html", "w").write(frag)
open("index.html", "w").write('<!doctype html>\n<html lang="en"><head><meta name="viewport" content="width=device-width, initial-scale=1"></head><body style="margin:0">\n' + frag + "\n</body></html>")
open("check.js", "w").write("\n".join(re.findall(r"<script>(.*?)</script>", frag, re.S)))
print("built", len(frag.encode()), "bytes")
''')

### `app/app.py`
The Databricks App: serves the page.

In [ ]:
write('app/app.py', r'''
"""Databricks App: serves the single-page finder. All scoring is precomputed by the pipeline; the page only filters and ranks."""
import os
from flask import Flask, send_from_directory
HERE = os.path.dirname(os.path.abspath(__file__))
app = Flask(__name__)

@app.route("/")
def index():
    return send_from_directory(HERE, "index.html")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("DATABRICKS_APP_PORT", os.environ.get("PORT", 8000))))
''')

### `app/app.yaml`
Databricks App start command.

In [ ]:
write('app/app.yaml', r'''
command: ["python", "app.py"]
''')

### `app/requirements.txt`
Databricks App dependencies.

In [ ]:
write('app/requirements.txt', r'''
flask
''')

### `app/template.html`
The app page (HTML, CSS, JavaScript): ranking, cards, explanations.

In [ ]:
write('app/template.html', r'''
<meta charset="utf-8">
<title>CareSaathi</title>
<meta name="claude:viewport-fit-sides" content="cover">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Nunito:wght@500;700;800;900&family=Noto+Sans+Devanagari:wght@500;700&family=IBM+Plex+Mono:wght@500&display=swap">
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<style>
/* Layout: three short picture questions, then a delivery-app style answer: a real map on the left with you and five numbered places, a ranked list on the right; each card opens to WHY and HOW. */
:root{
  --bg:#FFFFFF; --surface:#FFFFFF; --ink:#14213D; --muted:#667085; --line:#E4E8EE; --soft:#F3F6FA;
  --accent:#1769E0; --accentink:#0E4FB3; --accentbg:#E7F0FD; --ok:#1E8E3E; --okbg:#E1F3E4; --warn:#9A6B00; --warnbg:#FFF2CC; --bad:#D2453D; --badbg:#FDE6E3;
  --font:"Nunito","Noto Sans Devanagari",system-ui,sans-serif; --mono:"IBM Plex Mono",ui-monospace,Menlo,monospace;
  color-scheme:light;
}
*{box-sizing:border-box}
body{background:var(--bg);color:var(--ink);font-family:var(--font);font-size:16px;line-height:1.5}
.wrap{max-width:1180px;margin:0 auto;padding-block:14px 40px;padding-left:max(18px,env(safe-area-inset-left,0px));padding-right:max(18px,env(safe-area-inset-right,0px));display:flex;flex-direction:column;gap:18px}
p{margin:0} button,input{font:inherit;color:inherit} a{color:var(--accentink)}
:focus-visible{outline:3px solid var(--accent);outline-offset:3px}
.bar{display:flex;align-items:center;gap:12px;min-height:44px}
.back{width:42px;height:42px;border-radius:50%;border:0;background:var(--surface);font-size:20px;cursor:pointer;flex:none}
.brand{font-weight:900;font-size:21px;letter-spacing:-.01em;flex:none} .brand span{color:var(--accent)}
.dots{display:flex;gap:6px;flex:1;max-width:420px;margin:0 auto}
.dots i{height:6px;flex:1;border-radius:3px;background:var(--line)} .dots i.on{background:var(--accent)}
.lang{border:0;background:var(--surface);border-radius:999px;padding:9px 16px;font-weight:800;font-size:14px;cursor:pointer;flex:none}
h1{font-size:34px;line-height:1.2;font-weight:900;margin:8px 0 0;text-wrap:balance;letter-spacing:-.01em}
.hint{color:var(--muted);font-size:15px;font-weight:600}
.screen{display:flex;flex-direction:column;gap:18px}
.narrow{max-width:600px;width:100%;margin:0 auto}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(160px,1fr));gap:14px}
.tile{background:var(--surface);border:0;border-radius:26px;padding:22px 12px 18px;cursor:pointer;display:flex;flex-direction:column;align-items:center;gap:12px;min-width:0;text-align:center;transition:transform .12s;box-shadow:0 1px 0 var(--line)}
.tile:hover,.opt:hover{box-shadow:0 0 0 2px var(--accent) inset} .tile:active,.opt:active{transform:scale(.97)}
.bub{width:84px;height:84px;border-radius:28px;display:flex;align-items:center;justify-content:center;background:var(--accentbg);color:var(--accentink)}
.bub svg{width:54px;height:54px;display:block}
.h0{background:#FFE7CF;color:#B9560A} .h1{background:#FDE2E0;color:#B5352E} .h2{background:#E1F3E4;color:#1B7A36} .h3{background:#E3ECFB;color:#2B55A8} .h4{background:#F1E6FA;color:#6B3FA0}
.tile b{font-weight:800;font-size:16.5px;overflow-wrap:anywhere}
.big{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px}
.opt{background:var(--surface);border:0;border-radius:26px;padding:22px 20px;cursor:pointer;display:flex;align-items:center;gap:16px;text-align:left;transition:transform .12s;box-shadow:0 1px 0 var(--line)}
.opt .bub{flex:none;width:72px;height:72px} .opt.red .bub{background:var(--badbg);color:var(--bad)}
.opt b{font-size:19px;font-weight:900;display:block} .opt span{color:var(--muted);font-size:14.5px;font-weight:600}
.chips{display:flex;gap:8px;flex-wrap:wrap}
.chip{background:var(--surface);border:0;border-radius:999px;padding:8px 15px;font-size:14.5px;font-weight:800;cursor:pointer}
.chip[aria-pressed="true"]{background:var(--accent);color:#fff}
.field{background:var(--surface);border:2px solid transparent;border-radius:20px;padding:17px 20px;font-size:21px;font-weight:800;width:100%;min-width:0}
.field:focus{border-color:var(--accent);outline:none}
.cta{background:var(--accent);color:#fff;border:0;border-radius:20px;padding:17px;font-size:19px;font-weight:900;cursor:pointer;width:100%}
.ghost{background:var(--surface);border:0;border-radius:20px;padding:15px;font-size:16px;font-weight:800;cursor:pointer;width:100%}
.err{color:var(--bad);font-weight:700;font-size:15px}
.res{display:grid;grid-template-columns:minmax(0,1.05fr) minmax(0,1fr);gap:20px;align-items:start}
.mapcol{position:sticky;top:12px;display:flex;flex-direction:column;gap:8px;min-width:0}
#map{width:100%;height:min(72vh,640px);min-height:360px;border-radius:26px;overflow:hidden;background:#E4E8EE;z-index:0}
.listcol{display:flex;flex-direction:column;gap:12px;min-width:0}
.rule{background:var(--ink);color:#fff;border-radius:20px;padding:14px 18px;font-weight:700;font-size:15px}
.rule b{color:#9CC5FF}
.banner{border-radius:20px;padding:13px 18px;font-weight:800;font-size:15px}
.banner.red{background:var(--badbg);color:var(--bad)} .banner.amber{background:var(--warnbg);color:var(--warn)}
.card{background:var(--surface);border-radius:24px;box-shadow:0 1px 0 var(--line);overflow:hidden}
.card.sel{box-shadow:0 0 0 2px var(--accent)}
.chead{display:flex;gap:14px;align-items:center;padding:16px 18px;cursor:pointer;border:0;background:transparent;width:100%;text-align:left}
.rk{width:40px;height:40px;border-radius:50%;background:var(--accent);color:#fff;font-weight:900;font-size:18px;display:flex;align-items:center;justify-content:center;flex:none}
.rk.x{background:var(--badbg);color:var(--bad)}
.cname{font-weight:900;font-size:17.5px;line-height:1.25;overflow-wrap:anywhere}
.cmeta{color:var(--muted);font-size:13.5px;font-weight:700;margin-top:2px}
.kmb{margin-left:auto;text-align:right;flex:none} .kmb .n{font-size:24px;font-weight:900;line-height:1;font-variant-numeric:tabular-nums} .kmb .u{font-size:12px;font-weight:800;color:var(--muted)}
.pills{display:flex;gap:6px;flex-wrap:wrap;margin-top:6px}
.pill{font-size:12.5px;font-weight:800;border-radius:999px;padding:3px 10px}
.b2{color:var(--ok);background:var(--okbg)} .b1{color:var(--warn);background:var(--warnbg)} .b0{color:var(--bad);background:var(--badbg)} .pn{color:var(--muted);background:var(--soft)}
.cbody{padding:0 18px 18px;display:flex;flex-direction:column;gap:14px;font-size:14.5px}
.q3{display:grid;gap:10px}
.qa{background:var(--soft);border-radius:16px;padding:12px 14px}
.qa h4{margin:0 0 4px;font-size:12.5px;font-weight:900;letter-spacing:.04em;text-transform:uppercase;color:var(--accentink)}
.quote{background:var(--surface);border-radius:12px;padding:9px 12px;font-weight:700;margin:6px 0;overflow-wrap:anywhere}
.check{list-style:none;margin:6px 0 0;padding:0;display:flex;flex-wrap:wrap;gap:6px}
.check li{border-radius:999px;padding:3px 10px;font-size:13px;font-weight:800} .check .y{background:var(--okbg);color:var(--ok)} .check .n{background:var(--surface);color:var(--muted)}
.eq{font-family:var(--mono);font-size:13px;background:var(--ink);color:#fff;border-radius:12px;padding:10px 12px;overflow-x:auto;white-space:nowrap}
.eq b{color:#9CC5FF;font-weight:500}
.btns{display:flex;gap:10px;flex-wrap:wrap}
.call{background:var(--ok);color:#fff;border-radius:16px;padding:12px 20px;font-weight:900;font-size:16px;text-decoration:none;display:inline-flex;align-items:center;gap:8px}
.call svg{width:20px;height:20px}
.sec{background:var(--soft);border:0;border-radius:16px;padding:12px 18px;font-weight:900;font-size:15px;cursor:pointer;text-decoration:none;color:var(--ink);display:inline-block}
.phone{font-family:var(--mono);font-size:14px;color:var(--muted);user-select:all;align-self:center}
.warnl{background:var(--badbg);color:var(--bad);border-radius:12px;padding:8px 12px;font-weight:800;margin-top:6px}
ul.ask{margin:4px 0 0;padding-left:18px} ul.ask li{margin:2px 0}
details{background:var(--surface);border-radius:22px;padding:2px 18px;box-shadow:0 1px 0 var(--line)}
details>summary{cursor:pointer;font-weight:900;padding:14px 0;list-style:none;display:flex;justify-content:space-between;gap:10px}
details>summary::-webkit-details-marker{display:none}
details>summary::after{content:"＋";color:var(--muted)} details[open]>summary::after{content:"－"}
.inner{display:flex;flex-direction:column;gap:10px;padding:0 0 16px;font-size:14.5px}
.flow{display:grid;gap:8px}
.step{display:flex;gap:12px;align-items:flex-start;background:var(--soft);border-radius:16px;padding:12px 14px}
.step i{font-style:normal;width:28px;height:28px;border-radius:50%;background:var(--ink);color:#fff;font-weight:900;display:flex;align-items:center;justify-content:center;flex:none;font-size:14px}
.small{font-size:12.5px;color:var(--muted);font-weight:700}
.lab{font-size:12.5px;font-weight:900;color:var(--muted);letter-spacing:.04em;text-transform:uppercase}
@media (max-width:860px){.res{grid-template-columns:minmax(0,1fr)}.mapcol{position:static}#map{height:46vh}.big{grid-template-columns:minmax(0,1fr)}.brand{display:none}h1{font-size:28px}}
.ladder{display:flex;flex-direction:column;gap:6px;margin-top:6px}
.rung{display:grid;grid-template-columns:28px minmax(0,1fr);gap:10px;padding:10px 12px;border-radius:14px}
.rung.ok{background:var(--okbg)} .rung.no{background:var(--badbg)}
.rn{width:26px;height:26px;border-radius:50%;display:grid;place-items:center;font-weight:900;font-size:14px;color:#fff;background:var(--ok)} .rung.no .rn{background:var(--bad)}
.rt{font-size:12px;font-weight:900;letter-spacing:.04em;text-transform:uppercase;color:#14602B} .rung.no .rt{color:#9C2F29}
.rq{font-weight:700;font-size:14px;overflow-wrap:anywhere} .rs{font-size:12.5px;color:var(--muted);font-weight:700} .rs.bad{color:#9C2F29}
.needs{display:flex;flex-wrap:wrap;gap:5px;margin-top:5px}
.need{display:inline-block;font-size:12.5px;font-weight:800;border-radius:999px;padding:3px 10px;background:#fff} .need.ok{color:#14602B;border:1.5px solid #9BD3A7} .need.no{color:#9C2F29;border:1.5px solid #F0B1AB}
.mustbox{background:var(--badbg);color:#7A231E;border:2px solid var(--bad);border-radius:14px;padding:10px 13px;font-weight:700;font-size:14px}
.tts{float:right;margin:-2px 0 4px 8px;width:30px;height:30px;border-radius:50%;border:0;background:#fff;box-shadow:0 1px 3px rgba(31,42,68,.18);font-size:14px;cursor:pointer;line-height:1} .tts:active{transform:scale(.94)}
.whyline{font-weight:800;font-size:15px}
.two{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px}
.lab.g{color:var(--ok)} .lab.r{color:var(--bad)}
ul.hl{list-style:none;margin:6px 0 0;padding:0;display:flex;flex-direction:column;gap:5px}
ul.hl li{border-radius:12px;padding:7px 11px;font-weight:700;font-size:13.5px;overflow-wrap:anywhere}
ul.hl.ok li{background:var(--okbg);color:#14602B} ul.hl.no li{background:var(--badbg);color:#9C2F29} ul.hl li.none{background:var(--soft);color:var(--muted)}
.quote.q2{background:var(--soft);font-size:16px}
details.sub{background:var(--soft);border-radius:16px;padding:0 14px;box-shadow:none}
details.sub>summary{padding:11px 0;font-size:14.5px}
table.math{border-collapse:collapse;width:100%;font-size:13.5px}
table.math td{padding:5px 8px 5px 0;border-bottom:1px solid var(--line);vertical-align:top}
table.math td.num{font-family:var(--mono);white-space:nowrap;font-weight:500}
table.math tr.tot td{border-bottom:0;font-weight:800}
.other{background:var(--surface);border-radius:26px;padding:18px;display:flex;flex-direction:column;gap:10px;box-shadow:0 1px 0 var(--line)}
.other .rowi{display:flex;gap:10px;flex-wrap:wrap}
.other input{flex:1 1 240px;min-width:0;background:var(--soft);border:2px solid transparent;border-radius:16px;padding:13px 16px;font-size:17px;font-weight:700}
.other input:focus{border-color:var(--accent);outline:none}
.other button.gobtn{background:var(--ink);color:#fff;border:0;border-radius:16px;padding:13px 22px;font-weight:900;cursor:pointer}
@media (max-width:860px){.two{grid-template-columns:minmax(0,1fr)}}
.cost{background:var(--warnbg);color:#6B4A00;border-radius:14px;padding:10px 13px;font-weight:700;font-size:14px}
/* map library styles (inlined) */
.leaflet-pane,.leaflet-tile,.leaflet-marker-icon,.leaflet-marker-shadow,.leaflet-tile-container,.leaflet-pane>svg,.leaflet-pane>canvas,.leaflet-zoom-box,.leaflet-image-layer,.leaflet-layer{position:absolute;left:0;top:0}
.leaflet-container{overflow:hidden;-webkit-tap-highlight-color:transparent;outline:0;font:12px var(--font)}
.leaflet-tile,.leaflet-marker-icon,.leaflet-marker-shadow{user-select:none;-webkit-user-drag:none}
.leaflet-tile{visibility:hidden}.leaflet-tile-loaded{visibility:inherit}
.leaflet-container .leaflet-tile,.leaflet-container img.leaflet-tile{max-width:none!important;max-height:none!important;width:256px;height:256px;padding:0}
.leaflet-pane{z-index:400}.leaflet-tile-pane{z-index:200}.leaflet-overlay-pane{z-index:400}.leaflet-shadow-pane{z-index:500}.leaflet-marker-pane{z-index:600}.leaflet-tooltip-pane{z-index:650}.leaflet-popup-pane{z-index:700}
.leaflet-map-pane canvas{z-index:100}.leaflet-map-pane svg{z-index:200}
.leaflet-control{position:relative;z-index:800;pointer-events:auto;float:left;clear:both}
.leaflet-top,.leaflet-bottom{position:absolute;z-index:1000;pointer-events:none}.leaflet-top{top:0}.leaflet-right{right:0}.leaflet-bottom{bottom:0}.leaflet-left{left:0}
.leaflet-right .leaflet-control{float:right;margin-right:12px}.leaflet-top .leaflet-control{margin-top:12px}.leaflet-left .leaflet-control{margin-left:12px}.leaflet-bottom .leaflet-control{margin-bottom:6px}
.leaflet-zoom-animated{transform-origin:0 0}.leaflet-zoom-anim .leaflet-zoom-animated{transition:transform .25s cubic-bezier(0,0,.25,1)}
.leaflet-interactive{cursor:pointer}.leaflet-grab{cursor:grab}.leaflet-dragging .leaflet-grab{cursor:grabbing}
.leaflet-bar{border-radius:14px;overflow:hidden;box-shadow:0 1px 4px rgba(0,0,0,.2)}
.leaflet-bar a{display:block;width:36px;height:36px;line-height:36px;text-align:center;text-decoration:none;color:var(--ink);background:#fff;font-weight:900;font-size:18px;border-bottom:1px solid var(--line)}
.leaflet-control-attribution{background:rgba(255,255,255,.8);padding:0 6px;font-size:10px;border-radius:8px 0 0 0}
.leaflet-div-icon{background:none;border:none}
.leaflet-tooltip{position:absolute;padding:5px 10px;background:#fff;border-radius:10px;white-space:nowrap;pointer-events:none;box-shadow:0 1px 4px rgba(0,0,0,.25);font-weight:800;color:var(--ink)}
.pin{width:36px;height:36px;border-radius:50% 50% 50% 4px;transform:rotate(-45deg);background:var(--accent);border:3px solid #fff;box-shadow:0 2px 6px rgba(0,0,0,.35);display:flex;align-items:center;justify-content:center}
.pin span{transform:rotate(45deg);color:#fff;font-weight:900;font-size:15px}
.me{width:22px;height:22px;border-radius:50%;background:#2B6BE4;border:4px solid #fff;box-shadow:0 0 0 6px rgba(43,107,228,.25),0 2px 6px rgba(0,0,0,.3)}

.card,.tile,.other,details,.cost{border:1px solid var(--line)} .tile{box-shadow:0 1px 2px rgba(20,33,61,.05)} .card.sel{border-color:var(--accent)}
.help3{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px} @media(max-width:720px){.help3{grid-template-columns:1fr}}
.help3>div{border:1px solid var(--line);border-radius:16px;padding:12px 14px;font-size:13.5px} .help3 b{display:block;font-size:14px;margin-bottom:2px} .help3 a{color:var(--accentink);font-weight:800;text-decoration:none}
.pill.free{background:var(--okbg);color:#14602B}
#asha{border:1px solid var(--line);border-radius:16px;padding:14px} #asha table{width:100%;border-collapse:collapse;font-size:13.5px} #asha td,#asha th{text-align:left;padding:6px 8px;border-bottom:1px solid var(--line);vertical-align:top}
@media print{body *{visibility:hidden} #asha,#asha *{visibility:visible} #asha{position:absolute;left:0;top:0;width:100%;border:0} #asha button{display:none}}

.cmpbox{display:flex;flex-direction:column;gap:6px}
.cmp{display:grid;grid-template-columns:34px minmax(0,1fr) auto;gap:12px;align-items:center;background:var(--soft);border-radius:12px;padding:10px 14px}
.ci{width:34px;height:34px;color:var(--accent)} .ci svg{width:34px;height:34px}
.cl b{display:block;font-size:14px} .cl i{display:block;font-size:12px;color:var(--muted);font-style:italic}
.cv{text-align:right} .cv .big{font-size:20px;font-weight:900} .cv .na{font-size:14px;color:var(--muted);font-weight:800}
.nft{width:100%;border-collapse:collapse;font-size:13.5px;margin-top:6px} .nft th{text-align:right;color:var(--muted);font-size:12px;padding:4px 8px} .nft td{padding:7px 8px;border-top:1px solid var(--line);text-align:right} .nft td:first-child{text-align:left} .nft i{display:block;color:var(--muted);font-size:11.5px}
.docs{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px} @media(max-width:640px){.docs{grid-template-columns:1fr}}
.doc{display:flex;gap:10px;align-items:center;background:#fff;border:1px solid var(--line);border-radius:12px;padding:8px 10px} .av{flex:none;width:36px;height:36px;border-radius:50%;background:var(--accentbg);color:var(--accentink);display:grid;place-items:center;font-weight:900;font-size:13px}
.locbox{position:relative} .sugg{position:absolute;left:0;right:0;top:calc(100% + 4px);background:#fff;border:1px solid var(--line);border-radius:14px;box-shadow:0 8px 24px rgba(20,33,61,.12);z-index:5;overflow:hidden}
.srow{display:flex;gap:12px;align-items:center;width:100%;text-align:left;background:#fff;border:0;border-top:1px solid var(--line);padding:10px 14px;font:inherit;color:var(--ink);cursor:pointer} .srow:first-child{border-top:0} .srow:hover{background:var(--soft)} .srow .small{display:block}
.gpsrow{color:var(--accentink)} .si{flex:none;width:30px;height:30px;border-radius:50%;background:var(--soft);display:grid;place-items:center;color:var(--muted);font-size:18px} .si svg{width:20px;height:20px;color:var(--accent)}
.cities{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px}
.cityb{display:flex;flex-direction:column;align-items:center;gap:6px;background:#fff;border:1px solid var(--line);border-radius:16px;padding:12px 6px;font:inherit;font-weight:800;color:var(--ink);cursor:pointer} .cityb:hover{border-color:var(--accent)}
.cityb span{width:64px;height:64px;border-radius:50%;background:var(--accentbg);color:var(--accent);display:grid;place-items:center} .cityb svg{width:48px;height:48px}

.who{display:flex;flex-direction:column;gap:8px;border:1px solid var(--line);border-radius:16px;padding:12px 14px}
.drow{display:flex;gap:12px;align-items:flex-start;padding:6px 0} .drow+.drow{border-top:1px solid var(--line);padding-top:10px}
.dav{flex:none;width:46px;height:46px;border-radius:50%;display:grid;place-items:center;font-weight:900;color:var(--accentink);font-size:15px}
.dtx b{color:var(--accentink);font-size:15px} .dsp{font-weight:700;font-size:13.5px} .whyno{margin:0;font-size:14px}
.dpill{background:var(--accentbg);color:var(--accentink)} .dirb{text-decoration:none;display:inline-flex;align-items:center}
.pick2{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px} .sel{appearance:auto;font:inherit;font-weight:700;background:#fff;cursor:pointer} .sel:disabled{opacity:.5;cursor:default}
</style>

<div class="wrap">
  <div class="bar"><button class="back" id="back" aria-label="Back">←</button><div class="brand">Care<span>Saathi</span></div><div class="dots" id="dots"></div><button class="lang" id="lang">हिन्दी</button></div>
  <div id="view"></div>
</div>

<script>
const D=/*__DATA__*/null;
const ICON=/*__ICONS__*/{};
const V=D.meta.val, FAC=D.fac, PL=D.places, CL=D.claims, M=D.meta, W=M.weights;
const NEEDS=[
 ['kidney','Kidney / dialysis','किडनी / डायलिसिस',0,[['dialysis','Dialysis','डायलिसिस']]],
 ['birth','Childbirth','प्रसव',1,[['emergency_obstetric','Delivery / C-section','प्रसव / सिज़ेरियन']]],
 ['newborn','Sick newborn','बीमार नवजात',1,[['nicu','Newborn ICU','नवजात आईसीयू']]],
 ['heart','Heart','दिल',0,[['cardiac_surgery','Heart surgery','हृदय की सर्जरी'],['cath_lab','Angiography / stent','एंजियोग्राफी / स्टेंट']]],
 ['injury','Accident / injury','दुर्घटना / चोट',1,[['trauma_emergency','Emergency / trauma','आपातकाल / ट्रॉमा'],['ortho_surgery','Bone and joint surgery','हड्डी की सर्जरी']]],
 ['burn','Burn','जलना',1,[['burn_care','Burn care','जलने का इलाज']]],
 ['breath','Critical / breathing','गंभीर / साँस',1,[['icu','ICU','आईसीयू'],['ventilator','Ventilator','वेंटिलेटर']]],
 ['stroke','Stroke','लकवा / स्ट्रोक',1,[['stroke','Stroke care','स्ट्रोक का इलाज']]],
 ['bite','Snakebite / poison','साँप का काटना / ज़हर',1,[['snakebite_poison','Snakebite / poison care','साँप / ज़हर का इलाज']]],
 ['scan','Scan','स्कैन',0,[['mri','MRI','एमआरआई'],['ct_scan','CT scan','सीटी स्कैन']]],
 ['cancer','Cancer','कैंसर',0,[['chemotherapy','Chemotherapy','कीमोथेरेपी'],['radiotherapy','Radiotherapy','रेडियोथेरेपी']]],
 ['eye','Eye','आँख',0,[['cataract_surgery','Cataract surgery','मोतियाबिंद का ऑपरेशन']]],
 ['tb','TB / chest','टीबी / छाती',0,[['tb','TB care','टीबी का इलाज']]],
 ['blood','Blood','खून',1,[['blood_bank','Blood bank','ब्लड बैंक']]]];
const TX={en:{q1:'What do you need help with?',h1:'Tap a picture.',q2:'Is it an emergency?',u1:'Yes, right now',u1s:'Nearest places that pass our evidence check',u0:'No, I can plan',u0s:'Best-supported places in your area',which:'Which one?',q3:'Where are you?',h3:'Type a PIN code or a town, or share your location.',ph:'PIN or town',go:'Find care',gps:'Use my location',nf:'We could not find that place. Try a 6-digit PIN or a district name.',km:'km',call:'Call',area:'About this area',notrec:'Nearby places we left out, and why',how:'How we decide',b2:'Strong evidence',b1:'Some evidence',b0:'Weak evidence',pub:'Public',priv:'Private',again:'Start again'},
 hi:{q1:'आपको किस चीज़ में मदद चाहिए?',h1:'तस्वीर दबाएँ।',q2:'क्या यह आपातकाल है?',u1:'हाँ, अभी',u1s:'सबसे पास की भरोसेमंद जगहें',u0:'नहीं, समय है',u0s:'आपके इलाके की सबसे भरोसेमंद जगहें',which:'कौन सा?',q3:'आप कहाँ हैं?',h3:'पिन कोड या शहर लिखें, या अपनी जगह बताएँ।',ph:'पिन या शहर',go:'खोजें',gps:'मेरी जगह इस्तेमाल करें',nf:'यह जगह नहीं मिली। 6 अंकों का पिन या ज़िले का नाम लिखें।',km:'किमी',call:'कॉल करें',area:'इस इलाके के बारे में',notrec:'पास की जगहें जिन्हें हमने नहीं चुना',how:'हम कैसे तय करते हैं',b2:'मज़बूत प्रमाण',b1:'कुछ प्रमाण',b0:'कमज़ोर प्रमाण',pub:'सरकारी',priv:'निजी',again:'फिर से शुरू करें'}};
const SIGN=['Map point, postcode and city all point to different places','The text names two or more towns over 60 km away','The row was built from the websites of two or more other organisations'];
const SIGNW=['all_differ','far2','other_sites_2plus'];
const NEGNAME={eye:'eye-care',dental:'dental',skin_hair:'skin and hair',physio:'physiotherapy',alt_med:'ayurveda / homeopathy'};
const KW=[['kidney',/kidney|dialysis|urine|can.?t pee|किडनी|गुर्द/i],['birth',/pregnan|labou?r|deliver|birth|contraction|water broke|गर्भ|प्रसव/i],['newborn',/newborn|new born|baby|infant|नवजात|शिशु/i],['heart',/chest|heart|palpitat|सीने|छाती|दिल/i],['injury',/accident|injur|fractur|bleed|fell|fall|broken|wound|crash|hit by|cut|दुर्घटना|चोट/i],['burn',/burn|fire|scald|जल/i],['breath',/breath|unconscious|faint|not responding|not waking|choking|साँस|सांस|बेहोश/i],['stroke',/stroke|paraly|face droop|slurred|can.?t speak|cannot speak|one side|numb|लकवा/i],['bite',/snake|bite|poison|साँप|ज़हर|जहर/i],['scan',/scan|mri|\bct\b|x-?ray/i],['cancer',/cancer|tumou?r|lump|कैंसर/i],['eye',/eye|vision|cataract|can.?t see|cannot see|not able to see|blind|blurr|sight|दिख|आँख|आंख/i],['tb',/cough|\btb\b|tubercul|sputum|खाँसी|खांसी|टीबी/i],['blood',/blood|transfus|खून/i]];
let sugg=null, typed='', sector='any', rad=0;
const GL={'RO water':'Water purifier for dialysis (RO plant)','sessions or machine count':'Number of dialysis machines or sessions','nephrologist':'Kidney doctor (nephrologist)','ICU':'Intensive care unit (ICU)','operation theatre':'Operating theatre','anaesthesia':'Anaesthesia doctor','blood bank':'Blood bank or blood storage','round the clock':'Open 24 hours','round the clock emergency':'24-hour emergency','newborn care':'Newborn care unit','obstetrician':'Delivery doctor (obstetrician)','ventilator':'Breathing machine (ventilator)','warmer or incubator':'Baby warmer or incubator','neonatologist':'Newborn doctor (neonatologist)','phototherapy':'Light therapy for newborn jaundice','CPAP':'Breathing support for babies (CPAP)','paediatrician':'Child doctor (paediatrician)','cardiac OT':'Operating theatre','cath lab':'Heart catheter lab','cath lab or DSA':'Catheter lab','heart lung machine':'Heart-lung machine','cardiac surgeon':'Heart surgeon','cardiac ICU':'Heart ICU','cardiologist':'Heart doctor (cardiologist)','cardiac surgery backup':'Heart surgery backup','plastic surgeon':'Plastic surgeon','skin graft or bank':'Skin graft or skin bank','tesla spec':'MRI strength stated (Tesla)','radiologist':'Scan doctor (radiologist)','slice spec':'CT slice count stated','CT':'CT scanner','MRI':'MRI scanner','CT or MRI':'CT or MRI scanner','ambulance':'Ambulance','orthopaedic surgeon':'Bone surgeon','neurologist':'Brain doctor (neurologist)','xray':'X-ray','lab':'Laboratory','oxygen':'Oxygen supply','pharmacy':'Pharmacy','surgeon ortho or neuro':'Bone or brain surgeon','neurosurgery':'Brain surgery','defibrillator':'Heart shock machine (defibrillator)','monitors':'Patient monitors','bed count':'Number of beds stated','ventilator count':'Number of ventilators stated','intensivist or anaesthetist':'ICU doctor','C arm':'X-ray arm for surgery (C-arm)','physiotherapy':'Physiotherapy','implants volume':'Number of implant surgeries stated','pulmonologist':'Lung doctor','sputum or naat':'TB test (sputum or NAAT)','DOTS':'Free TB medicine programme (DOTS)','physician':'General doctor','medical oncologist':'Cancer doctor','day care':'Day-care ward','pathologist':'Lab doctor (pathologist)','tumour board':'Cancer team review','linac or cobalt':'Radiation machine','radiation oncologist':'Radiation doctor','physicist':'Radiation physicist','planning':'Radiation planning system','AERB':'Radiation safety licence','ophthalmologist':'Eye doctor','phaco machine':'Cataract machine (phaco)','operating microscope':'Operating microscope','biometry':'Eye measurement (biometry)','YAG laser':'Eye laser (YAG)','volume':'Number of cases stated','licence':'Blood bank licence','component separation':'Blood component separation','dialysis backup':'Dialysis available','ABG':'Blood gas test','echo TMT':'Heart scan (echo / TMT)','chemotherapy':'Chemotherapy','radiotherapy':'Radiotherapy'};
const gl=n=>GL[n]||n;
const SVCNAME={};NEEDS.forEach(n=>n[4].forEach(s=>SVCNAME[s[0]]=s[1]));
let lang='en', step=1, need=null, svc=null, urgent=false, here=null, last=null, locErr='', open=0, map=null;
const $=id=>document.getElementById(id);
const esc=s=>String(s==null?'':s).replace(/[&<>"']/g,m=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[m]));
const nz=s=>String(s).toUpperCase().replace(/&/g,'AND').replace(/[^A-Z]/g,'');
const T=k=>TX[lang][k]||TX.en[k];
const tr=(en,hi)=>lang==='hi'?hi:en;
const curNeed=()=>NEEDS.find(n=>n[0]===need);
const curSvc=()=>curNeed()[4].find(s=>s[0]===svc);
const hav=(a,b,c,d)=>{const R=Math.PI/180,x=Math.sin((c-a)*R/2)**2+Math.cos(a*R)*Math.cos(c*R)*Math.sin((d-b)*R/2)**2;return 12742*Math.asin(Math.sqrt(x));};
const CITY={};for(const k in D.cities)CITY[k.toLowerCase()]=[k,...D.cities[k]];

function resolve(q){q=q.trim();if(!q)return null;
  let m=q.replace(/\s/g,'').match(/^(\d{6})$/);if(m){const p=D.pins[m[1]];return p?{lat:p[0],lon:p[1],pl:p[2],label:'PIN '+m[1]}:null;}
  m=q.match(/^(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)$/);if(m)return nearPlace(+m[1],+m[2],q);
  const c=CITY[q.toLowerCase()];if(c)return{lat:c[1],lon:c[2],pl:c[3],label:c[0]};
  const k=Object.keys(CITY).find(x=>x.startsWith(q.toLowerCase()));if(k){const c2=CITY[k];return{lat:c2[1],lon:c2[2],pl:c2[3],label:c2[0]};}
  return null;}
function nearPlace(lat,lon,label){let best=null,bd=1e9;for(const k in D.pins){const p=D.pins[k],d=(p[0]-lat)**2+(p[1]-lon)**2;if(d<bd){bd=d;best=p;}}return{lat,lon,pl:best?best[2]:null,label:label||'My location'};}

/* Ranking.  Emergency: nearest first among places that pass the evidence check.  Planned: highest trust first inside the local area. */
function search(){
  const cl=CL[svc].c, all=[];
  for(const k in cl){const f=FAC[k],c=cl[k];all.push({i:+k,f,c,d:hav(here.lat,here.lon,f[3],f[4])});}
  const ok=all.filter(x=>!x.c[8]&&x.c[1]>=1&&(sector==='any'||x.f[7]===sector)), radii=urgent?[10,25,50,75,100,150,200]:[25,50,75,100,150,200,300];
  let R=radii[radii.length-1];if(rad)R=rad;else for(const r of radii){if(ok.filter(x=>x.d<=r).length>=5){R=r;break;}}
  const pool=ok.filter(x=>x.d<=R);
  const key=x=>x.f[0].toLowerCase().replace(/[^a-z]/g,'').slice(0,12), seen=[];
  let srt=urgent?pool.sort((a,b)=>a.d-b.d):pool.sort((a,b)=>b.c[0]-a.c[0]||(b.c[12]||0)-(a.c[12]||0)||a.d-b.d);
  const sure=x=>x.f[5]!=null&&x.f[5]>=0.6?1:0;
  if(urgent){srt=srt.filter(sure).concat(srt.filter(x=>!sure(x)));const out=[];let i=0;while(i<srt.length){let j=i;while(j<srt.length&&sure(srt[j])===sure(srt[i])&&srt[j].d-srt[i].d<=M.tie_km)j++;out.push(...srt.slice(i,j).sort((a,b)=>b.c[1]-a.c[1]||(b.f[7]==='public')-(a.f[7]==='public')||a.d-b.d));i=j;}srt=out;}
  else{const out=[];let i=0;while(i<srt.length){let j=i;while(j<srt.length&&srt[i].c[0]-srt[j].c[0]<=M.tie_pts)j++;out.push(...srt.slice(i,j).sort((a,b)=>a.d-b.d));i=j;}srt=out;}
  srt=srt.filter(x=>{const k=key(x);if(seen.some(s=>s.k===k&&hav(s.x.f[3],s.x.f[4],x.f[3],x.f[4])<2))return false;seen.push({k,x});return true;});
  const top=srt.slice(0,5);
  const off=all.filter(x=>x.d<=Math.max(R,top.length?top[top.length-1].d:R)&&(x.c[8]||x.c[1]===0)).sort((a,b)=>a.d-b.d).slice(0,4);
  const nextOut=ok.filter(x=>x.d>R).sort((a,b)=>a.d-b.d)[0];
  return{top,off,R,n:pool.length,all,nextOut};}

function parts(c){const co=CL[svc].coreq.length,share=co?c[3].length/co:0;
  const e1=c[2]===3?W.t3:(c[2]===2?W.t2:1), e2=Math.pow(W.coreq,share), e3=c[7]?W.ownurl:1;let C=1;const fired=[];
  for(let b=0;b<3;b++)if(c[6]>>b&1){C*=W[SIGNW[b]];fired.push(b);}
  return{e1,e2,e3,E:e1*e2*e3,C,fired,share};}
function reasonOff(x){const c=x.c;if(c[8])return'Left out by rule: this is a '+(NEGNAME[c[9]]||'single-specialty')+' facility and the service is outside its specialty, so the text probably belongs to another hospital.';
  const s=[];if(c[2]===1)s.push('only a loose mention of the service');for(let b=0;b<3;b++)if(c[6]>>b&1)s.push(SIGN[b].toLowerCase());if(!s.length)s.push('too little supporting text');return'Left out: weak evidence ('+s.join('; ')+').';}
const opLabel=f=>f[7]==='public'?T('pub'):(f[7]==='private'?T('priv'):'');

function ladder(x){const f=x.f,c=x.c,co=CL[svc].coreq,found=new Set(c[3]),p=parts(c),sv=curSvc()[1].toLowerCase(),L=c[10]||['',0,'','','',''],mk=new Set();(CL[svc].must||[]).forEach(g=>g.k.forEach(k=>mk.add(k)));
  const q=(s,fld)=>'<div class="rq">“'+esc(s)+'”</div><div class="rs">from its '+esc(fld)+' list</div>';
  const rung=(n,t,ok,inner)=>'<div class="rung '+(ok?'ok':'no')+'"><div class="rn">'+(ok?'✓':'?')+'</div><div style="min-width:0">'+TTS+'<div class="rt">'+n+' · '+t+'</div>'+inner+'</div></div>';
  let h='<div class="ladder">';
  h+=rung(1,'Department',L[1]>0,L[1]>0?'<div class="rq">Its specialty list has “'+esc(L[0])+'”'+(L[1]===1?' (the parent department)':'')+'</div><div class="rs">Weakest step: many records list departments they do not run.</div>':'<div class="rq">No matching department on its specialty list</div>');
  h+=rung(2,'Procedure',!!L[2],L[2]?q(L[2],L[3]):'<div class="rq">No sentence names the procedure itself</div>');
  const chips=co.map((n,k)=>'<span class="need '+(found.has(k)?'ok':'no')+'">'+(found.has(k)?'✓ ':'? ')+esc(gl(n))+(mk.has(k)?' <b>· must-have</b>':'')+'</span>').join('');
  h+=rung(3,'Equipment and unit',!!L[4],(L[4]?(L[6]>=0?'<div class="rs">About '+esc(gl(co[L[6]]).toLowerCase())+':</div>':'')+q(L[4],L[5]):'<div class="rq">No sentence describes the unit or its machines</div>')+'<div class="rs" style="margin-top:6px">What '+esc(sv)+' needs, and whether the record mentions it:</div><div class="needs">'+chips+'</div>');
  const loc=f[5]!=null&&f[5]>=0.85?'Address, postcode and map point agree':(f[5]!=null&&f[5]>=0.6?'Two of address, postcode and map point agree':'Address, postcode and map point disagree, so the distance may be wrong');
  const bad=p.fired.map(b=>SIGN[b]);if(!f[8])bad.push('No phone number on the record');
  h+=rung(4,'This place',f[5]!=null&&f[5]>=0.6&&!p.fired.length,'<div class="rq">'+esc(loc)+'</div>'+(c[7]?'<div class="rs">Its own website has a page for this service.</div>':'')+bad.map(b=>'<div class="rs bad">'+esc(b)+'</div>').join(''));
  return h+'</div>';}

let TB=null;const VOICE=()=>{const v=speechSynthesis.getVoices();return v.find(x=>/^en[-_]IN/i.test(x.lang))||v.find(x=>/^en[-_]GB/i.test(x.lang))||v.find(x=>/^en[-_]US/i.test(x.lang))||v.find(x=>/^en/i.test(x.lang))||null;};
try{speechSynthesis.onvoiceschanged=()=>{};speechSynthesis.getVoices();}catch(e){}
function mark(b,s){if(!b)return;b.dataset.st=s;if(b.classList.contains('tts'))b.textContent=s==='play'?'⏸':(s==='pause'?'▶':'🔊');else b.textContent=s==='play'?'⏸ Pause':(s==='pause'?'▶ Resume':(b.dataset.lab||b.textContent));}
function talk(text,btn){try{const S=speechSynthesis;
  if(btn&&TB===btn&&btn.dataset.st&&btn.dataset.st!=='idle'){if(btn.dataset.st==='pause'){S.resume();mark(btn,'play');}else{S.pause();mark(btn,'pause');}return;}
  S.cancel();if(TB)mark(TB,'idle');TB=btn||null;if(btn&&!btn.dataset.lab)btn.dataset.lab=btn.textContent;
  const u=new SpeechSynthesisUtterance(text);const v=VOICE();if(v)u.voice=v;u.lang=v?v.lang:'en-US';u.rate=.9;
  u.onend=u.onerror=()=>{if(TB===btn){mark(btn,'idle');TB=null;}};S.speak(u);mark(btn,'play');}catch(e){}}
const BYST={};for(const k in CITY){const c=CITY[k],p=PL[c[3]];if(!p||!p[0])continue;(BYST[p[0]]=BYST[p[0]]||[]).push(c[0]);}for(const s in BYST)BYST[s]=[...new Set(BYST[s])].sort();const STATES=Object.keys(BYST).sort();
const TTS='<button class="tts" aria-label="Read aloud" title="Read aloud">🔊</button>';
function say(el){try{const c=el.parentElement.cloneNode(true);c.querySelectorAll('.tts').forEach(b=>b.remove());talk(c.innerText.replace(/[✓?·₹☎🔊⏸▶]/g,' ').replace(/\s+/g,' '),el);}catch(e){}}
function ashaList(){const r=last,s=curSvc();if(!r||!r.top.length)return'';
  return '<div id="asha"><b>'+esc(s[1])+' · '+esc(here.label)+' · '+(urgent?'emergency':'planned')+'</b><p class="small">For a health worker to read out or print. Call each place before sending anyone. Emergency: 108.</p><table><tr><th>#</th><th>Place</th><th>km</th><th>Phone</th><th>Evidence</th><th>Ask first</th></tr>'+r.top.map((x,k)=>{const m=mustMiss(x.c);return '<tr><td>'+(k+1)+'</td><td><b>'+esc(x.f[0])+'</b><br>'+esc(x.f[1])+(x.f[7]==='public'?' · public, usually free':(x.f[7]==='private'?' · private, ask price and Ayushman':''))+'</td><td>'+Math.round(x.d)+'</td><td>'+esc(x.f[8]||'none on record')+'</td><td>'+esc(TX.en['b'+x.c[1]])+'</td><td>'+esc(m.length?m.map(q=>q.name).join('; '):'Is it working today?')+'</td></tr>';}).join('')+'</table><p class="small">Take: ID, Ayushman or ration card, MCP card, old prescriptions. This page works without internet once it is open; only the street map needs a connection.</p><button class="chip" id="ashaprint">Print</button> <button class="chip" id="ashasay">'+esc(tr('Read aloud','पढ़कर सुनाएँ'))+'</button></div>';}
function wireAsha(){const b=$('ashabtn');if(!b)return;b.onclick=e=>{e.preventDefault();const x=$('ashabox');x.innerHTML=x.innerHTML?'':ashaList();if($('ashaprint'))$('ashaprint').onclick=()=>window.print();if($('ashasay'))$('ashasay').onclick=()=>{try{talk(last.top.map((x,k)=>'Choice '+(k+1)+': '+x.f[0]+', '+x.f[1]+', about '+Math.round(x.d)+' kilometres. '+(x.f[8]?'Phone '+x.f[8].replace(/\D/g,'').split('').join(' ')+'. ':'No phone on record. ')).join(' '),$('ashasay'));}catch(e){}};if(x.innerHTML)x.scrollIntoView({behavior:'smooth',block:'nearest'});};}
const CITYPIC=[["Delhi", "M8 54h48M14 54V22h36v32M26 54V40a6 6 0 0 1 12 0v14M12 22h40M16 18h32v4M22 14h20v4M28 14c0-3 8-3 8 0"], ["Mumbai", "M8 54h48M16 54V26h32v28M26 54V40a6 6 0 0 1 12 0v14M16 26v-6M48 26v-6M24 26v-4M40 26v-4M14 18a2 2 0 0 1 4 0M46 18a2 2 0 0 1 4 0M22 22a2 2 0 0 1 4 0M38 22a2 2 0 0 1 4 0M19 40h3M42 40h3"], ["Kolkata", "M18 46V16M24 46V16M40 46V16M46 46V16M18 16h6M40 16h6M6 42h52M8 42L21 18M34 42L21 18M30 42L43 18M56 42L43 18M8 54c3-2 5-2 8 0s5 2 8 0 5-2 8 0 5 2 8 0 5-2 8 0"], ["Chennai", "M8 54h48M14 54l4-10h28l4 10M18 44l3-8h22l3 8M21 36l3-8h16l3 8M24 28l3-7h10l3 7M27 21c0-4 10-4 10 0M32 17v-4M28 54v-6a4 4 0 0 1 8 0v6"], ["Hyderabad", "M8 54h48M18 54V30h28v24M25 54V42a7 7 0 0 1 14 0v12M14 54V14M50 54V14M12 12a2 2 0 0 1 4 0M48 12a2 2 0 0 1 4 0M11 30h6M47 30h6M11 22h6M47 22h6M18 26h28"], ["Bengaluru", "M6 54h52M8 54V36h48v18M16 54V40M24 54V40M32 54V40M40 54V40M48 54V40M22 36v-8h20v8M24 28a8 8 0 0 1 16 0M32 20v-5"], ["Patna", "M6 54h52M10 54a22 22 0 0 1 44 0M30 32h4M14 47l7-5M43 42l7 5M29 54v-6h6v6"], ["Lucknow", "M8 54h48M16 54V20h32v34M24 54V34a8 8 0 0 1 16 0v20M16 20l4-6 4 6 4-6 4 6 4-6 4 6 4-6 4 6M12 54V26M52 54V26M10 24a2 2 0 0 1 4 0M50 24a2 2 0 0 1 4 0"], ["_village", "M8 54h48M14 54V34l18-14 18 14v20M10 37l22-19 22 19M28 54V44h8v10M40 30h4v-8"]];
const cityIcon=d=>'<svg viewBox="0 0 64 64" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><path d="'+d+'"/></svg>';

(function(){const ns=new Array(FAC.length).fill(0);for(const s in CL)for(const k in CL[s].c){const c=CL[s].c[k];if(c[1]===2&&!c[8])ns[+k]++;}FAC.forEach((f,i)=>f.ns=ns[i]);})();
const stOf=f=>(PL[f[2]]||['',''])[0];
const med=a=>{a=a.filter(x=>x!=null).sort((x,y)=>x-y);return a.length?a[Math.floor(a.length/2)]:null;};
const avg=a=>{a=a.filter(x=>x!=null);return a.length?a.reduce((x,y)=>x+y,0)/a.length:null;};
const CMP={};
function cstat(key,get,fn){if(CMP[key])return CMP[key];const by={},all=[];FAC.forEach((f,i)=>{const v=get(f,i);if(v==null)return;all.push(v);(by[stOf(f)]=by[stOf(f)]||[]).push(v);});const r={india:fn(all),st:{}};for(const s in by)r.st[s]=fn(by[s]);return CMP[key]=r;}
const fmt=(v,u)=>v==null?'—':(Math.round(v*10)/10).toLocaleString('en-IN')+(u||'');
function crow(icon,label,dir,val,u,S,f,note){const st=S?S.st[stOf(f)]:null;
  return '<div class="cmp"><span class="ci">'+(ICON[icon]||'')+'</span><div class="cl"><b>'+esc(label)+'</b>'+(dir?'<i>'+(dir>0?'↑ Higher is better':'↓ Lower is better')+'</i>':'')+(note?'<i>'+esc(note)+'</i>':'')+'</div><div class="cv"><div class="big">'+(val==null?'<span class="na">Not stated</span>':esc(fmt(val,u)))+'</div>'+(S?'<div class="small">'+esc(stOf(f))+': '+esc(fmt(st,u))+' · India: '+esc(fmt(S.india,u))+'</div>':'')+'</div></div>';}
function compareHtml(x){const f=x.f,c=x.c,co=CL[svc].coreq.length,sv=curSvc()[1];
  const needS=cstat('need_'+svc,(g,i)=>{const q=CL[svc].c[i];return q&&q[1]>=1&&!q[8]?100*q[3].length/co:null;},avg);
  const nsS=cstat('ns',g=>g.ns,avg), bedS=cstat('beds',g=>g[11]&&g[11][0],med), docS=cstat('docs',g=>g[11]&&g[11][1],med), yrS=cstat('yrs',g=>g[11]&&g[11][2]?2026-g[11][2]:null,med);
  const loc=f[5]==null?null:(f[5]>=0.85?'3 of 3':(f[5]>=0.6?'2 of 3':'They disagree'));
  let h='<div class="cmpbox"><div class="lab">How this place compares</div><p class="small">Each number is taken from the record. The comparison is with places in our table in '+esc(stOf(f))+' and in all of India. "Not stated" means the record is silent.</p>';
  h+=crow('stethoscope','Things '+sv.toLowerCase()+' needs, found in its record',1,100*c[3].length/co,'%',needS,f,c[3].length+' of '+co);
  h+=crow('hospital-symbol','Services with strong evidence',1,f.ns,'',nsS,f,'out of 19 we check; state and India show the average');
  h+=crow('hospitalized','Beds',0,f[11]&&f[11][0],'',bedS,f,'as stated');
  h+=crow('doctor','Doctors',0,f[11]&&f[11][1],'',docS,f,'as stated');
  h+=crow('calendar','Years open',0,f[11]&&f[11][2]?2026-f[11][2]:null,'',yrS,f,'');
  const em=CL.trauma_emergency.c[x.i];h+='<div class="cmp"><span class="ci">'+ICON.urgent+'</span><div class="cl"><b>Provides emergency services?</b><i>from the record</i></div><div class="cv"><div class="big">'+(em&&em[1]>=1&&!em[8]?'Yes':(em?'Unclear':'Not stated'))+'</div></div></div>';
  h+='<div class="cmp"><span class="ci">'+ICON['geo-location']+'</span><div class="cl"><b>Address, postcode and map point agree</b><i>79% of places in India: 3 of 3</i></div><div class="cv"><div class="big">'+esc(loc||'—')+'</div></div></div>';
  return h+'</div>';}
const NF=(function(){const seen={},by={},all=[];for(const k in D.nfhs){const v=D.nfhs[k],st=k.split('|')[0],id=st+'|'+v[7];if(seen[id])continue;seen[id]=1;all.push(v);(by[st]=by[st]||[]).push(v);}
  const agg=a=>a[0].map((_,j)=>avg(a.map(v=>typeof v[j]==='number'?v[j]:null)));const r={india:agg(all),st:{}};for(const s in by)r.st[s]=agg(by[s]);return r;})();
function nfTable(key,n){const st=NF.st[key.split('|')[0]]||[],I=NF.india;const rows=[[0,'Births that happen in a hospital or clinic',1],[1,'Of those, in a public facility',0],[4,'Families with health insurance',1],[5,'Women who can read',1]];
  if(svc==='emergency_obstetric')rows.push([2,'C-sections in public facilities',0],[6,'Women with anaemia',-1]);
  return '<table class="nft"><tr><th></th><th>Here</th><th>State</th><th>India</th></tr>'+rows.filter(r=>n[r[0]]!=null).map(r=>'<tr><td>'+esc(r[1])+(r[2]?' <i>'+(r[2]>0?'↑ higher is better':'↓ lower is better')+'</i>':'')+'</td><td><b>'+fmt(n[r[0]],'%')+'</b></td><td>'+fmt(st[r[0]],'%')+'</td><td>'+fmt(I[r[0]],'%')+'</td></tr>').join('')+'</table><p class="small">Government survey NFHS-5 (2019-21). State and India values are averages of districts.</p>';}

const DOCMATCH={dialysis:[/nephro/i,'kidney'],emergency_obstetric:[/gyn|obstet|\bDGO\b|MRCOG/i,'delivery and women'],nicu:[/p(a)?ediat|neonat|\bDCH\b/i,'children and newborns'],cardiac_surgery:[/cardiac surg|cardio ?thoracic|ctvs|cvts/i,'heart surgery'],cath_lab:[/cardiolog|cardiac/i,'heart'],trauma_emergency:[/orthop|ortho\b|trauma|neuro ?surg|general surg|emergency|laparoscop/i,'injuries'],ortho_surgery:[/orthop|ortho\b|joint|spine/i,'bones and joints'],burn_care:[/plastic|burn/i,'burns'],icu:[/critical care|intensiv|ana?esth|pulmon/i,'intensive care'],ventilator:[/critical care|intensiv|ana?esth|pulmon/i,'breathing support'],stroke:[/neurolog|neuro ?surg/i,'brain and nerves'],mri:[/radiolog|\bDMRD\b/i,'scans'],ct_scan:[/radiolog|\bDMRD\b/i,'scans'],chemotherapy:[/oncolog|cancer/i,'cancer'],radiotherapy:[/oncolog|\bDMRT\b/i,'cancer'],cataract_surgery:[/ophthal|\bDOMS\b|\beye\b/i,'eyes'],tb:[/pulmon|chest|respirat|\btb\b/i,'lungs'],blood_bank:[/patholog|transfus/i,'blood'],snakebite_poison:[/physician|internal medicine|emergency|general medicine/i,'emergencies and poisoning']};
function docsFor(f){const W=f[12];if(!W)return{m:[],o:[],W:null};const rx=(DOCMATCH[svc]||[/$^/])[0],m=[],o=[];W[1].forEach(d=>(rx.test((d[2]||'')+' '+(d[1]||''))?m:o).push(d));return{m,o,W};}
const DCOL=['#E7F0FD','#E3F5E8','#FDF0E1','#F1E8FB','#FDE6E3'];
function docRow(d,match){const n=d[0].replace(/^Dr\.?\s*/,''),ini=n.split(/\s+/).filter(w=>/^[A-Za-z]/.test(w)).map(w=>w[0]).slice(0,2).join('').toUpperCase(),col=DCOL[n.length%DCOL.length];
  return '<div class="drow"><span class="dav" style="background:'+col+'">'+esc(ini)+'</span><div class="dtx"><b>'+esc(d[0])+'</b>'+(match?' <span class="pill free">'+esc(tr('Treats','इलाज करते हैं'))+' '+esc((DOCMATCH[svc]||['',''])[1])+'</span>':'')+(d[2]?'<div class="dsp">'+esc(d[2].replace(/\b\w/g,c=>c.toUpperCase()))+'</div>':'')+(d[1]?'<div class="small">'+esc(d[1])+'</div>':'')+((d[3]||d[4])?'<div class="small">'+(d[3]?esc(d[3])+' years experience':'')+(d[3]&&d[4]?' · ':'')+(d[4]?esc(d[4]):'')+'</div>':'')+'</div></div>';}
function whoHtml(f){const r=docsFor(f),sv=curSvc()[1].toLowerCase();
  let h='<div class="who"><div class="lab">'+TTS+esc(tr('Who will treat you','आपका इलाज कौन करेगा'))+'</div>';
  if(r.m.length)h+=r.m.slice(0,3).map(d=>docRow(d,1)).join('');
  else h+='<p class="whyno">'+(r.W?'Its website names '+r.W[1].length+' doctor'+(r.W[1].length>1?'s':'')+', but none for '+esc(sv)+'.':'We could not find doctors on its website.')+' Ask on the phone: <b>which doctor will treat me, and what is their degree?</b></p>';
  if(r.o.length)h+='<details class="sub"><summary>'+r.o.length+' other doctor'+(r.o.length>1?'s':'')+' on its website</summary><div class="inner">'+r.o.map(d=>docRow(d,0)).join('')+'</div></details>';
  if(r.W)h+='<p class="small">From the hospital\'s own website, read on '+esc(r.W[2])+'. Not verified: degrees and experience are as the website states them, and doctors move. <a href="'+esc(r.W[0])+'" target="_blank" rel="noopener">Open the page</a></p>';
  return h+'</div>';}
function mustMiss(c){const co=CL[svc].coreq,must=CL[svc].must||[];return (c[11]||[]).map(k=>must[k]).filter(Boolean).map(g=>({name:g.k.map(k=>gl(co[k])).join(' or '),why:g.why}));}
function body(x,rank,r){const f=x.f,c=x.c,co=CL[svc].coreq,found=new Set(c[3]),p=parts(c),sv=curSvc()[1].toLowerCase(),miss=mustMiss(c),raw=c[12]!=null?c[12]:c[0];
  const asks=[...miss.map(m=>'Do you have this: '+m.name+'?'),...(docsFor(f).m.length?[]:['Which doctor will treat me, and what is their degree?']),'Is '+sv+' working today, and can you take a patient now?',...co.filter((n,k)=>!found.has(k)&&!/count|volume|spec/.test(n)).slice(0,miss.length?1:2).map(n=>'Do you have this: '+gl(n)+'?')].filter((a,k,arr)=>arr.indexOf(a)===k).slice(0,4);
  const why=urgent?'Number '+rank+' of '+r.n+' places within '+r.R+' km that passed the check. Nearest goes first. Places within '+M.tie_km+' km of each other count as equally near, because that is the usual error of our map points; among those, stronger evidence goes first, then public.':'Number '+rank+' of '+r.n+' places within '+r.R+' km that passed the check. Highest trust goes first. Scores within '+M.tie_pts+' points count as equal, because the score moves that much when we refit it on resampled data; among those, the nearer place goes first.';
  const info=[];if(f[11]){if(f[11][0])info.push(f[11][0]+' beds');if(f[11][1])info.push(f[11][1]+' doctors');if(f[11][2])info.push('since '+f[11][2]);}
  let h='<div class="cbody"><div class="btns">'+(f[8]?'<a class="call" href="tel:'+esc(f[8].replace(/[^+\d]/g,''))+'">'+ICON.phone+esc(T('call'))+'</a><span class="phone">'+esc(f[8])+'</span>':'')+'<a class="sec dirb" href="https://www.google.com/maps/dir/?api=1&destination='+f[3]+','+f[4]+'" target="_blank" rel="noopener">'+esc(tr('Directions','रास्ता'))+'</a><button class="sec" data-say="'+(rank-1)+'">'+esc(tr('Listen','सुनें'))+'</button></div>';
  h+='<p class="whyline">'+TTS+esc(why)+'</p>';
  if(miss.length)h+='<div class="mustbox">'+TTS+'<b>☎ Confirm by phone first: '+esc(miss.map(m=>m.name).join('; '))+'.</b> '+esc(miss.map(m=>m.why).join(' '))+' The record does not mention it, so this place cannot get our top grade. It may still have it: ask.</div>';
  const cost=f[7]==='public'?'Public facility: treatment is usually free or low cost.'+(svc==='emergency_obstetric'?' Delivery, C-section, medicines and transport are free in public facilities (JSSK scheme).':'')+(svc==='dialysis'?' Dialysis is free for poor families at national programme centres.':''):(f[7]==='private'?'Private facility: ask the price before you go. Ask if they accept the Ayushman card (PM-JAY pays up to ₹5 lakh a year for eligible families). A national survey (2017-18) found a private hospital stay cost about 7 times a government one: on average ₹27,000 in rural and ₹39,000 in urban areas.':'We do not know if this is public or private. Ask the price before you go.');
  h+=whoHtml(f);
  h+='<div class="cost">'+TTS+'₹ '+esc(cost)+'</div>';
  h+='<div><div class="lab">Why we think it can do this: 4 steps, in the record\'s own words</div>'+ladder(x)+'<p class="small" style="margin-top:6px"><span class="need ok">✓ green</span> the record says it &nbsp; <span class="need no">? red</span> the record is silent, which is not the same as "no"</p></div>';
  h+=compareHtml(x);
  h+='<div>'+TTS+'<div class="lab">Ask on the phone before you go</div><ul class="ask">'+asks.map(a=>'<li>'+esc(a)+'</li>').join('')+'</ul></div>';
  const lines=[['Start',M.base,'the same for every claim'],['Text strength','× '+p.e1.toFixed(1),['','only a loose mention','names the procedure or unit','describes equipment or a unit'][c[2]]],['Needs found','× '+p.e2.toFixed(1),found.size+' of '+co.length+' things this service needs']];
  if(c[7])lines.push(['Own website page','× '+p.e3.toFixed(1),'the facility\'s own site has a page for it']);
  p.fired.forEach(b=>lines.push(['Warning','× '+W[SIGNW[b]],SIGN[b]]));
  const odds=M.base*p.E*p.C;
  h+='<details class="sub"><summary>Show the math: trust '+c[0]+' out of 100</summary><div class="inner"><table class="math">'+lines.map(l=>'<tr><td>'+esc(l[0])+'</td><td class="num">'+esc(String(l[1]))+'</td><td>'+esc(l[2])+'</td></tr>').join('')+'<tr class="tot"><td>Result</td><td class="num">'+odds.toFixed(1)+'</td><td>'+odds.toFixed(1)+' → trust = 100 × '+odds.toFixed(1)+' ÷ ('+odds.toFixed(1)+' + 1) = <b>'+raw+'</b>. This is a ranking index, not the chance that the service is really there.</td></tr>'+(raw>c[0]?'<tr class="tot"><td>Must-have rule</td><td class="num">'+c[0]+'</td><td>A must-have is not on the record, so the score stops at '+c[0]+', just under "strong"</td></tr>':'')+'</table><p class="small">Good signs multiply the number up, warning signs multiply it down. The multipliers were fitted on '+M.n_labelled+' claims we could label: '+V.n_name+' from the facility\'s name alone and '+V.n_read+' after an AI reader read the record. Nobody visited or phoned these facilities.</p></div></details>';
  if(f[10]&&f[10].length)h+='<details class="sub"><summary>Doctors named on the record ('+f[10].length+')</summary><div class="inner"><ul class="ask">'+f[10].map(d=>'<li>'+esc(d)+'</li>').join('')+'</ul><p class="small">Copied from the record, not verified.</p></div></details>';
  return h+'</div>';}

function areaHtml(){const pl=here.pl!=null?PL[here.pl]:null;let h='';if(!pl)return h;const key=nz(pl[0])+'|'+nz(pl[1]),n=D.nfhs[key];
  if(n){const share=n[0]&&n[1]?Math.round(n[1]/n[0]*100):null;const G=[['Hard to reach','Basic care reaches fewer people here than in two thirds of districts. Expect long travel; use the ambulance numbers and ask a health worker to go with you.'],['Partly reached','Basic care reaches people here about as often as in a typical district.'],['Well reached','Basic care reaches most people here.']][n[8]];
    h+='<p class="small">'+esc(pl[1])+', '+esc(pl[0])+' · government survey NFHS-5</p>';h+=nfTable(key,n);if(G)h+='<p><span class="pill b'+n[8]+'">'+G[0]+'</span> '+G[1]+'</p>';
    if(share!=null)h+='<p>'+share+'% of births in a facility here happen in <b>public</b> facilities. Our table is only '+M.public_share_table+'% public, so this list is mostly private hospitals. Check your district hospital too.</p>';
    if(svc==='emergency_obstetric'&&n[2]!=null)h+='<p>C-sections: '+n[2]+'% of births in public facilities'+(n[3]!=null?', '+n[3]+'% in private':'')+'.'+(n[2]<5?' Public facilities here rarely perform them; ask before you travel.':'')+'</p>';
    if(svc==='emergency_obstetric'&&n[6]!=null)h+='<p>'+n[6]+'% of women here are anaemic, which raises the risk in childbirth. Ask whether blood is available.</p>';
    if(n[10]!=null)h+='<p>Families here pay about ₹'+Math.round(n[10]).toLocaleString('en-IN')+' out of pocket for a delivery even in a public facility (India district median ₹2,809).</p>';
    if(n[9]!=null)h+='<p>'+n[9]+'% of girls aged 15 to 19 here are already mothers or pregnant (India district median 4.8%)'+(n[11]!=null?', and '+n[11]+'% of young women married before 18':'')+'. Young mothers need a facility that can handle a difficult birth.</p>';
    if(n[4]!=null)h+='<p>'+n[4]+'% of households have health insurance.'+(n[4]<25?' Ask about government schemes (PM-JAY).':'')+'</p>';}
  if(svc==='dialysis'){const p=D.pmndp[key];if(p&&p.length)h+='<div><div class="lab">Public dialysis centres here (PMNDP, free or low cost)</div><p>'+p.map(esc).join(' · ')+'</p><p class="small">From the official list. Not in our table, so no distance or evidence.</p></div>';}
  return h;}

function drawMap(r){if(map){map.remove();map=null;}if(!window.L||!$('map'))return;
  map=L.map('map',{zoomControl:true,attributionControl:true,preferCanvas:true});
  let failed=false;const tiles=L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png',{maxZoom:18,attribution:'© OpenStreetMap contributors'}).addTo(map);
  tiles.on('tileerror',()=>{if(failed)return;failed=true;const b=map.getBounds().pad(.4);let k=0;for(const p in D.pins){const q=D.pins[p];if(b.contains([q[0],q[1]])&&k++<4000)L.circleMarker([q[0],q[1]],{radius:2,stroke:false,fillColor:'#B4BCC8',fillOpacity:.7,interactive:false}).addTo(map);}$('mapnote').textContent='Street map could not load here, so grey dots show post offices instead. On the deployed app you see streets.';});
  const pts=[[here.lat,here.lon]];
  for(const x of r.all){if(x.c[8]||x.c[1]===0){if(x.d<=r.R)L.circleMarker([x.f[3],x.f[4]],{radius:4,color:'#8A94A3',weight:1.5,fill:true,fillColor:'#fff',fillOpacity:.8}).addTo(map).bindTooltip(esc(x.f[0])+' · left out');}}
  r.top.forEach((x,j)=>{pts.push([x.f[3],x.f[4]]);L.marker([x.f[3],x.f[4]],{icon:L.divIcon({className:'',html:'<div class="pin"><span>'+(j+1)+'</span></div>',iconSize:[36,36],iconAnchor:[18,36]}),zIndexOffset:1000-j}).addTo(map).bindTooltip(esc(x.f[0])+' · '+Math.round(x.d)+' km',{direction:'top',offset:[0,-34]}).on('click',()=>{open=j;paintCards();const el=document.querySelector('[data-card="'+j+'"]');if(el)el.scrollIntoView({behavior:'smooth',block:'nearest'});});});
  L.marker([here.lat,here.lon],{icon:L.divIcon({className:'',html:'<div class="me"></div>',iconSize:[22,22],iconAnchor:[11,11]}),zIndexOffset:2000}).addTo(map).bindTooltip('You: '+esc(here.label),{direction:'top',offset:[0,-12]});
  if(pts.length>1)map.fitBounds(pts,{padding:[50,50],maxZoom:14});else map.setView(pts[0],11);}

function cardsHtml(r){return r.top.map((x,j)=>'<div class="card'+(j===open?' sel':'')+'" data-card="'+j+'"><button class="chead" data-open="'+j+'"><span class="rk">'+(j+1)+'</span><span style="min-width:0"><span class="cname">'+esc(x.f[0])+'</span><div class="cmeta">'+esc(x.f[1])+(PL[x.f[2]]?', '+esc(PL[x.f[2]][0]):'')+'</div><div class="pills"><span class="pill b'+x.c[1]+'">'+esc(T('b'+x.c[1]))+' · '+x.c[0]+'</span>'+(opLabel(x.f)?'<span class="pill pn">'+esc(opLabel(x.f))+'</span>':'')+(docsFor(x.f).m.length?'<span class="pill dpill">'+docsFor(x.f).m.length+' '+esc(tr('doctor'+(docsFor(x.f).m.length>1?'s':'')+' for this','डॉक्टर'))+'</span>':'')+(x.f[7]==='public'?'<span class="pill free">'+esc(tr('Usually free','अक्सर मुफ़्त'))+'</span>':'')+(x.f[6]&&x.f[6]!=='hospital'?'<span class="pill pn">'+esc(x.f[6]==='farmacy'?'pharmacy':x.f[6])+', not a hospital</span>':'')+'</div></span><span class="kmb"><div class="n">'+Math.round(x.d)+'</div><div class="u">'+esc(T('km'))+'</div></span></button>'+(j===open?body(x,j+1,r):'')+'</div>').join('');}
function paintCards(){$('cards').innerHTML=cardsHtml(last);$('cards').querySelectorAll('[data-open]').forEach(b=>b.onclick=()=>{const j=+b.dataset.open;open=open===j?-1:j;paintCards();if(map&&open>=0){const x=last.top[open];map.panTo([x.f[3],x.f[4]]);}});$('cards').querySelectorAll('[data-say]').forEach(b=>b.onclick=()=>speak(last.top[+b.dataset.say],b));$('cards').querySelectorAll('.tts').forEach(b=>b.onclick=e=>{e.stopPropagation();say(b);});}

function speak(x,btn){if(!x)return;try{const en='For '+curSvc()[1]+', choice '+(last.top.indexOf(x)+1)+' is '+x.f[0]+' in '+x.f[1]+', about '+Math.round(x.d)+' kilometres away. '+TX.en['b'+x.c[1]]+'. '+(mustMiss(x.c).length?'First ask on the phone if they have: '+mustMiss(x.c).map(m=>m.name).join(', ')+'. ':'')+'Please call before you go.';const hi=curSvc()[2]+' के लिए विकल्प: '+x.f[0]+', '+x.f[1]+', लगभग '+Math.round(x.d)+' किलोमीटर दूर। '+TX.hi['b'+x.c[1]]+'। जाने से पहले फ़ोन करें।';talk(en,btn);}catch(e){}}
function results(){const r=search();last=r;open=0;const s=curSvc();let h='<div class="screen"><div class="chips"><button class="chip" data-go="1">'+esc(tr(s[1],s[2]))+'</button><button class="chip" data-go="2">'+(urgent?esc(T('u1')):esc(T('u0')))+'</button><button class="chip" data-go="3">'+esc(here.label)+'</button></div>';
  h+='<div class="res"><div class="mapcol"><div id="map"></div><p class="small" id="mapnote"><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#2B6BE4;vertical-align:-1px"></span> you &nbsp; <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:var(--accent);vertical-align:-1px"></span> our choices, numbered &nbsp; <span style="display:inline-block;width:9px;height:9px;border-radius:50%;border:1.5px solid #8A94A3;background:#fff;vertical-align:-1px"></span> mentions this service but was left out (weak evidence or wrong specialty). Distances are straight lines; roads are longer.</p></div><div class="listcol">';
  if(urgent)h+='<div class="banner red">Ambulance 108 · pregnancy and newborn 102 · any emergency 112</div>';
  if(urgent&&r.top.length&&r.top[0].d>50)h+='<div class="banner amber">The nearest place with enough evidence is about '+Math.round(r.top[0].d)+' km away. Do not wait: call 108 now and go to the nearest government hospital first. Our table lists few public hospitals, so one may be closer than anything shown here.</div>';
  h+='<div class="rule">'+(urgent?'<b>Emergency rule.</b> We first remove places with weak evidence or a rule violation. Then we list the <b>nearest</b> ones; when two are equally near (within '+M.tie_km+' km), stronger evidence first.':'<b>Planned rule.</b> We first remove places with weak evidence or a rule violation. Then, inside your area, we list the ones with the <b>highest trust score</b>; when scores are equal within '+M.tie_pts+' points, the nearer one first.')+' Found '+r.n+' within '+r.R+' km.</div>';
  if(!r.top.length)h+='<div class="banner amber">No place with enough evidence within '+r.R+' km. This can mean there is no such care nearby, or that our table does not see it (it misses most public facilities).'+(r.nextOut?' Nearest one with evidence: '+esc(r.nextOut.f[0])+', '+esc(r.nextOut.f[1])+', about '+Math.round(r.nextOut.d)+' km.':'')+'</div>';
  h+='<div class="chips"><span class="small" style="align-self:center">Show</span>'+[['any','All'],['public','Public (low cost)'],['private','Private']].map(o=>'<button class="chip" data-sec="'+o[0]+'" aria-pressed="'+(sector===o[0])+'">'+o[1]+'</button>').join('')+'<span class="small" style="align-self:center;margin-left:8px">Within</span>'+[[0,'Auto'],[30,'30 km'],[50,'50 km'],[100,'100 km']].map(o=>'<button class="chip" data-rad="'+o[0]+'" aria-pressed="'+(rad===o[0])+'">'+o[1]+'</button>').join('')+'</div>';
  if(sector==='public')h+='<p class="small">Our table lists few public facilities (5% of rows). Your district hospital may not appear here.</p>';
  h+='<div id="cards" style="display:flex;flex-direction:column;gap:12px"></div>';
  const ah=areaHtml();if(ah)h+='<details><summary>'+esc(T('area'))+'</summary><div class="inner">'+ah+'</div></details>';
  if(r.off.length)h+='<details><summary>'+esc(T('notrec'))+' ('+r.off.length+')</summary><div class="inner">'+r.off.map(x=>'<div><b>'+esc(x.f[0])+'</b> <span class="small">· '+Math.round(x.d)+' km · '+esc(x.f[1])+'</span><p>'+esc(reasonOff(x))+'</p><p class="small">“'+esc(x.c[4])+'”</p></div>').join('')+'</div></details>';
  h+='<div class="help3"><div><b>🚑 '+esc(tr('Free ride','मुफ़्त सवारी'))+'</b><a href="tel:108">108</a> ambulance · <a href="tel:102">102</a> pregnancy and newborn · <a href="tel:112">112</a> any emergency</div><div><b>🪪 '+esc(tr('Take with you','साथ ले जाएँ'))+'</b>Aadhaar or any ID · Ayushman or ration card · mother-child (MCP) card · old prescriptions. Ayushman help: <a href="tel:14555">14555</a></div><div><b>👩‍⚕️ '+esc(tr('No phone or internet?','फ़ोन या इंटरनेट नहीं?'))+'</b>Your village health worker (ASHA) can search for you. <a href="#" id="ashabtn">'+esc(tr('Open the health worker list','स्वास्थ्य कार्यकर्ता सूची खोलें'))+'</a></div></div><div id="ashabox"></div>';
  h+='<details><summary>'+esc(T('how'))+'</summary><div class="inner"><div class="flow">'
   +'<div class="step"><i>1</i><div><b>Find the sentences.</b> For each of 9,989 facility records we find every sentence about the service. A service word inside the hospital\'s own name does not count.</div></div>'
   +'<div class="step"><i>2</i><div><b>Grade the evidence.</b> Loose mention ×1, named procedure or unit ×'+W.t2+', equipment or unit with detail ×'+W.t3+'. Then up to ×'+W.coreq+' for the share of things the service needs (theatre, anaesthesia, blood…) found in the record, and ×'+W.ownurl+' if its own website has a page for it.</div></div>'
   +'<div class="step"><i>3</i><div><b>Check the row is one real place.</b> Map point, postcode and city all differ ×'+W.all_differ+'. Text names 2+ towns over 60 km away ×'+W.far2+'. Built from 2+ other organisations\' websites ×'+W.other_sites_2plus+'.</div></div>'
   +'<div class="step"><i>4</i><div><b>Score.</b> odds = '+M.base+' × evidence × one-real-place; trust = 100 × odds ÷ (1 + odds). Strong ≥ '+M.high+', some ≥ '+M.low+', weak below.</div></div>'
   +'<div class="step"><i>5</i><div><b>Must-have check.</b> Some things a service cannot run without (for example an operating theatre for a C-section). If the record does not mention one, the score stops at '+M.cap+', just under "strong", and the card says to confirm it by phone. The place stays on the list, because a silent record is not a "no".</div></div>'
   +'<div class="step"><i>6</i><div><b>Remove.</b> Weak evidence is removed. A single-specialty facility (eye, dental, skin, physio, ayurveda) claiming acute care outside its specialty is removed. Both are shown in the left-out list with the reason.</div></div>'
   +'<div class="step"><i>7</i><div><b>Rank.</b> Emergency: nearest first; places within '+M.tie_km+' km of each other count as equally near (the usual gap between a record\'s map point and its postcode centre), and among those stronger evidence goes first, then public. The same hospital listed twice is shown once. Planned: highest trust first inside your area; scores within '+M.tie_pts+' points count as equal (the score moves that much when refitted on resampled labels), and among those the nearer place goes first. In an emergency a place whose location we are unsure of goes after the others. The area is the smallest circle around you that holds 5 places that passed (steps from 25 to 300 km; from 10 to 200 km in an emergency).</div></div></div>'
   +'<p><b>Where the numbers come from.</b> The multipliers are fitted, not chosen by hand, on '+M.n_labelled+' claims we could label. '+V.n_name+' labels come from the facility\'s name alone (a kidney centre that claims dialysis is "likely real"; a dental clinic that claims dialysis is "likely false"). '+V.n_read+' were set after an AI reader read the record. <b>Nobody visited or phoned any facility, so "real" means "the record is believable", not "the service was seen working".</b></p>'
   +'<p><b>Checks, on claims held out from fitting.</b> The score puts a likely-real claim above a likely-false one '+Math.round(V.auc*100)+' times in 100 (AUC '+V.auc+'). It is weakest for '+esc((SVCNAME[V.svc_min]||V.svc_min).toLowerCase())+' (AUC '+V.svc_min_auc+'). Likely real: '+V.real[2]+'% of "strong" ('+V.n[2]+' claims), '+V.real[1]+'% of "some" ('+V.n[1]+'), '+V.real[0]+'% of "weak" ('+V.n[0]+'). The labelled claims are not a random sample, so read the score as an order, not as a probability.</p>'
   +'<p><b>Known weak points.</b> The warning "built from other organisations\' websites" partly measures how long a record is: with record length held fixed its multiplier moves from ×'+W.other_sites_2plus+' to about ×0.8. The "things this service needs" are searched in the whole record, so a match can come from another department. The strong and some cut-offs ('+M.high+' and '+M.low+') were set by us after looking at the labelled claims. The must-have rule is a medical precaution and does not make the strong grade more accurate on our labels.</p>'
   +'<p><b>Limits.</b> The table is 5% public facilities. No text for a service does not mean the place lacks it. Distances are straight lines. This tool does not diagnose. Always call first.</p></div></details>';
  h+='<button class="chip" id="again" style="align-self:center">↺ '+esc(T('again'))+'</button></div></div></div>';return h;}

function render(){try{speechSynthesis.cancel();TB=null;}catch(e){}const v=$('view');$('dots').innerHTML=[1,2,3,4].map(i=>'<i class="'+(i<=step?'on':'')+'"></i>').join('');$('back').style.visibility=step>1?'visible':'hidden';$('lang').textContent=lang==='en'?'हिन्दी':'English';
  if(step===1){v.innerHTML='<div class="screen"><div><h1>'+esc(T('q1'))+'</h1><p class="hint">'+esc(T('h1'))+'</p></div><div class="grid">'+NEEDS.map((n,i)=>'<button class="tile" data-n="'+n[0]+'"><span class="bub h'+(i%5)+'">'+ICON[n[0]]+'</span><b>'+esc(tr(n[1],n[2]))+'</b></button>').join('')+'</div><div class="other"><b>'+esc(tr('Not sure? Describe the problem in your own words','पक्का नहीं? अपनी भाषा में समस्या लिखें'))+'</b><div class="rowi"><input id="sym" placeholder="'+esc(tr('for example: fell from a bike and the leg is bleeding','जैसे: बाइक से गिर गए, पैर से खून बह रहा है'))+'" value="'+esc(typed)+'"><button class="gobtn" id="mic" style="background:var(--accent)">'+esc(tr('Speak','बोलें'))+'</button><button class="gobtn" id="symgo">'+esc(tr('Suggest','सुझाव'))+'</button></div>'+(sugg?'<div class="chips">'+sugg.map(k=>{const n=NEEDS.find(x=>x[0]===k);return '<button class="chip" data-sg="'+k+'" style="background:var(--accentbg);color:var(--accentink)">'+esc(tr(n[1],n[2]))+'</button>';}).join('')+'<button class="chip" data-sg="injury" style="background:var(--soft)">'+esc(tr('General hospital with emergency','आपातकाल वाला सामान्य अस्पताल'))+'</button></div><p class="small">'+esc(sugg.length?'These match your words. They are suggestions, not a diagnosis.':'We could not match your words to a category. A general hospital with an emergency department is the safe choice; in danger call 108.')+'</p>':'')+'</div><p class="small">This tool does not diagnose; a health worker or doctor decides what care is needed.</p></div>';
    const sg=()=>{typed=$('sym').value;sugg=KW.filter(k=>k[1].test(typed)).map(k=>k[0]).slice(0,3);render();};$('symgo').onclick=sg;$('mic').onclick=()=>{const SR=window.SpeechRecognition||window.webkitSpeechRecognition;if(!SR){$('sym').placeholder='Voice input is not available in this browser. Please type.';return;}try{const r=new SR();r.lang=lang==='hi'?'hi-IN':'en-IN';r.onresult=e=>{$('sym').value=e.results[0][0].transcript;sg();};r.onerror=()=>{$('sym').placeholder='Could not hear you. Please type.';};r.start();$('mic').textContent='…';}catch(e){$('sym').placeholder='Voice input is not available here. Please type.';}};$('sym').addEventListener('keydown',e=>{if(e.key==='Enter')sg();});
    v.querySelectorAll('[data-sg]').forEach(b=>b.onclick=()=>{const n=NEEDS.find(x=>x[0]===b.dataset.sg);need=n[0];svc=n[4][0][0];urgent=!!n[3];step=2;render();});
    v.querySelectorAll('.tile').forEach(b=>b.onclick=()=>{const n=NEEDS.find(x=>x[0]===b.dataset.n);need=n[0];svc=n[4][0][0];urgent=!!n[3];step=2;render();});}
  else if(step===2){const n=curNeed();let h='<div class="screen narrow"><div><h1>'+esc(T('q2'))+'</h1><p class="hint">'+esc(tr(n[1],n[2]))+'</p></div>';
    if(n[4].length>1)h+='<div><p class="hint" style="margin-bottom:8px">'+esc(T('which'))+'</p><div class="chips">'+n[4].map(s=>'<button class="chip" data-s="'+s[0]+'" aria-pressed="'+(s[0]===svc)+'">'+esc(tr(s[1],s[2]))+'</button>').join('')+'</div></div>';
    h+='<div class="big"><button class="opt red" id="u1"><span class="bub">'+ICON.urgent+'</span><span><b>'+esc(T('u1'))+'</b><span>'+esc(T('u1s'))+'</span></span></button><button class="opt" id="u0"><span class="bub">'+ICON.plan+'</span><span><b>'+esc(T('u0'))+'</b><span>'+esc(T('u0s'))+'</span></span></button></div></div>';v.innerHTML=h;
    v.querySelectorAll('.chip').forEach(b=>b.onclick=()=>{svc=b.dataset.s;render();});$('u1').onclick=()=>{urgent=true;step=3;render();};$('u0').onclick=()=>{urgent=false;step=3;render();};}
  else if(step===3){v.innerHTML='<div class="screen narrow"><div><h1>'+esc(T('q3'))+'</h1><p class="hint">'+esc(T('h3'))+'</p></div><div class="locbox"><input class="field" id="loc" autocomplete="off" placeholder="'+esc(T('ph'))+'" value="'+esc(here&&here.label!=='My location'?here.label.replace(/^PIN /,''):'')+'"><div id="sugg" class="sugg" hidden></div></div>'+(locErr?'<p class="err">'+esc(locErr)+'</p>':'')+'<div class="lab">'+esc(tr('Or tap your city','या अपना शहर चुनें'))+'</div><div class="cities">'+CITYPIC.map(c=>c[0]==='_village'?'<button class="cityb" data-vil="1"><span>'+cityIcon(c[1])+'</span>'+esc(tr('Village or town','गाँव या कस्बा'))+'</button>':'<button class="cityb" data-c="'+c[0]+'"><span>'+cityIcon(c[1])+'</span>'+esc(c[0])+'</button>').join('')+'</div><div class="lab">'+esc(tr('Or choose from the list','या सूची से चुनें'))+'</div><div class="pick2"><select class="field sel" id="stsel"><option value="">State</option>'+STATES.map(s=>'<option>'+esc(s)+'</option>').join('')+'</select><select class="field sel" id="ctsel" disabled><option value="">City or town</option></select></div><button class="cta" id="go">'+esc(T('go'))+'</button><button class="ghost" id="gps">'+esc(T('gps'))+'</button></div>';
    const go=q=>{const r=resolve(q!=null?q:$('loc').value);if(!r){locErr=T('nf');render();return;}locErr='';here=r;step=4;render();};
    $('go').onclick=()=>go();$('loc').addEventListener('keydown',e=>{if(e.key==='Enter')go();});v.querySelectorAll('[data-c]').forEach(b=>b.onclick=()=>go(b.dataset.c));v.querySelector('[data-vil]').onclick=()=>{$('loc').focus();$('loc').placeholder=tr('Type your village, town or 6-digit PIN','अपना गाँव, कस्बा या 6 अंकों का पिन लिखें');};
    $('stsel').onchange=()=>{const s=$('stsel').value,c=$('ctsel');c.innerHTML='<option value="">City or town ('+(s?BYST[s].length:0)+')</option>'+(s?BYST[s].map(n=>'<option>'+esc(n)+'</option>').join(''):'');c.disabled=!s;};
    $('ctsel').onchange=()=>{const n=$('ctsel').value;if(n)go(n);};
    const sg=$('sugg');$('loc').addEventListener('input',()=>{const q=$('loc').value.trim().toLowerCase();if(!q){sg.hidden=true;return;}let rows=[];
      if(/^\d{2,6}$/.test(q))rows=Object.keys(D.pins).filter(p=>p.startsWith(q)).slice(0,7).map(p=>['PIN '+p,PL[D.pins[p][2]],p]);
      else{const ks=Object.keys(CITY),sz=k=>{const c=CITY[k];let n=0;for(const f of FAC)if(Math.abs(f[3]-c[1])<.12&&Math.abs(f[4]-c[2])<.12)n++;return n;};const a1=ks.filter(k=>k.startsWith(q)).slice(0,60).map(k=>[k,sz(k)+(PL[CITY[k][3]]&&PL[CITY[k][3]][1].toLowerCase()===k?1e5:0)]).sort((x,y)=>y[1]-x[1]).map(x=>x[0]);rows=a1.concat(ks.filter(k=>!k.startsWith(q)&&k.includes(q))).slice(0,7).map(k=>[CITY[k][0],PL[CITY[k][3]],CITY[k][0]]);}
      sg.innerHTML='<button class="srow gpsrow" data-gp="1"><span class="si">'+ICON['geo-location']+'</span><b>'+esc(T('gps'))+'</b></button>'+rows.map(r=>'<button class="srow" data-q="'+esc(r[2])+'"><span class="si">⌕</span><span><b>'+esc(r[0])+'</b><span class="small">'+esc(r[1]?r[1][1]+', '+r[1][0]:'')+'</span></span></button>').join('');sg.hidden=false;
      sg.querySelectorAll('[data-q]').forEach(b=>b.onclick=()=>go(b.dataset.q));sg.querySelector('[data-gp]').onclick=()=>$('gps').click();});
    $('gps').onclick=()=>{const fail=()=>{locErr='Location sharing is not available here. Type a PIN code or a town.';render();};try{navigator.geolocation.getCurrentPosition(p=>{here=nearPlace(p.coords.latitude,p.coords.longitude,'My location');locErr='';step=4;render();},fail,{timeout:8000});}catch(e){fail();}};}
  else{v.innerHTML=results();paintCards();wireAsha();v.querySelectorAll('[data-sec]').forEach(b=>b.onclick=()=>{sector=b.dataset.sec;render();});v.querySelectorAll('[data-rad]').forEach(b=>b.onclick=()=>{rad=+b.dataset.rad;render();});v.querySelectorAll('[data-go]').forEach(b=>b.onclick=()=>{step=+b.dataset.go;render();});$('again').onclick=()=>{step=1;render();};drawMap(last);}
  window.scrollTo(0,0);}
$('back').onclick=()=>{if(step>1){step--;render();}};
$('lang').onclick=()=>{lang=lang==='en'?'hi':'en';render();};
render();
</script>
''')

### Small data files
Labels (`analysis/labels/`), the district table, the public dialysis-centre list, the doctor list read from hospital websites, and the icons, packed as one compressed archive.

In [ ]:
DATA = '...'
tarfile.open(fileobj=io.BytesIO(base64.b64decode(DATA)), mode='r:gz').extractall(WORK)
print(sorted(os.listdir(WORK)))

## 1. Read the three Marketplace tables
List-valued columns are kept as JSON text, exactly as in the source.

In [ ]:
from pyspark.sql import functions as F, types as T
import pandas as pd
SOURCE = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
TARGET = "workspace.default"

def as_text(df):
    cols = []
    for f in df.schema.fields:
        c = F.col(f"`{f.name}`")
        nested = isinstance(f.dataType, (T.ArrayType, T.StructType, T.MapType))
        cols.append((F.to_json(c) if nested else c.cast("string")).alias(f.name))
    return df.select(cols)

fac = as_text(spark.table(f"{SOURCE}.facilities")).toPandas()
fac.to_csv(f"{WORK}/facilities.csv", index=False)
pin = as_text(spark.table(f"{SOURCE}.india_post_pincode_directory")).toPandas()
half = len(pin) // 2
pin.iloc[:half].to_csv(f"{WORK}/indiaPincodeDirectory1.csv", index=False)
pin.iloc[half:].to_csv(f"{WORK}/indiaPincodeDirectory2.csv", index=False)
nfhs = as_text(spark.table(f"{SOURCE}.nfhs_5_district_health_indicators")).toPandas()
nfhs.to_csv(f"{WORK}/nfhs_5_district_health_indicators.csv", index=False)
print("facilities", fac.shape, "| PIN rows", len(pin), "| NFHS districts", len(nfhs))

## 2. Run the pipeline
Each step prints its own checks (claim counts, fitted multipliers with 90% bootstrap ranges, held-out AUC, band shares).

In [ ]:
STEPS = [
    ("pipeline/location.py",         "Where each facility really is: coordinates, postcode and city compared"),
    ("pipeline/claims.py",           "Sentences about each service; evidence tier per (facility, service)"),
    ("pipeline/feat_attribution.py", "Far-away towns named in the text"),
    ("pipeline/feat_sources.py",     "Whose websites each row was built from"),
    ("pipeline/trust.py",            "Fit the multipliers on labelled claims; score every claim"),
    ("pipeline/export_app.py",       "Pack the data the app needs"),
]
for script, what in STEPS:
    print(f"\n=== {script}: {what}")
    r = subprocess.run([sys.executable, script], cwd=WORK, capture_output=True, text=True)
    print("\n".join(l for l in r.stdout.splitlines() if "Warning" not in l)[-2500:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise RuntimeError(f"{script} failed")
r = subprocess.run([sys.executable, "build.py"], cwd=f"{WORK}/app", capture_output=True, text=True); print(r.stdout, r.stderr[-500:])

## 3. Save results as Delta tables

In [ ]:
for name, path in [("caresaathi_trust", "pipeline/out/trust.csv"), ("caresaathi_claims", "pipeline/out/claims.csv"), ("caresaathi_location", "pipeline/out/location.csv")]:
    pdf = pd.read_csv(f"{WORK}/{path}", low_memory=False)
    spark.createDataFrame(pdf.astype(object).where(pdf.notna(), None).astype(str).replace({"None": None})).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{TARGET}.{name}")
    print(f"{TARGET}.{name}: {len(pdf):,} rows")

## 4. One answer: dialysis in Patna
Highest trust first, with the sentence each score rests on.

In [ ]:
display(spark.sql(f"""
  SELECT name, address_city, round(cast(trust AS double), 1) AS trust, tier, coreq_found, coreq_total, quote_field, quote
  FROM {TARGET}.caresaathi_trust
  WHERE service = 'dialysis' AND lower(address_city) = 'patna'
  ORDER BY cast(trust AS double) DESC LIMIT 10
"""))

## 5. Serve it
The rebuilt page is `/tmp/caresaathi/app/index.html`. To publish it, copy `app.py`, `app.yaml`, `requirements.txt` and `index.html` into a workspace folder and deploy that folder as a Databricks App.